# 04 · Generalization, robustness, and the final decision lock
        Review every candidate and seed, selected epochs, training/validation gaps, preprocessing sensitivity and model complexity.
        No TEST score is available in this stage. Choose the maximum mean validation AP across the three seeds; exact ties prefer fewer features
        and lower complexity. A nonsignificant difference is not proof of equivalence and is not used as a feature-count rule.
        The final predictor is the uniform three-seed ensemble; individual results remain visible.

### Authorized runtime and immutable run identity
        Use the existing private Spark/Snowflake session. Each notebook embeds its helpers, uses no repository imports,
        and writes only new run-specific artifacts. Changing code, data, seeds, or protocol requires a new run ID.
        Patient-level arrays and predictions stay in the authorized environment. Do not export notebook outputs containing private rows.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
RUN_ID = "R20261006A"  # Same across all five. New ID required for any changed input/code/protocol.
SOURCE_PREFIX = "TAK861_TX_READY_V63_DL_POC"
EXPERIMENT_PREFIX = SOURCE_PREFIX + "_RIGOROUS1028_V1"
if "spark" not in globals():
    raise RuntimeError("Run in the existing authorized Databricks/Spark environment.")
connection = globals().get("sf_options_dl_poc", globals().get("sf_options"))
if not isinstance(connection, dict) or not connection:
    raise RuntimeError("Use the existing private Snowflake connection; never paste credentials into shared source.")
sf_options_dl_poc = dict(connection)
sf_options_dl_poc.update(sfDatabase="DSVC_TAKEDA_TA_PRIVATE", sfSchema="DS_ML")
import re
if not re.fullmatch(r"[A-Z][A-Z0-9_]{0,23}", RUN_ID):
    raise ValueError("Invalid run ID")
RUN_PREFIX = EXPERIMENT_PREFIX + "_" + RUN_ID


### Versioned implementation
The isolated modules below prevent helper-name collisions and make this notebook independently executable.

In [ ]:
import types, sys, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
EMBEDDED_SOURCES = {'features': '"""TRAIN-only screening for the saved monthly claim-count representation.\n\nSelection is a *proxy shortlist*, never evidence of the optimal Transformer\nfeature count. Original tensor indices are preserved separately from rank order.\nNo method here verifies upstream event cutoffs, counting lineage, or encoding.\n"""\nfrom dataclasses import dataclass\nimport hashlib\nimport inspect\nimport warnings\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score\nfrom sklearn.model_selection import StratifiedKFold\n\n\nDEFAULTS = dict(seed=42, n_splits=3, expected_features=1028, expected_steps=12,\n                candidate_sizes=(16, 32, 64, 128, 256, 512), batch_rows=256,\n                scaling="log1p", scale_floor=1.0, near_constant_fraction=0.995,\n                low_variance_threshold=1e-6, proxy_C=0.1, proxy_max_iter=1500,\n                proxy_tol=1e-4, association_bins=5)\n\n\ndef _require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef _config(config):\n    incoming = {} if config is None else dict(config)\n    _require(not (set(incoming) - set(DEFAULTS)), "Unknown feature-selection configuration.")\n    cfg = {**DEFAULTS, **incoming}\n    for key in ("n_splits", "expected_features", "expected_steps", "batch_rows",\n                "proxy_max_iter", "association_bins"):\n        _require(type(cfg[key]) is int and cfg[key] > 0, key + " must be a positive integer.")\n    _require(cfg["n_splits"] >= 3, "Use at least three patient-grouped folds.")\n    _require(cfg["expected_steps"] % 4 == 0, "Four equally sized temporal blocks are required.")\n    _require(cfg["scaling"] in {"log1p", "floored_standard"}, "Unknown count scaling.")\n    _require(cfg["scale_floor"] >= 1.0, "Scale floor must be >=1 to avoid amplifying rare counts.")\n    _require(0 < cfg["near_constant_fraction"] <= 1, "Invalid near-constant threshold.")\n    _require(cfg["low_variance_threshold"] >= 0, "Invalid low-variance threshold.")\n    _require(cfg["proxy_C"] > 0 and cfg["proxy_tol"] > 0, "Invalid logistic settings.")\n    _require(all(type(k) is int and k > 0 for k in cfg["candidate_sizes"]), "Invalid candidate sizes.")\n    cfg["candidate_sizes"] = tuple(sorted(set(cfg["candidate_sizes"])))\n    return cfg\n\n\ndef _rows(rows, n):\n    arr = np.asarray(rows)\n    if arr.dtype == bool:\n        _require(arr.shape == (n,), "Boolean row selector has wrong shape.")\n        arr = np.flatnonzero(arr)\n    _require(arr.ndim == 1 and np.issubdtype(arr.dtype, np.integer), "Rows must be integer indices.")\n    _require(len(arr) > 0 and np.all((arr >= 0) & (arr < n)), "Rows are empty or out of range.")\n    _require(len(np.unique(arr)) == len(arr), "Duplicate fitting rows are not allowed.")\n    return arr.astype(np.int64, copy=False)\n\n\ndef _mask(valid, shape):\n    if valid is None:\n        return None\n    valid = np.asarray(valid)\n    _require(valid.shape == shape[:2] and valid.dtype == bool, "Validity must be boolean (N,T).")\n    return valid\n\n\ndef patient_snapshot_weights(patient_ids):\n    """Equal total mass per patient, normalized to mean one per snapshot."""\n    ids = np.asarray(patient_ids)\n    _require(ids.ndim == 1 and len(ids) > 0 and not pd.isna(ids).any(), "Invalid patient identifiers.")\n    _, inverse, counts = np.unique(ids.astype(str), return_inverse=True, return_counts=True)\n    weights = 1.0 / counts[inverse]\n    return weights / weights.mean()\n\n\ndef _feature_map(feature_map, width):\n    frame = pd.DataFrame(feature_map).copy()\n    required = {"FEATURE_INDEX", "FEATURE_NAME", "FEATURE_COLUMN"}\n    _require(required <= set(frame.columns), "Feature map requires FEATURE_INDEX/NAME/COLUMN.")\n    _require(len(frame) == width, "Feature map width differs from tensor width.")\n    _require(np.array_equal(frame.FEATURE_INDEX.to_numpy(), np.arange(width)),\n             "Feature map must already be in exact zero-based tensor order; do not silently sort.")\n    for column in ("FEATURE_NAME", "FEATURE_COLUMN"):\n        _require(frame[column].map(lambda x: isinstance(x, str) and bool(x.strip())).all(),\n                 column + " contains invalid names.")\n        _require(frame[column].is_unique, column + " is not unique.")\n    _require(frame.FEATURE_COLUMN.tolist() == [f"F{i:04d}" for i in range(width)],\n             "Feature aliases do not match the saved F0000... tensor contract.")\n    forbidden = {"RESP", "PATIENT_ID", "END_DT", "TIME_STEP", "SPLIT"}\n    _require(not forbidden.intersection(frame.FEATURE_NAME.str.upper()), "Identifier/target in features.")\n    return frame.reset_index(drop=True)\n\n\ndef validate_count_inputs(X, metadata, feature_map, valid=None, expected_features=1028,\n                          expected_steps=12, value_rows=None, batch_rows=256):\n    """Validate identity globally, but values only on requested rows.\n\n    The selection entrypoint passes TRAIN rows. Calling with no value_rows audits\n    all input values and belongs in preparation or after the evaluation lock.\n    NaN is preserved as missing; inf, negatives, fractions and nonzero padding fail.\n    """\n    _require(isinstance(X, np.ndarray) and X.ndim == 3, "X must be an ndarray/memmap (N,T,F).")\n    _require(X.shape[1:] == (expected_steps, expected_features), "Unexpected count tensor shape.")\n    _require(np.issubdtype(X.dtype, np.number) and not np.iscomplexobj(X), "Counts must be real numeric.")\n    _require(batch_rows > 0, "batch_rows must be positive.")\n    fm = _feature_map(feature_map, X.shape[2])\n    meta = pd.DataFrame(metadata)\n    _require(len(meta) == len(X), "Metadata rows do not match tensor rows.")\n    _require({"PATIENT_ID", "END_DT", "RESP", "SPLIT"} <= set(meta), "Incomplete frozen metadata.")\n    _require(not meta[["PATIENT_ID", "END_DT", "RESP", "SPLIT"]].isna().any().any(), "Null identity/label/split.")\n    _require(meta.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x)).all(), "Patient IDs must be strings.")\n    dates = pd.to_datetime(meta.END_DT, errors="raise")\n    keys = pd.DataFrame({"PATIENT_ID": meta.PATIENT_ID.to_numpy(), "END_DT": dates.to_numpy()})\n    _require(not keys.duplicated().any(), "Duplicate patient-snapshot keys.")\n    _require(meta.RESP.isin([0, 1]).all(), "RESP must be binary.")\n    _require(set(meta.SPLIT) <= {"train", "validation", "test"} and "train" in set(meta.SPLIT),\n             "Expected frozen train/validation/test split labels.")\n    _require(meta.groupby("PATIENT_ID").SPLIT.nunique().max() == 1, "A patient crosses frozen splits.")\n    valid = _mask(valid, X.shape)\n    rows = np.arange(len(X)) if value_rows is None else _rows(value_rows, len(X))\n    missing = 0\n    for start in range(0, len(rows), batch_rows):\n        rr = rows[start:start + batch_rows]\n        block = X[rr]\n        vv = np.ones(block.shape[:2], bool) if valid is None else valid[rr]\n        observed = block[vv]\n        _require(not np.isinf(observed).any(), "Infinite count value.")\n        finite = observed[np.isfinite(observed)]\n        _require(np.all(finite >= 0), "Negative count value.")\n        _require(np.all(finite == np.floor(finite)), "Fractional count: encoded values are not raw counts.")\n        _require(np.all(np.isfinite(block[~vv])) and np.all(block[~vv] == 0),\n                 "Explicit padded positions must be finite zero; zero alone does not define padding.")\n        missing += int(np.isnan(observed).sum())\n    digest = hashlib.sha256(fm.to_csv(index=False, lineterminator="\\n").encode()).hexdigest()\n    return dict(shape=list(X.shape), feature_map_sha256=digest, value_rows_audited=len(rows),\n                missing_values=missing, mask_source="all_positions_retained" if valid is None else "supplied_boolean_mask",\n                zero_is_padding=False, upstream_provenance="NOT_VERIFIED_BY_THIS_CHECK")\n\n\n@dataclass\nclass CountPreprocessor:\n    median: np.ndarray\n    mean: np.ndarray\n    scale: np.ndarray\n    eligible_indices: np.ndarray\n    quality: pd.DataFrame\n    scaling: str\n    scale_floor: float\n    n_features: int\n    n_steps: int\n    fit_patients: int\n    fit_snapshots: int\n    batch_rows: int = 256\n\n    def to_dict(self):\n        return dict(schema=1, median=self.median.tolist(), mean=self.mean.tolist(), scale=self.scale.tolist(),\n                    eligible_indices=self.eligible_indices.tolist(),\n                    quality=self.quality.astype(object).where(pd.notna(self.quality), None).to_dict("records"),\n                    scaling=self.scaling, scale_floor=float(self.scale_floor), n_features=self.n_features,\n                    n_steps=self.n_steps, fit_patients=self.fit_patients, fit_snapshots=self.fit_snapshots,\n                    batch_rows=self.batch_rows)\n\n    @classmethod\n    def from_dict(cls, state):\n        state = dict(state)\n        _require(state.pop("schema", None) == 1, "Unsupported preprocessing schema.")\n        for key in ("median", "mean", "scale"):\n            state[key] = np.asarray(state[key], dtype=np.float64)\n        state["eligible_indices"] = np.asarray(state["eligible_indices"], dtype=np.int64)\n        state["quality"] = pd.DataFrame(state["quality"])\n        obj = cls(**state)\n        _require(all(v.shape == (obj.n_features,) and np.isfinite(v).all()\n                     for v in (obj.median, obj.mean, obj.scale)), "Invalid saved preprocessing vectors.")\n        _require((obj.median >= 0).all() and (obj.scale > 0).all(), "Invalid saved median/scale.")\n        _require(np.array_equal(obj.eligible_indices, np.unique(obj.eligible_indices)) and\n                 np.all((obj.eligible_indices >= 0) & (obj.eligible_indices < obj.n_features)),\n                 "Invalid saved eligible feature order.")\n        return obj\n\n\ndef _weighted_quantile(x, weights, quantiles):\n    order = np.argsort(x, kind="stable")\n    values, weights = x[order], weights[order]\n    cumulative = np.cumsum(weights)\n    return np.interp(np.asarray(quantiles) * cumulative[-1], cumulative - 0.5 * weights, values)\n\n\ndef fit_count_preprocessor(X, train_rows, patient_ids, valid=None, config=None):\n    """One feature at a time avoids a second full raw tensor in memory.\n\n    Removes only all-missing, fully observed exact constants, and exact duplicate\n    sequences. A constant observed value with variable missingness is retained.\n    Near-constant, rare and low-variance features are diagnostics, not exclusions.\n    """\n    cfg = _config(config)\n    _require(X.ndim == 3, "Expected three-dimensional count tensor.")\n    rows = _rows(train_rows, len(X))\n    ids = np.asarray(patient_ids)\n    _require(ids.shape == (len(X),), "Patient ID shape differs from tensor rows.")\n    valid = _mask(valid, X.shape)\n    vv = np.ones((len(rows), X.shape[1]), bool) if valid is None else valid[rows]\n    sw = patient_snapshot_weights(ids[rows])\n    weights = sw[:, None] / np.maximum(vv.sum(axis=1, keepdims=True), 1)\n    weights = np.broadcast_to(weights, vv.shape)[vv]\n    _require(weights.sum() > 0, "No valid TRAIN timesteps.")\n    _, patient_codes = np.unique(ids[rows].astype(str), return_inverse=True)\n    f = X.shape[2]\n    medians, means, scales = np.zeros(f), np.zeros(f), np.ones(f)\n    ledger, eligible, hashes = [], [], {}\n    for j in range(f):\n        raw = np.asarray(X[rows, :, j], dtype=np.float64)\n        values = raw[vv]\n        _require(not np.isinf(values).any(), "Infinite fitting count.")\n        present = np.isfinite(values)\n        obs = values[present]\n        _require(np.all(obs >= 0) and np.all(obs == np.floor(obs)), "Fitting values must be nonnegative integer counts.")\n        nonzero_rows = np.any(vv & np.isfinite(raw) & (raw > 0), axis=1)\n        observed_rows = np.any(vv & np.isfinite(raw), axis=1)\n        missing_fraction = float(np.average(~present, weights=weights))\n        observed_variance = 0.0\n        mode_fraction = 1.0\n        raw_quantiles = [None, None, None]\n        raw_max = raw_mean = raw_sd = None\n        if len(obs):\n            ow = weights[present]\n            raw_quantiles = _weighted_quantile(obs, ow, [0.5, 0.95, 0.99]).tolist()\n            medians[j] = raw_quantiles[0]\n            raw_max, raw_mean = float(obs.max()), float(np.average(obs, weights=ow))\n            raw_sd = float(np.sqrt(np.average((obs - raw_mean) ** 2, weights=ow)))\n            unique, codes = np.unique(obs, return_inverse=True)\n            mode_fraction = float(np.bincount(codes, weights=ow).max() / ow.sum())\n            logged = np.log1p(obs)\n            observed_variance = float(np.average((logged - np.average(logged, weights=ow)) ** 2, weights=ow))\n        complete = np.where(present, values, medians[j])\n        logged = np.log1p(complete)\n        if cfg["scaling"] == "floored_standard":\n            means[j] = np.average(logged, weights=weights)\n            scales[j] = max(cfg["scale_floor"], np.sqrt(np.average((logged - means[j]) ** 2, weights=weights)))\n        canonical = np.where(np.isnan(values), -1.0, values).astype("<f8", copy=False)\n        fingerprint = hashlib.sha256(canonical.tobytes()).hexdigest()\n        duplicate = None\n        for previous in hashes.get(fingerprint, []):\n            if np.array_equal(X[rows, :, previous][vv], values, equal_nan=True):\n                duplicate = previous\n                break\n        if len(obs) == 0:\n            reason = "all_missing_in_fit"\n        elif np.ptp(obs) == 0 and present.all():\n            reason = "constant_in_fit"\n        elif duplicate is not None:\n            reason = "exact_duplicate_in_fit"\n        else:\n            reason = "eligible"\n            eligible.append(j)\n            hashes.setdefault(fingerprint, []).append(j)\n        ledger.append(dict(FEATURE_INDEX=j, ELIGIBLE=reason == "eligible", QUALITY_REASON=reason,\n                           DUPLICATE_OF=-1 if duplicate is None else duplicate,\n                           MISSING_FRACTION=missing_fraction, IMPUTED_FIT_VALUES=int((~present).sum()),\n                           OBSERVED_PATIENTS=int(len(np.unique(patient_codes[observed_rows]))),\n                           NONZERO_PATIENTS=int(len(np.unique(patient_codes[nonzero_rows]))),\n                           OBSERVED_COUNT_CELLS=int(len(obs)), RAW_Q50=raw_quantiles[0],\n                           RAW_Q95=raw_quantiles[1], RAW_Q99=raw_quantiles[2],\n                           RAW_MAX=raw_max, RAW_MEAN=raw_mean, RAW_SD=raw_sd,\n                           OBSERVED_MODE_FRACTION=mode_fraction,\n                           NEAR_CONSTANT=bool(mode_fraction >= cfg["near_constant_fraction"]),\n                           LOG_COUNT_VARIANCE=observed_variance,\n                           LOW_VARIANCE=bool(observed_variance <= cfg["low_variance_threshold"]),\n                           MISSINGNESS_VARIES=bool(present.any() and (~present).any())))\n    _require(len(eligible) > 0, "No eligible feature remains in fitting partition.")\n    return CountPreprocessor(medians, means, scales, np.asarray(eligible, dtype=np.int64),\n                             pd.DataFrame(ledger), cfg["scaling"], cfg["scale_floor"], f, X.shape[1],\n                             len(np.unique(ids[rows])), len(rows), cfg["batch_rows"])\n\n\ndef _indices(state, feature_indices):\n    indices = state.eligible_indices if feature_indices is None else np.asarray(feature_indices)\n    _require(indices.ndim == 1 and np.issubdtype(indices.dtype, np.integer), "Invalid feature indices.")\n    _require(len(indices) > 0 and len(np.unique(indices)) == len(indices) and\n             np.isin(indices, state.eligible_indices).all(), "Selection contains ineligible/duplicate features.")\n    return indices.astype(np.int64, copy=False)\n\n\ndef transform_counts(X, state, valid=None, rows=None, feature_indices=None, out=None):\n    """Apply the frozen TRAIN state, leaving inputs untouched and padding zero."""\n    _require(X.shape[1:] == (state.n_steps, state.n_features), "Tensor/preprocessor shape mismatch.")\n    valid = _mask(valid, X.shape)\n    rows = np.arange(len(X)) if rows is None else _rows(rows, len(X))\n    columns = _indices(state, feature_indices)\n    shape = (len(rows), state.n_steps, len(columns))\n    if out is None:\n        out = np.empty(shape, dtype=np.float32)\n    _require(out.shape == shape and out.dtype == np.float32, "Output must be float32 with selected tensor shape.")\n    for start in range(0, len(rows), state.batch_rows):\n        rr = rows[start:start + state.batch_rows]\n        block = np.array(X[np.ix_(rr, np.arange(state.n_steps), columns)], dtype=np.float64)\n        vv = np.ones(block.shape[:2], bool) if valid is None else valid[rr]\n        observed = block[vv]\n        finite = observed[np.isfinite(observed)]\n        _require(not np.isinf(observed).any() and np.all(finite >= 0) and np.all(finite == np.floor(finite)),\n                 "Invalid raw counts during transform.")\n        block = np.where(np.isnan(block), state.median[columns][None, None, :], block)\n        block = (np.log1p(block) - state.mean[columns]) / state.scale[columns]\n        block[~vv] = 0.0\n        out[start:start + len(rr)] = block.astype(np.float32)\n    return out\n\n\ndef temporal_count_summaries(X, state, rows, valid=None, feature_indices=None):\n    """log1p sums in four blocks and newest 1/3/6/12 positions (for T=12).\n\n    Sums use raw count medians only where counts are missing. No count is copied\n    across a timestep; explicit padded positions contribute zero. The screening\n    summaries intentionally use log1p sums regardless of tensor scaling ablation.\n    Output (rows, summaries, features) has feature identity on the last axis.\n    """\n    rows = _rows(rows, len(X))\n    valid = _mask(valid, X.shape)\n    columns = _indices(state, feature_indices)\n    t = state.n_steps\n    _require(t % 4 == 0, "Four equal temporal blocks required.")\n    q = t // 4\n    windows = [(i * q, (i + 1) * q) for i in range(4)] + [(0, n) for n in (1, q, 2 * q, t)]\n    names = [f"block_{i}" for i in range(4)] + [f"newest_{n}" for n in (1, q, 2 * q, t)]\n    output = np.empty((len(rows), len(windows), len(columns)), dtype=np.float32)\n    for start in range(0, len(rows), state.batch_rows):\n        rr = rows[start:start + state.batch_rows]\n        raw = np.asarray(X[np.ix_(rr, np.arange(t), columns)], dtype=np.float64)\n        raw = np.where(np.isnan(raw), state.median[columns][None, None, :], raw)\n        if valid is not None:\n            raw[~valid[rr]] = 0.0\n        for i, (begin, end) in enumerate(windows):\n            output[start:start + len(rr), i] = np.log1p(raw[:, begin:end].sum(axis=1))\n    _require(np.isfinite(output).all(), "Nonfinite temporal summaries.")\n    return output, names\n\n\ndef _weighted_corr(a, b, w):\n    a, b = a - np.average(a, weights=w), b - np.average(b, weights=w)\n    denominator = np.sqrt(np.sum(w * a * a) * np.sum(w * b * b))\n    return 0.0 if denominator <= 0 else float(np.sum(w * a * b) / denominator)\n\n\ndef _weighted_midranks(x, w):\n    unique, codes = np.unique(x, return_inverse=True)\n    masses = np.bincount(codes, weights=w, minlength=len(unique))\n    ranks = (np.cumsum(masses) - 0.5 * masses) / masses.sum()\n    return ranks[codes]\n\n\ndef _binned_mi(x, y, w, bins, n_patients):\n    edges = np.unique(_weighted_quantile(x, w, np.linspace(0, 1, bins + 1)[1:-1]))\n    codes = np.searchsorted(edges, x, side="right")\n    joint = np.bincount(codes * 2 + y, weights=w, minlength=(len(edges) + 1) * 2).reshape(-1, 2)\n    joint /= joint.sum()\n    expected = joint.sum(axis=1, keepdims=True) * joint.sum(axis=0, keepdims=True)\n    present = joint > 0\n    mi = float(np.sum(joint[present] * np.log(joint[present] / expected[present])))\n    occupied = int((joint.sum(axis=1) > 0).sum())\n    # Conservative patient-count null-bias correction; descriptive, no p-value claim.\n    corrected = max(0.0, mi - (occupied - 1) / (2.0 * n_patients))\n    return mi, corrected\n\n\ndef rank_count_features(summaries, y, patient_ids, original_indices, summary_names, bins=5):\n    """Type-aware count ranking: weighted rank association plus binned nonlinear MI.\n\n    Counts have ties and zero inflation. Weighted midranks handle ties; bounded\n    quantile bins avoid treating each count as a separate discrete category.\n    Scores are descriptive, not significance/causality or reliable interaction tests.\n    Recent-minus-old block contrast adds a coarse temporal-change diagnostic.\n    """\n    y = np.asarray(y, dtype=np.int64)\n    weights = patient_snapshot_weights(patient_ids)\n    _require(set(np.unique(y)) == {0, 1}, "Ranking requires both RESP classes.")\n    yrank = _weighted_midranks(y, weights)\n    records = []\n    n_patients = len(np.unique(patient_ids))\n    for j, original in enumerate(original_indices):\n        x = summaries[:, :, j]\n        vectors = [x[:, s] for s in range(x.shape[1])] + [x[:, 0] - x[:, 3]]\n        correlations, mis, corrected = [], [], []\n        for vector in vectors:\n            correlations.append(_weighted_corr(_weighted_midranks(vector, weights), yrank, weights))\n            mi, adjustment = _binned_mi(vector, y, weights, bins, n_patients)\n            mis.append(mi)\n            corrected.append(adjustment)\n        association = np.maximum(np.abs(correlations), np.sqrt(2 * np.asarray(corrected)))\n        best = int(np.argmax(association))\n        records.append(dict(FEATURE_INDEX=int(original), RANKING_SCORE=float(association[best]),\n                            MAX_ABS_SPEARMAN=float(np.max(np.abs(correlations))),\n                            MAX_BINNED_MI=float(np.max(mis)), MAX_CORRECTED_MI=float(np.max(corrected)),\n                            TEMPORAL_CONTRAST_SCORE=float(association[-1]),\n                            STRONGEST_SUMMARY=(list(summary_names) + ["recent_minus_oldest_block"])[best],\n                            ASSOCIATION_DIRECTION=float(correlations[best])))\n    result = pd.DataFrame(records).sort_values(["RANKING_SCORE", "FEATURE_INDEX"], ascending=[False, True])\n    result["RANK"] = np.arange(1, len(result) + 1)\n    return result.reset_index(drop=True)\n\n\ndef _proxy(cfg):\n    parameter = inspect.signature(LogisticRegression).parameters.get("penalty")\n    kwargs = {"l1_ratio": 0.0} if parameter is None or parameter.default == "deprecated" else {"penalty": "l2"}\n    return LogisticRegression(**kwargs, C=cfg["proxy_C"], solver="lbfgs", max_iter=cfg["proxy_max_iter"],\n                              tol=cfg["proxy_tol"], random_state=cfg["seed"])\n\n\ndef _patient_folds(labels, ids, cfg):\n    patients, inverse = np.unique(ids.astype(str), return_inverse=True)\n    patient_labels = np.zeros(len(patients), dtype=int)\n    np.maximum.at(patient_labels, inverse, labels)\n    _require(np.bincount(patient_labels, minlength=2).min() >= cfg["n_splits"],\n             "Too few positive/negative TRAIN patients for grouped folds.")\n    splitter = StratifiedKFold(cfg["n_splits"], shuffle=True, random_state=cfg["seed"])\n    for fit_groups, score_groups in splitter.split(patients, patient_labels):\n        yield np.flatnonzero(np.isin(inverse, fit_groups)), np.flatnonzero(np.isin(inverse, score_groups))\n\n\ndef one_se_candidates(cv_summary):\n    """Choose proxy best/one-SE/one-smaller; all-eligible stays a control."""\n    table = cv_summary.copy()\n    _require(len(table) > 0 and np.isfinite(table[["MEAN_AP", "SE_AP"]]).all().all(), "Invalid CV summary.")\n    ordered = table.sort_values(["MEAN_AP", "FINAL_FEATURE_COUNT", "CANDIDATE"], ascending=[False, True, True])\n    best = ordered.iloc[0]\n    eligible = table.loc[table.MEAN_AP >= best.MEAN_AP - best.SE_AP]\n    one = eligible.sort_values(["FINAL_FEATURE_COUNT", "MEAN_AP"], ascending=[True, False]).iloc[0]\n    smaller = table.loc[table.FINAL_FEATURE_COUNT < one.FINAL_FEATURE_COUNT].sort_values("FINAL_FEATURE_COUNT")\n    choices = {"proxy_best": str(best.CANDIDATE), "proxy_one_se": str(one.CANDIDATE)}\n    if len(smaller):\n        choices["one_smaller"] = str(smaller.iloc[-1].CANDIDATE)\n    choices["all_eligible"] = "all_eligible"\n    return choices, float(best.MEAN_AP - best.SE_AP)\n\n\n@dataclass\nclass FeatureSelectionResult:\n    selected_indices: np.ndarray\n    eligible_indices: np.ndarray\n    candidate_indices: dict\n    candidate_tags: dict\n    ranking: pd.DataFrame\n    quality: pd.DataFrame\n    cv_scores: pd.DataFrame\n    cv_summary: pd.DataFrame\n    fold_rankings: pd.DataFrame\n    fold_audit: pd.DataFrame\n    preprocessor: CountPreprocessor\n    chosen_candidate: str\n    input_audit: dict\n    config: dict\n\n\ndef select_ranked_features(X, metadata, feature_map, valid=None, config=None):\n    """No outer VALIDATION/TEST values or labels enter ranking or proxy scores.\n\n    Metadata identity/split checks are global; actual value validation and every\n    fitted statistic are TRAIN-only. The proxy is weighted regularized logistic\n    regression, intentionally a transparent floor, not a nonlinear model claim.\n    """\n    cfg = _config(config)\n    meta = pd.DataFrame(metadata).reset_index(drop=True)\n    train = np.flatnonzero(meta.SPLIT.to_numpy() == "train")\n    audit = validate_count_inputs(X, meta, feature_map, valid, cfg["expected_features"],\n                                  cfg["expected_steps"], value_rows=train, batch_rows=cfg["batch_rows"])\n    fm = _feature_map(feature_map, X.shape[2])\n    ids = meta.PATIENT_ID.to_numpy()\n    labels = meta.RESP.to_numpy(dtype=int)\n    candidate_requests = {f"top_{k}": k for k in cfg["candidate_sizes"] if k < X.shape[2]}\n    candidate_requests["all_eligible"] = X.shape[2]\n    scores, ranks, audits = [], [], []\n    for fold, (fit_local, score_local) in enumerate(_patient_folds(labels[train], ids[train], cfg)):\n        fit_rows, score_rows = train[fit_local], train[score_local]\n        state = fit_count_preprocessor(X, fit_rows, ids, valid, cfg)\n        fit_values, names = temporal_count_summaries(X, state, fit_rows, valid)\n        score_values, _ = temporal_count_summaries(X, state, score_rows, valid)\n        ranking = rank_count_features(fit_values, labels[fit_rows], ids[fit_rows], state.eligible_indices,\n                                      names, cfg["association_bins"])\n        ranked = ranking.FEATURE_INDEX.to_numpy()\n        lookup = {int(original): j for j, original in enumerate(state.eligible_indices)}\n        ranking["FOLD"] = fold\n        ranks.append(ranking)\n        fit_weights, score_weights = patient_snapshot_weights(ids[fit_rows]), patient_snapshot_weights(ids[score_rows])\n        cached = {}\n        for tag, requested in candidate_requests.items():\n            selected = np.sort(ranked[:min(requested, len(ranked))])\n            key = tuple(selected)\n            if key not in cached:\n                positions = [lookup[int(i)] for i in selected]\n                a = fit_values[:, :, positions].reshape(len(fit_rows), -1)\n                b = score_values[:, :, positions].reshape(len(score_rows), -1)\n                model = _proxy(cfg)\n                with warnings.catch_warnings(record=True) as caught:\n                    warnings.simplefilter("always", ConvergenceWarning)\n                    model.fit(a, labels[fit_rows], sample_weight=fit_weights)\n                _require(not any(issubclass(w.category, ConvergenceWarning) for w in caught),\n                         "Proxy did not converge; do not select using an unfinished fit. Increase declared proxy budget.")\n                cached[key] = (average_precision_score(labels[score_rows], model.predict_proba(b)[:, 1], sample_weight=score_weights),\n                               average_precision_score(labels[fit_rows], model.predict_proba(a)[:, 1], sample_weight=fit_weights))\n            ap, fit_ap = cached[key]\n            scores.append(dict(FOLD=fold, CANDIDATE=tag, FEATURE_COUNT=len(selected), AP=float(ap),\n                               FIT_AP=float(fit_ap), GAP=float(fit_ap - ap), METRIC="patient_weighted_proxy_AP"))\n        audits.append(dict(FOLD=fold, FIT_PATIENTS=len(np.unique(ids[fit_rows])), SCORE_PATIENTS=len(np.unique(ids[score_rows])),\n                           PATIENT_OVERLAP=int(len(set(ids[fit_rows]) & set(ids[score_rows]))),\n                           FIT_SNAPSHOTS=len(fit_rows), SCORE_SNAPSHOTS=len(score_rows),\n                           FIT_POSITIVES=int(labels[fit_rows].sum()), SCORE_POSITIVES=int(labels[score_rows].sum()),\n                           ELIGIBLE_FEATURES=len(state.eligible_indices),\n                           FIT_ROWS_SHA256=hashlib.sha256(fit_rows.astype("<i8").tobytes()).hexdigest(),\n                           SCORE_ROWS_SHA256=hashlib.sha256(score_rows.astype("<i8").tobytes()).hexdigest()))\n        del fit_values, score_values\n    state = fit_count_preprocessor(X, train, ids, valid, cfg)\n    summaries, names = temporal_count_summaries(X, state, train, valid)\n    ranking = rank_count_features(summaries, labels[train], ids[train], state.eligible_indices, names, cfg["association_bins"])\n    del summaries\n    fold_rankings = pd.concat(ranks, ignore_index=True)\n    stability = fold_rankings.groupby("FEATURE_INDEX").agg(FOLD_ELIGIBILITY_COUNT=("RANK", "count"),\n                                                          MEAN_FOLD_RANK=("RANK", "mean"), SD_FOLD_RANK=("RANK", "std"),\n                                                          MEAN_FOLD_SCORE=("RANKING_SCORE", "mean"))\n    ranking = ranking.merge(stability, on="FEATURE_INDEX", how="left").merge(fm, on="FEATURE_INDEX", how="left")\n    ranking["FOLD_ELIGIBILITY_FRACTION"] = ranking.FOLD_ELIGIBILITY_COUNT.fillna(0) / cfg["n_splits"]\n    quality = fm.merge(state.quality, on="FEATURE_INDEX", validate="one_to_one")\n    cv_scores = pd.DataFrame(scores)\n    summary = cv_scores.groupby("CANDIDATE", sort=False).agg(MEAN_AP=("AP", "mean"), SD_AP=("AP", "std"),\n                                                           N_FOLDS=("AP", "size"), MEAN_FIT_AP=("FIT_AP", "mean"),\n                                                           MEAN_GAP=("GAP", "mean")).reset_index()\n    summary["SE_AP"] = summary.SD_AP / np.sqrt(summary.N_FOLDS)\n    ranked = ranking.FEATURE_INDEX.to_numpy(dtype=int)\n    full_candidates = {tag: np.sort(ranked[:min(k, len(ranked))]) for tag, k in candidate_requests.items()}\n    summary["FINAL_FEATURE_COUNT"] = summary.CANDIDATE.map(lambda tag: len(full_candidates[tag]))\n    choices, boundary = one_se_candidates(summary)\n    summary["ONE_SE_BOUNDARY"] = boundary\n    summary["WITHIN_ONE_SE"] = summary.MEAN_AP >= boundary\n    shortlist, tags, seen = {}, {}, {}\n    # Preserve the literal all_eligible control name even when another tag selects the same set.\n    for tag in ("all_eligible", "proxy_best", "proxy_one_se", "one_smaller"):\n        if tag not in choices:\n            continue\n        values = full_candidates[choices[tag]]\n        key = tuple(values)\n        canonical = seen.get(key, tag)\n        tags[tag] = canonical\n        if key not in seen:\n            seen[key] = canonical\n            shortlist[canonical] = values\n    selected = full_candidates[choices["proxy_one_se"]]\n    quality["PROXY_ONE_SE_SELECTED"] = quality.FEATURE_INDEX.isin(selected)\n    quality["SELECTION_REASON"] = np.where(~quality.ELIGIBLE, quality.QUALITY_REASON,\n                                           np.where(quality.PROXY_ONE_SE_SELECTED, "proxy_one_se_shortlist", "eligible_control_only_or_other_candidate"))\n    audit.update(selection_scope="TRAIN_ONLY", proxy="L2_logistic_on_log1p_temporal_sums",\n                 proxy_weighting="inverse_patient_snapshot_count", proxy_is_transformer_optimum=False,\n                 summary_names=names, outer_test_diagnostics_computed=False)\n    return FeatureSelectionResult(selected, state.eligible_indices, shortlist, tags, ranking, quality, cv_scores,\n                                  summary, fold_rankings, pd.DataFrame(audits), state, choices["proxy_one_se"], audit, cfg)\n\n\ndef feature_diagnostics(X, metadata, feature_map, state, valid=None, threshold=0.90, max_features=128):\n    """TRAIN latest-patient correlations; no O(F) VIF regressions or target filtering.\n\n    Correlation uses annual counts, so complementary temporal shape can remain.\n    Effective rank/condition are bounded to the first max_features original-order\n    eligible columns and are explicitly a diagnostic submatrix, not global VIF.\n    """\n    _require(0 < threshold <= 1 and max_features > 1, "Invalid redundancy settings.")\n    meta = pd.DataFrame(metadata).reset_index(drop=True)\n    train = meta.loc[meta.SPLIT.eq("train")].copy()\n    train["_date"] = pd.to_datetime(train.END_DT)\n    rows = train.sort_values(["PATIENT_ID", "_date"]).groupby("PATIENT_ID", sort=False).tail(1).index.to_numpy()\n    summary, _ = temporal_count_summaries(X, state, rows, valid)\n    values = summary[:, -1, :]\n    frame = pd.DataFrame(values).rank(method="average").to_numpy(dtype=float)\n    centered = frame - frame.mean(axis=0)\n    norms = np.sqrt((centered ** 2).sum(axis=0))\n    safe = norms > 0\n    centered[:, safe] /= norms[safe]\n    centered[:, ~safe] = 0\n    corr = centered.T @ centered\n    a, b = np.where(np.triu(np.abs(corr) >= threshold, k=1))\n    fm = _feature_map(feature_map, state.n_features).set_index("FEATURE_INDEX")\n    pairs = pd.DataFrame(dict(FEATURE_INDEX_A=state.eligible_indices[a], FEATURE_INDEX_B=state.eligible_indices[b],\n                              SPEARMAN=corr[a, b]))\n    for side in ("A", "B"):\n        pairs["FEATURE_NAME_" + side] = pairs["FEATURE_INDEX_" + side].map(fm.FEATURE_NAME)\n    sub = centered[:, :min(max_features, centered.shape[1])]\n    eigenvalues = np.maximum(np.linalg.eigvalsh(sub.T @ sub), 0)\n    positive = eigenvalues[eigenvalues > 1e-10]\n    entropy_rank = 0.0\n    if len(positive):\n        probabilities = positive / positive.sum()\n        entropy_rank = float(np.exp(-np.sum(probabilities * np.log(probabilities))))\n    return dict(correlation_pairs=pairs.sort_values("SPEARMAN", key=lambda x: x.abs(), ascending=False),\n                summary=dict(scope="TRAIN_latest_snapshot_per_patient", patients=len(rows),\n                             correlated_pairs=len(pairs), spectral_columns=sub.shape[1],\n                             spectral_original_indices=state.eligible_indices[:sub.shape[1]].tolist(),\n                             effective_rank=entropy_rank, singular=bool(len(positive) < sub.shape[1]),\n                             nonzero_eigenvalue_condition=float(positive.max() / positive.min()) if len(positive) else None,\n                             global_vif_computed=False, redundancy_auto_removal=False))\n\n\ndef shift_summary(X, metadata, feature_map, state, valid=None, comparison="validation"):\n    """Descriptive TRAIN vs VALIDATION only, never a feature-selection input."""\n    _require(comparison == "validation", "TEST shift diagnostics are forbidden before the evaluation lock.")\n    meta = pd.DataFrame(metadata).reset_index(drop=True)\n    fm = _feature_map(feature_map, state.n_features)\n    values = {}\n    for split in ("train", comparison):\n        rows = np.flatnonzero(meta.SPLIT.to_numpy() == split)\n        _require(len(rows) > 0, "Missing shift population: " + split)\n        summaries, _ = temporal_count_summaries(X, state, rows, valid)\n        annual = summaries[:, -1, :]\n        weights = patient_snapshot_weights(meta.PATIENT_ID.to_numpy()[rows])\n        mean = np.average(annual, axis=0, weights=weights)\n        variance = np.average((annual - mean) ** 2, axis=0, weights=weights)\n        values[split] = (mean, variance, np.average(annual > 0, axis=0, weights=weights))\n    train_mean, train_var, train_presence = values["train"]\n    val_mean, val_var, val_presence = values[comparison]\n    denominator = np.sqrt((train_var + val_var) / 2)\n    standardized = np.divide(val_mean - train_mean, denominator, out=np.zeros_like(denominator), where=denominator > 0)\n    result = fm.iloc[state.eligible_indices].copy()\n    result["TRAIN_LOG_ANNUAL_MEAN"] = train_mean\n    result["VALIDATION_LOG_ANNUAL_MEAN"] = val_mean\n    result["STANDARDIZED_MEAN_DIFFERENCE"] = standardized\n    result["TRAIN_NONZERO_FRACTION"] = train_presence\n    result["VALIDATION_NONZERO_FRACTION"] = val_presence\n    result["USED_FOR_SELECTION"] = False\n    return result.reset_index(drop=True)\n', 'training': '"""Auditable, bounded model search. Inputs are private raw count tensors.\n\nThis module never receives TEST rows during fitting. Each run is a complete\nTRAIN/VALIDATION experiment; seeds are repetitions, not selection candidates.\n"""\nimport copy\nimport hashlib\nimport io\nimport json\nimport random\nimport time\n\nimport numpy as np\nfrom sklearn.metrics import average_precision_score, roc_auc_score, log_loss\n\n\ndef stable_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),\n                                    allow_nan=False).encode()).hexdigest()\n\n\ndef seed_run(seed):\n    import torch\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.use_deterministic_algorithms(True)\n    torch.backends.cudnn.benchmark = False\n\n\ndef sample_weights(groups, mode="snapshot"):\n    groups = np.asarray(groups)\n    if groups.ndim != 1 or len(groups) == 0 or any(v is None for v in groups):\n        raise ValueError("Expected nonempty patient groups")\n    if mode == "snapshot":\n        return np.ones(len(groups), dtype=np.float32)\n    if mode != "patient":\n        raise ValueError("Unknown weighting unit")\n    _, inv, counts = np.unique(groups, return_inverse=True, return_counts=True)\n    w = 1. / counts[inv]\n    return (w / w.mean()).astype(np.float32)\n\n\ndef class_weight(y, weights, balance):\n    y, weights = np.asarray(y), np.asarray(weights)\n    if (y.ndim != 1 or weights.shape != y.shape or set(np.unique(y)) != {0, 1}\n            or not np.isfinite(weights).all() or (weights <= 0).any()\n            or balance not in {"none", "sqrt", "full"}):\n        raise ValueError("Class weights require aligned binary labels, positive weights, and a known balance mode")\n    ratio = float(weights[y == 0].sum() / weights[y == 1].sum())\n    return {"none": 1., "sqrt": ratio ** .5, "full": ratio}[balance]\n\n\ndef raw_representation(x, representation):\n    """No zero-derived padding. Missing measurements remain missing."""\n    x = np.asarray(x)\n    if (x.ndim != 3 or x.shape[1] != 12 or min(x.shape) < 1 or\n            not np.issubdtype(x.dtype, np.number) or np.iscomplexobj(x)):\n        raise ValueError("Expected N x 12 x F nonnegative counts; NaN means missing")\n    # Validate before float32 conversion so fractions cannot round into integers.\n    # A row batch avoids several full-size boolean copies of the 1 GiB tensor.\n    for start in range(0, len(x), 256):\n        block = x[start:start+256]\n        if np.isinf(block).any() or (block < 0).any():\n            raise ValueError("Expected finite nonnegative counts; NaN means missing")\n        finite = block[np.isfinite(block)]\n        if (finite != np.floor(finite)).any():\n            raise ValueError("Fractional input is not a raw integer count")\n    x = np.asarray(x, dtype=np.float32)\n    if representation == "monthly":\n        return x.copy()\n    if representation == "without_step0":\n        out = x.copy()\n        out[:, 0, :] = 0\n        return out\n    if representation == "quarterly":\n        # np.sum deliberately propagates missing constituent months.\n        return x.reshape(len(x), 4, 3, x.shape[2]).sum(axis=2)\n    if representation == "flattened":\n        return x.reshape(len(x), -1).copy()\n    if representation == "annual":\n        return x.sum(axis=1)\n    if representation == "windows":\n        return np.concatenate([x[:, :n].sum(axis=1) for n in (1, 3, 6, 12)], axis=1)\n    raise ValueError("Unknown representation")\n\n\ndef fit_numeric_transform(raw, scale=False, log=True):\n    raw = np.asarray(raw)\n    if raw.ndim not in (2, 3) or min(raw.shape) < 1 or np.isinf(raw).any() or (log and (raw < 0).any()):\n        raise ValueError("Invalid fitting measurements")\n    a = np.log1p(raw) if log else raw.copy()\n    flat = a.reshape(-1, a.shape[-1])\n    # Train-only medians; all-missing fields remain 0 and are explicitly counted.\n    import warnings\n    with warnings.catch_warnings():\n        warnings.simplefilter("ignore", RuntimeWarning)\n        median = np.nanmedian(flat, axis=0)\n    all_missing = np.isnan(median)\n    median[all_missing] = 0\n    fitted = np.where(np.isnan(flat), median, flat)\n    center = fitted.mean(axis=0) if scale else np.zeros(flat.shape[1])\n    # A unit floor avoids exploding rare standardized counts.\n    train_sd = fitted.std(axis=0)\n    divisor = np.maximum(train_sd, 1.) if scale else np.ones(flat.shape[1])\n    return dict(median=median.tolist(), center=center.tolist(), divisor=divisor.tolist(),\n                log=bool(log), scale=bool(scale), missing_cells=int(np.isnan(raw).sum()),\n                all_missing_fields=int(all_missing.sum()), fitted_on="TRAIN only",\n                training_transformed_sd=(train_sd / divisor).tolist())\n\n\ndef numeric_transform(raw, state):\n    raw = np.asarray(raw)\n    for key in ("median", "center", "divisor"):\n        vector = np.asarray(state[key])\n        if vector.shape != (raw.shape[-1],) or not np.isfinite(vector).all():\n            raise ValueError("Preprocessing state width/values do not match model input")\n    if (np.asarray(state["divisor"]) <= 0).any():\n        raise ValueError("Preprocessing divisor must be positive")\n    a = np.log1p(raw) if state["log"] else np.asarray(raw).copy()\n    a = np.where(np.isnan(a), np.asarray(state["median"]), a)\n    result = (a - np.asarray(state["center"])) / np.asarray(state["divisor"])\n    if not np.isfinite(result).all():\n        raise ValueError("Nonfinite transformed model input")\n    return result.astype(np.float32)\n\n\ndef _resolved_config(config, input_dim):\n    """Record actual defaults so the serialized experiment is reproducible."""\n    cfg = copy.deepcopy(config)\n    if "columns" in cfg:\n        cfg["columns"] = np.asarray(cfg["columns"]).tolist()\n    if cfg.get("family") not in {"transformer", "lightgbm"}:\n        raise ValueError("Unsupported candidate family")\n    if cfg.get("representation") not in {"monthly", "quarterly", "without_step0", "flattened", "annual", "windows"}:\n        raise ValueError("Unknown representation")\n    defaults = dict(scale=False, weighting="snapshot", balance="none", columns=list(range(input_dim)))\n    if cfg["family"] == "transformer":\n        defaults.update(d_model=64, heads=4, ff=128, dropout=.2, depth=1, pooling="mean",\n                        projection_l1=0., explicit_l2=0., weight_decay=1e-3,\n                        learning_rate=3e-4, batch_size=256, scheduler="plateau",\n                        optimizer_betas=[.9, .999], optimizer_eps=1e-8, gradient_clip=1.)\n    else:\n        defaults.update(n_estimators=600, learning_rate=.03, num_leaves=15, max_depth=-1,\n                        min_child_samples=50, colsample_bytree=.8, subsample=.8,\n                        reg_lambda=1., reg_alpha=0., early_stopping_rounds=30)\n    unknown = set(cfg) - set(defaults) - {"family", "representation", "subset", "drop_step0"}\n    if unknown:\n        raise ValueError("Unknown candidate settings: " + ", ".join(sorted(unknown)))\n    cfg = {**defaults, **cfg}\n    if "drop_step0" in cfg and not isinstance(cfg["drop_step0"], bool):\n        raise ValueError("drop_step0 must be boolean")\n    if cfg["weighting"] not in {"snapshot", "patient"} or cfg["balance"] not in {"none", "sqrt", "full"}:\n        raise ValueError("Unknown weighting/balance mode")\n    if not isinstance(cfg["scale"], bool):\n        raise ValueError("scale must be boolean")\n    if len(cfg["columns"]) != input_dim or len(set(cfg["columns"])) != input_dim:\n        raise ValueError("Configured original columns must uniquely match the selected tensor width")\n    if not np.isfinite(cfg["learning_rate"]) or cfg["learning_rate"] <= 0:\n        raise ValueError("Learning rate must be finite and positive")\n    if cfg["family"] == "transformer":\n        if cfg["representation"] not in {"monthly", "quarterly", "without_step0"}:\n            raise ValueError("Sequence architecture needs sequence representation")\n        for key in ("d_model", "heads", "ff", "depth", "batch_size"):\n            if type(cfg[key]) is not int or cfg[key] < 1:\n                raise ValueError(key + " must be a positive integer")\n        if cfg["d_model"] < 2 or cfg["d_model"] % cfg["heads"]:\n            raise ValueError("Invalid embedding/head dimensions")\n        if cfg["pooling"] not in {"mean", "attention"} or cfg["scheduler"] not in {"plateau", "none"}:\n            raise ValueError("Unknown pooling/scheduler")\n        if not 0 <= cfg["dropout"] < 1:\n            raise ValueError("Dropout must be in [0,1)")\n        for key in ("projection_l1", "explicit_l2", "weight_decay"):\n            if not np.isfinite(cfg[key]) or cfg[key] < 0:\n                raise ValueError(key + " must be finite and nonnegative")\n        if cfg["weight_decay"] and cfg["explicit_l2"]:\n            raise ValueError("Choose explicit L2 or AdamW decay, never both; set weight_decay=0 explicitly")\n        if (len(cfg["optimizer_betas"]) != 2 or not all(0 <= b < 1 for b in cfg["optimizer_betas"])\n                or not np.isfinite(cfg["optimizer_eps"]) or cfg["optimizer_eps"] <= 0\n                or not np.isfinite(cfg["gradient_clip"]) or cfg["gradient_clip"] <= 0):\n            raise ValueError("Invalid optimizer/clipping configuration")\n    else:\n        if cfg["representation"] not in {"flattened", "annual", "windows"}:\n            raise ValueError("LightGBM needs a tabular representation")\n        for key in ("n_estimators", "num_leaves", "min_child_samples", "early_stopping_rounds"):\n            if type(cfg[key]) is not int or cfg[key] < 1:\n                raise ValueError(key + " must be a positive integer")\n        if cfg["num_leaves"] < 2 or not all(0 < cfg[k] <= 1 for k in ("colsample_bytree", "subsample")):\n            raise ValueError("Invalid tree leaf/subsampling configuration")\n        if type(cfg["max_depth"]) is not int or cfg["max_depth"] == 0 or cfg["max_depth"] < -1:\n            raise ValueError("max_depth must be -1 or a positive integer")\n        if any(not np.isfinite(cfg[k]) or cfg[k] < 0 for k in ("reg_alpha", "reg_lambda")):\n            raise ValueError("Tree penalties must be finite and nonnegative")\n    return cfg\n\n\ndef _configured_representation(x, config):\n    # The declared ablation must also work for direct model restoration, without\n    # requiring an undocumented preprocessing operation by its caller.\n    if config.get("drop_step0", False):\n        x = raw_representation(x, "without_step0")\n    return raw_representation(x, config["representation"])\n\n\ndef make_encoder(input_dim, config):\n    import torch\n    from torch import nn\n    d = int(config.get("d_model", 64))\n    heads = int(config.get("heads", 4))\n    if heads < 1 or d % heads or d < 2:\n        raise ValueError("Invalid embedding/head dimensions")\n    if config.get("pooling", "mean") not in {"mean", "attention"}:\n        raise ValueError("Unknown pooling strategy")\n\n    class CountEncoder(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.projection = nn.Linear(input_dim, d)\n            drop = float(config.get("dropout", .2))\n            self.input_dropout = nn.Dropout(drop)\n            layer = nn.TransformerEncoderLayer(d, heads, int(config.get("ff", 128)),\n                       drop, activation="gelu", batch_first=True, norm_first=True)\n            self.encoder = nn.TransformerEncoder(layer, int(config.get("depth", 1)),\n                                                 enable_nested_tensor=False)\n            for layer in self.encoder.layers:\n                for p in layer.parameters():\n                    if p.ndim > 1:\n                        nn.init.xavier_uniform_(p)\n            self.norm = nn.LayerNorm(d)\n            self.pooling = config.get("pooling", "mean")\n            self.pool_score = nn.Linear(d, 1) if self.pooling == "attention" else None\n            self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(d, 1))\n            pos = torch.arange(12, dtype=torch.float32)[:, None]\n            frequency = torch.exp(torch.arange(0, d, 2) * (-np.log(10000.) / d))\n            pe = torch.zeros(12, d)\n            pe[:, 0::2] = torch.sin(pos * frequency)\n            pe[:, 1::2] = torch.cos(pos * frequency[:pe[:, 1::2].shape[1]])\n            self.register_buffer("position", pe)\n\n        def forward(self, x):\n            z = self.input_dropout(self.projection(x)) + self.position[:x.shape[1]]\n            z = self.norm(self.encoder(z))\n            pooled = (z * self.pool_score(z).softmax(1)).sum(1) if self.pool_score is not None else z.mean(1)\n            return self.head(pooled).squeeze(-1)\n\n    return CountEncoder()\n\n\ndef penalties(model, config):\n    """L1 = lambda1*mean(abs(projection W)); L2 = lambda2*mean(matrix W**2).\n\n    Biases and layer-normalization vectors are excluded from L2; sinusoidal\n    positions are fixed buffers. These terms are recomputed each optimizer step.\n    Normalization by parameter count is intentional and logged; coefficients are\n    not numerically equivalent to the unnormalized AdamW decay coefficient.\n    """\n    if any(not np.isfinite(config.get(k, 0.)) or config.get(k, 0.) < 0\n           for k in ("projection_l1", "explicit_l2", "weight_decay")):\n        raise ValueError("Regularization coefficients must be finite and nonnegative")\n    l1 = float(config.get("projection_l1", 0.)) * model.projection.weight.abs().mean()\n    l2_coef = float(config.get("explicit_l2", 0.))\n    if l2_coef and config.get("weight_decay", 0.):\n        raise ValueError("Choose explicit L2 or AdamW decay, never both")\n    matrices = [p for p in model.parameters() if p.ndim > 1]\n    l2 = l2_coef * sum(p.square().sum() for p in matrices) / sum(p.numel() for p in matrices)\n    return l1, l2\n\n\ndef quick_metrics(y, scores):\n    y, scores = np.asarray(y), np.asarray(scores)\n    if (y.ndim != 1 or scores.shape != y.shape or set(np.unique(y)) != {0, 1}\n            or not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any()):\n        raise ValueError("Metrics need finite scores and both classes")\n    return dict(average_precision=float(average_precision_score(y, scores)),\n                roc_auc=float(roc_auc_score(y, scores)),\n                log_loss=float(log_loss(y, np.clip(scores, 1e-7, 1-1e-7))))\n\n\ndef predict_encoder(model, values, device="cpu", batch_size=512):\n    import torch\n    model.eval()\n    result = []\n    with torch.inference_mode():\n        for start in range(0, len(values), batch_size):\n            scores = model(torch.from_numpy(values[start:start+batch_size]).to(device)).sigmoid()\n            result.append(scores.cpu().numpy())\n    return np.concatenate(result)\n\n\ndef train_candidate(x_train, y_train, groups_train, x_validation, y_validation,\n                    config, seed, *, epochs=30, patience=5, threads=8, device="cpu", groups_validation=None):\n    """Fit TRAIN/VALIDATION only; callers must enforce frozen split membership.\n\n    There is deliberately no TEST argument. Optional validation patient IDs allow\n    this function to independently verify patient disjointness as well.\n    """\n    import torch\n    x_train, x_validation = np.asarray(x_train), np.asarray(x_validation)\n    y_train, y_validation, groups_train = np.asarray(y_train), np.asarray(y_validation), np.asarray(groups_train)\n    if (x_train.ndim != 3 or x_validation.ndim != 3 or x_train.shape[1:] != x_validation.shape[1:]\n            or y_train.ndim != 1 or y_validation.ndim != 1 or groups_train.ndim != 1\n            or len(x_train) != len(y_train) or len(groups_train) != len(y_train)\n            or len(x_validation) != len(y_validation)):\n        raise ValueError("TRAIN alignment mismatch")\n    config = _resolved_config(config, x_train.shape[-1])\n    if groups_validation is not None:\n        groups_validation = np.asarray(groups_validation)\n        if groups_validation.shape != y_validation.shape or set(groups_train) & set(groups_validation):\n            raise ValueError("TRAIN/VALIDATION patient overlap or validation group mismatch")\n    if set(np.unique(y_train)) != {0, 1} or set(np.unique(y_validation)) != {0, 1}:\n        raise ValueError("TRAIN and VALIDATION need both labels")\n    if any(type(n) is not int or n < 1 for n in (epochs, patience, threads)) or type(seed) is not int or seed < 0:\n        raise ValueError("Invalid training limits")\n    started = time.perf_counter()\n    seed_run(seed)\n    torch.set_num_threads(threads)\n    family = config["family"]\n    representation = config["representation"]\n    raw_train = _configured_representation(x_train, config)\n    raw_val = _configured_representation(x_validation, config)\n    w = sample_weights(groups_train, config.get("weighting", "snapshot"))\n    positive_weight = class_weight(y_train, w, config.get("balance", "none"))\n    transform = fit_numeric_transform(raw_train, scale=config.get("scale", False),\n                                     log=family != "lightgbm")\n    a = numeric_transform(raw_train, transform)\n    b = numeric_transform(raw_val, transform)\n    del raw_train, raw_val\n    history, best_epoch = [], None\n    if family == "transformer":\n        from torch import nn\n        from torch.utils.data import DataLoader, TensorDataset\n        if a.ndim != 3:\n            raise ValueError("Sequence architecture needs sequence representation")\n        model = make_encoder(a.shape[-1], config).to(device)\n        wd = float(config.get("weight_decay", 1e-3))\n        if wd and config.get("explicit_l2", 0):\n            raise ValueError("Do not combine explicit L2 with AdamW decay")\n        # Biases and normalization are not decayed; matrices share one convention.\n        param_groups = [{"params": [p for p in model.parameters() if p.ndim > 1], "weight_decay": wd},\n                        {"params": [p for p in model.parameters() if p.ndim <= 1], "weight_decay": 0.}]\n        optimizer = torch.optim.AdamW(param_groups, lr=config["learning_rate"], betas=tuple(config["optimizer_betas"]), eps=config["optimizer_eps"])\n        scheduler = (torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=.5, patience=2)\n                     if config["scheduler"] == "plateau" else None)\n        loss_fn = nn.BCEWithLogitsLoss(reduction="none", pos_weight=torch.tensor(positive_weight, device=device))\n        loader = DataLoader(TensorDataset(torch.from_numpy(a), torch.as_tensor(y_train, dtype=torch.float32),\n                                         torch.from_numpy(w)), batch_size=config.get("batch_size", 256),\n                            shuffle=True, generator=torch.Generator().manual_seed(seed), num_workers=0)\n        best_ap, stale, best_state = -np.inf, 0, None\n        for epoch in range(1, epochs+1):\n            model.train()\n            total, n, penalty_total, l1_total, l2_total = 0., 0, 0., 0., 0.\n            for xx, yy, ww in loader:\n                xx, yy, ww = xx.to(device), yy.to(device), ww.to(device)\n                optimizer.zero_grad(set_to_none=True)\n                l1, l2 = penalties(model, config)\n                # Weights are normalized over the complete training population.\n                loss = (loss_fn(model(xx), yy) * ww).mean() + l1 + l2\n                if not torch.isfinite(loss):\n                    raise ValueError("Nonfinite optimization loss")\n                loss.backward()\n                nn.utils.clip_grad_norm_(model.parameters(), config["gradient_clip"], error_if_nonfinite=True)\n                optimizer.step()\n                total += float(loss.detach()) * len(xx)\n                penalty_total += float((l1+l2).detach()) * len(xx)\n                l1_total += float(l1.detach()) * len(xx)\n                l2_total += float(l2.detach()) * len(xx)\n                n += len(xx)\n            val_scores = predict_encoder(model, b, device)\n            train_scores = predict_encoder(model, a, device)\n            vm, tm = quick_metrics(y_validation, val_scores), quick_metrics(y_train, train_scores)\n            history.append(dict(epoch=epoch, optimization_loss=total/n, regularization_loss=penalty_total/n,\n                                projection_l1_loss=l1_total/n, explicit_l2_loss=l2_total/n,\n                                train_ap=tm["average_precision"], validation_ap=vm["average_precision"],\n                                train_log_loss=tm["log_loss"], validation_log_loss=vm["log_loss"],\n                                learning_rate=optimizer.param_groups[0]["lr"]))\n            if vm["average_precision"] > best_ap:\n                best_ap, best_epoch, stale = vm["average_precision"], epoch, 0\n                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n            else:\n                stale += 1\n            if scheduler is not None:\n                scheduler.step(vm["average_precision"])\n            if stale >= patience:\n                break\n        model.load_state_dict(best_state, strict=True)\n        train_scores, val_scores = predict_encoder(model, a, device), predict_encoder(model, b, device)\n        buffer = io.BytesIO()\n        torch.save(best_state, buffer)\n        model_blob = buffer.getvalue()\n        parameter_count = sum(p.numel() for p in model.parameters())\n        importance = model.projection.weight.detach().cpu().norm(dim=0).tolist()\n        artifact_format = "torch_state_dict_weights_only"\n    elif family == "lightgbm":\n        import lightgbm as lgb\n        if a.ndim != 2:\n            raise ValueError("LightGBM needs a tabular representation")\n        model = lgb.LGBMClassifier(n_estimators=config.get("n_estimators", 600),\n                    learning_rate=config.get("learning_rate", .03), num_leaves=config.get("num_leaves", 15),\n                    max_depth=config.get("max_depth", -1), min_child_samples=config.get("min_child_samples", 50),\n                    colsample_bytree=config.get("colsample_bytree", .8), subsample=config.get("subsample", .8),\n                    subsample_freq=1, reg_lambda=config.get("reg_lambda", 1.),\n                    reg_alpha=config.get("reg_alpha", 0.), scale_pos_weight=positive_weight,\n                    random_state=seed, n_jobs=threads, deterministic=True, force_col_wise=True, verbosity=-1)\n        metric_history = {}\n        def ap_metric(y, p):\n            return "average_precision", float(average_precision_score(y, p)), True\n        model.set_params(metric="None")\n        model.fit(a, y_train, sample_weight=w, eval_set=[(b, y_validation)], eval_metric=ap_metric,\n                  callbacks=[lgb.early_stopping(config["early_stopping_rounds"], first_metric_only=True, verbose=False),\n                             lgb.record_evaluation(metric_history)])\n        best_epoch = int(model.best_iteration_)\n        train_scores = model.booster_.predict(a, num_iteration=best_epoch)\n        val_scores = model.booster_.predict(b, num_iteration=best_epoch)\n        history = [dict(epoch=i+1, validation_ap=float(v))\n                   for i, v in enumerate(metric_history["valid_0"]["average_precision"])]\n        model_blob = model.booster_.model_to_string(num_iteration=best_epoch).encode()\n        parameter_count = int(sum(t["num_leaves"] for t in model.booster_.dump_model()["tree_info"]))\n        importance = model.booster_.feature_importance(importance_type="gain").tolist()\n        artifact_format = "lightgbm_text"\n    else:\n        raise ValueError("Unsupported candidate family")\n    importance_blocks = 1 if family == "transformer" else len(importance) // x_train.shape[-1]\n    original_importance = np.asarray(importance).reshape(importance_blocks, x_train.shape[-1]).sum(axis=0).tolist()\n    summary = dict(config=config, seed=int(seed), training_complete=True, best_epoch=best_epoch,\n                   epochs_observed=len(history), validation_evaluations=len(history),\n                   train_metrics=quick_metrics(y_train, train_scores),\n                   validation_metrics=quick_metrics(y_validation, val_scores),\n                   generalization_gap_ap=float(average_precision_score(y_train, train_scores) -\n                                               average_precision_score(y_validation, val_scores)),\n                   positive_weight=positive_weight, parameter_count=parameter_count,\n                   complexity_unit="parameters" if family == "transformer" else "tree_leaves",\n                   wall_seconds=time.perf_counter()-started, artifact_format=artifact_format,\n                   model_sha256=hashlib.sha256(model_blob).hexdigest(), transform=transform,\n                   transform_sha256=stable_hash(transform), config_sha256=stable_hash(config),\n                   raw_input_shape=list(x_train.shape[1:]),\n                   patient_disjointness_verified=groups_validation is not None,\n                   regularization_definition=("L1=lambda1*mean(abs(input_projection)); explicit_L2=lambda2*mean(all_matrix_weights_squared); AdamW decay excludes biases/norm" if family == "transformer" else "LightGBM native reg_alpha/reg_lambda"),\n                   importance=importance, aggregated_original_importance=original_importance,\n                   importance_blocks_per_original_feature=importance_blocks,\n                   importance_definition=("input_projection_column_L2_norm; model-internal diagnostic, not held-out reliance" if family == "transformer" else "LightGBM gain; summed across temporal summary blocks per original feature; not held-out reliance"),\n                   test_inference_performed=False,\n                   selection_scope="Repeated frozen VALIDATION; results are development estimates")\n    return dict(summary=summary, history=history, model_blob=model_blob,\n                train_scores=train_scores, validation_scores=val_scores)\n\n\ndef make_saved_predictor(result, *, device="cpu"):\n    """Restore once for frozen diagnostics; never fit or update preprocessing.\n\n    Call only after a persisted decision lock when using TEST. Reuse the\n    returned predictor for bounded validation perturbations to avoid repeated\n    checkpoint loading. Copy metadata so later caller mutations cannot change\n    the restored input contract.\n    """\n    import torch\n    s = result["summary"]\n    blob = result["model_blob"]\n    if hashlib.sha256(blob).hexdigest() != s["model_sha256"]:\n        raise ValueError("Model artifact hash mismatch")\n    cfg = copy.deepcopy(s["config"])\n    if stable_hash(cfg) != s["config_sha256"] or stable_hash(s["transform"]) != s["transform_sha256"]:\n        raise ValueError("Config/preprocessing artifact hash mismatch")\n    transform = copy.deepcopy(s["transform"])\n    raw_shape = tuple(s["raw_input_shape"])\n    if cfg["family"] == "transformer":\n        model = make_encoder(raw_shape[-1], cfg).to(device)\n        state = torch.load(io.BytesIO(blob), map_location=device, weights_only=True)\n        model.load_state_dict(state, strict=True)\n        model.eval()\n    elif cfg["family"] == "lightgbm":\n        import lightgbm as lgb\n        model = lgb.Booster(model_str=blob.decode())\n    else:\n        raise ValueError("Unsupported saved model family")\n\n    def predict(x):\n        if tuple(np.shape(x)[1:]) != raw_shape:\n            raise ValueError("Saved model raw input shape mismatch")\n        a = numeric_transform(_configured_representation(x, cfg), transform)\n        if cfg["family"] == "transformer":\n            return predict_encoder(model, a, device)\n        return model.predict(a)\n    return predict\n\n\ndef predict_saved(result, x, *, device="cpu"):\n    """One-shot restored prediction; keep TEST behind the persisted lock."""\n    return make_saved_predictor(result, device=device)(x)\n\n\ndef choose_finalists(results, expected_candidates, seeds):\n    """Mean seed AP wins; no picking one lucky seed. No uncertainty-based equivalence claim."""\n    if (not expected_candidates or len(set(expected_candidates)) != len(expected_candidates)\n            or not seeds or len(set(seeds)) != len(seeds)):\n        raise ValueError("Declare nonempty unique candidates and seeds")\n    if set(results) != set(expected_candidates):\n        raise ValueError("Search incomplete: do not select from a partially finished grid")\n    rows = []\n    for name in expected_candidates:\n        runs = results[name]\n        if sorted(r["summary"]["seed"] for r in runs) != sorted(seeds):\n            raise ValueError("Missing/duplicate seeds")\n        if not all(r["summary"]["training_complete"] for r in runs):\n            raise ValueError("Incomplete model run")\n        cfg = runs[0]["summary"]["config"]\n        if any(r["summary"]["config"] != cfg for r in runs):\n            raise ValueError("Seed repetitions have different configurations")\n        values = [r["summary"]["validation_metrics"]["average_precision"] for r in runs]\n        if not np.isfinite(values).all() or not all(0 <= value <= 1 for value in values):\n            raise ValueError("Invalid validation AP; cannot select a winner")\n        rows.append(dict(candidate=name, family=cfg["family"], mean_validation_ap=float(np.mean(values)),\n                         sd_validation_ap=float(np.std(values, ddof=1)) if len(values)>1 else None,\n                         n_features=len(cfg["columns"]),\n                         mean_complexity=float(np.mean([r["summary"]["parameter_count"] for r in runs])),\n                         fits=len(runs), validation_evaluations=sum(r["summary"]["validation_evaluations"] for r in runs)))\n    winners = {}\n    for family in sorted(set(r["family"] for r in rows)):\n        family_rows = [r for r in rows if r["family"] == family]\n        # Parsimony breaks exact score ties; non-significance is not equivalence.\n        winner = sorted(family_rows, key=lambda r: (-r["mean_validation_ap"], r["n_features"],\n                                                   r["mean_complexity"], r["candidate"]))[0]\n        winners[family] = winner["candidate"]\n    return winners, rows\n', 'workflow': '"""Stage contracts, immutable private artifacts and declared search plan.\n\nThe delivery builder embeds these helpers and the previously tested warehouse\nartifact codec. No repository imports are needed in the delivered notebooks.\n"""\nimport base64\nimport copy\nimport hashlib\nimport io\nimport json\nimport platform\nimport re\nfrom datetime import datetime, timezone\n\nimport numpy as np\nimport pandas as pd\n\n\ndef json_bytes(value):\n    def default(v):\n        if isinstance(v, np.ndarray):\n            return v.tolist()\n        if isinstance(v, np.generic):\n            return v.item()\n        raise TypeError(type(v).__name__)\n    return json.dumps(value, sort_keys=True, separators=(",", ":"),\n                      allow_nan=False, default=default).encode()\n\n\ndef array_bytes(array):\n    buffer = io.BytesIO()\n    np.save(buffer, array, allow_pickle=False)\n    return buffer.getvalue()\n\n\ndef frame_bytes(frame):\n    return frame.to_csv(index=False).encode()\n\n\ndef runtime_versions():\n    import importlib.metadata\n    versions = {}\n    for package in ("numpy", "pandas", "scipy", "scikit-learn", "torch", "lightgbm"):\n        try:\n            versions[package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            versions[package] = "UNAVAILABLE"\n    return dict(python=platform.python_version(), libraries=versions)\n\n\ndef load_count_inputs():\n    """Stream saved rows into a driver memmap; preserve NaNs and original order.\n\n    Structural/value assertions do not use held-out distributions or scores.\n    Never print or export patient rows from this function.\n    """\n    import tempfile\n    from pathlib import Path\n    from pyspark.sql import functions as F\n    mapping = [r.asDict() for r in read_sf("FEATURE_MAP").orderBy("FEATURE_INDEX").collect()]\n    if len(mapping) != 1028:\n        raise ValueError("Full 1,028 candidate universe required")\n    features = [r["FEATURE_NAME"] for r in mapping]\n    aliases = [r["FEATURE_COLUMN"] for r in mapping]\n    if [r["FEATURE_INDEX"] for r in mapping] != list(range(1028)) or aliases != [f"F{i:04d}" for i in range(1028)]:\n        raise ValueError("Unexpected FEATURE_MAP index/alias order")\n    source = checked_metadata(read_sf("SNAPSHOTS").select("PATIENT_ID", "END_DT", "RESP").toPandas())\n    frozen = checked_metadata(read_sf("PATIENT_SPLIT").select(\n        "PATIENT_ID", "END_DT", "RESP", "SPLIT", "SPLIT_CONFIG").toPandas(), include_split=True)\n    if not source.equals(frozen[["PATIENT_ID", "END_DT", "RESP"]]):\n        raise ValueError("Frozen cohort key or label mismatch")\n    expected = {"train": (16256, 8712, 941), "validation": (3481, 1867, 202), "test": (3414, 1868, 202)}\n    for split, (n, p, positives) in expected.items():\n        part = frozen[frozen.SPLIT == split]\n        if (len(part), part.PATIENT_ID.nunique(), int(part.RESP.sum())) != (n, p, positives):\n            raise ValueError("Frozen population changed")\n    monthly = read_sf("TENSOR_MONTHLY")\n    if set(monthly.columns) != set(["PATIENT_ID", "END_DT", "RESP", "TIME_STEP"] + aliases):\n        raise ValueError("Tensor schema differs from FEATURE_MAP")\n    # Cast to double first: do not round fractions or turn negatives into missing.\n    grouped = (monthly.groupBy("PATIENT_ID", "END_DT", "RESP")\n               .agg(F.collect_list(F.struct(F.col("TIME_STEP").alias("T"),\n                    F.array(*[F.col(name).cast("double") for name in aliases]).alias("V"))).alias("SEQUENCE")))\n    temporary = tempfile.TemporaryDirectory(prefix="tak861_counts_")\n    X = np.memmap(Path(temporary.name) / "counts.f32", dtype="<f4", mode="w+", shape=(len(frozen), 12, 1028))\n    positions = {(r.PATIENT_ID, r.END_DT): i for i, r in enumerate(frozen.itertuples())}\n    seen = np.zeros(len(frozen), dtype=bool)\n    try:\n        for row in grouped.toLocalIterator(prefetchPartitions=False):\n            key = (row["PATIENT_ID"], pd.Timestamp(row["END_DT"]).strftime("%Y-%m-%d"))\n            if key not in positions:\n                raise ValueError("Tensor contains an unexpected snapshot")\n            i = positions[key]\n            seq = sorted(row["SEQUENCE"], key=lambda r: r["T"])\n            if seen[i] or row["RESP"] != frozen.RESP.iloc[i] or [r["T"] for r in seq] != list(range(12)):\n                raise ValueError("Tensor duplicate, label mismatch or invalid timesteps")\n            values = np.array([r["V"] for r in seq], dtype=np.float64)\n            finite = np.isfinite(values)\n            if values.shape != (12, 1028) or np.isinf(values).any() or (values[finite] < 0).any() or np.any(values[finite] != np.floor(values[finite])):\n                raise ValueError("Invalid raw counts; missing counts may be NaN")\n            # Exact count storage: reject float32 integer precision loss rather than silently round.\n            converted = values.astype(np.float32)\n            if np.any(converted[finite].astype(np.float64) != values[finite]):\n                raise ValueError("Counts exceed exact float32 representation")\n            X[i] = converted\n            seen[i] = True\n        if not seen.all():\n            raise ValueError("Missing tensor snapshots")\n        X.flush()\n        X.flags.writeable = False\n    except BaseException:\n        X._mmap.close()\n        temporary.cleanup()\n        raise\n    hashes = input_fingerprints(X, frozen, features)\n    creation = json.loads(frozen.SPLIT_CONFIG.iloc[0])\n    original_order_hash = hashlib.sha256(json.dumps(features, ensure_ascii=False).encode()).hexdigest()\n    historical_order_match = creation.get("feature_order_sha256") == original_order_hash\n    return dict(X=X, y=frozen.RESP.to_numpy(dtype=np.int64), metadata=frozen,\n                features=features, feature_map=pd.DataFrame(mapping), hashes=hashes,\n                indices={s: np.flatnonzero(frozen.SPLIT.to_numpy() == s) for s in expected},\n                temporary_directory=temporary, split_creation=creation,\n                historical_split_feature_hash_matches_current=historical_order_match)\n\n\ndef release_count_inputs(data):\n    if data and isinstance(data.get("X"), np.memmap):\n        data["X"]._mmap.close()\n        data["temporary_directory"].cleanup()\n\n\ndef preparation_summary(data, implementation_hash):\n    summary = data["metadata"].groupby("SPLIT").agg(snapshots=("RESP", "size"),\n                         patients=("PATIENT_ID", "nunique"), positives=("RESP", "sum")).reset_index()\n    return dict(format_version=1, input_hashes=data["hashes"], tensor_shape=list(data["X"].shape),\n                feature_map=data["feature_map"].to_dict("records"), cohort=summary.to_dict("records"),\n                implementation_hash=implementation_hash,\n                historical_split_feature_hash_matches_current=data["historical_split_feature_hash_matches_current"],\n                orientation="0 newest: recovered declaration; historical execution still unresolved",\n                coverage="All 12 saved positions retained. Zero does not establish observation or padding.",\n                provenance="Reproduced structural checks on saved inputs, not event-time validation",\n                historical_test="Previously inspected; any future evaluation is retrospective",\n                value_contract="nonnegative integer or missing; no infinities; exact float32 storage")\n\n\ndef verify_stage(data, report, implementation_hash):\n    if report["input_hashes"] != data["hashes"] or report["implementation_hash"] != implementation_hash:\n        raise ValueError("Inputs/code changed since prior stage; use a new run ID and rerun in order")\n\n\ndef build_protocol(candidate_indices, candidate_tags, input_hashes, implementation_hash):\n    """Grid resources are chosen before model validation; k comes from TRAIN shortlist."""\n    subsets = {k: sorted(map(int, v)) for k, v in candidate_indices.items()}\n    if "all_eligible" not in subsets or not subsets["all_eligible"]:\n        raise ValueError("All-eligible control is mandatory")\n    for values in subsets.values():\n        if not values or len(values) != len(set(values)) or not set(values) <= set(subsets["all_eligible"]):\n            raise ValueError("Invalid feature shortlist")\n    if any(name not in subsets for name in candidate_tags.values()):\n        raise ValueError("Candidate tags reference unavailable TRAIN subsets")\n    # Phase-B subset is not selected by the screening proxy. Its family-specific\n    # count anchor is resolved from complete phase-A seed means under this rule.\n    default_subset = "__PHASE_A_FAMILY_ANCHOR__"\n    pairs = []\n    for name in sorted(subsets):\n        pairs.append(("count_" + name, name, {}, {}))\n    # One changed factor per Transformer arm; each tabular fit is separately declared.\n    pairs.extend([\n        ("capacity", default_subset, dict(d_model=128, depth=2, ff=256), dict(num_leaves=31)),\n        ("dropout", default_subset, dict(dropout=.4), dict(colsample_bytree=.6)),\n        ("decay", default_subset, dict(weight_decay=.01), dict(reg_lambda=10.)),\n        ("sqrt_weight", default_subset, dict(balance="sqrt"), dict(balance="sqrt")),\n        ("full_weight", default_subset, dict(balance="full"), dict(balance="full")),\n        ("projection_l1", default_subset, dict(projection_l1=1e-5), dict(reg_alpha=.1)),\n        ("explicit_l2", default_subset, dict(weight_decay=0., explicit_l2=1e-4), dict(min_child_samples=100)),\n        ("attention_pool", default_subset, dict(pooling="attention"), dict(learning_rate=.015)),\n        ("scale", default_subset, dict(scale=True), dict(representation="windows")),\n        ("quarterly", default_subset, dict(representation="quarterly"), dict(representation="annual")),\n        ("patient_weight", default_subset, dict(weighting="patient"), dict(weighting="patient")),\n        # All-eligible prevents the step-0 sensitivity depending on a label-ranked subset.\n        ("without_step0", "all_eligible", dict(representation="without_step0"), dict(drop_step0=True)),\n    ])\n    candidates = {}\n    for arm, subset, tf_changes, gb_changes in pairs:\n        declared_columns = subsets[subset] if subset in subsets else None\n        tf = dict(family="transformer", representation="monthly", subset=subset, columns=declared_columns,\n                  d_model=64, depth=1, heads=4, ff=128, dropout=.2, pooling="mean", scale=False,\n                  weight_decay=.001, explicit_l2=0., projection_l1=0., balance="none",\n                  weighting="snapshot", learning_rate=.0003, batch_size=256)\n        gb = dict(family="lightgbm", representation="flattened", subset=subset, columns=declared_columns,\n                  scale=False, num_leaves=15, min_child_samples=50, colsample_bytree=.8, subsample=.8,\n                  reg_lambda=1., reg_alpha=0., balance="none", weighting="snapshot", learning_rate=.03,\n                  n_estimators=600)\n        tf.update(tf_changes)\n        gb.update(gb_changes)\n        candidates["tf_"+arm], candidates["gb_"+arm] = tf, gb\n    candidates = {name: candidates[name] for name in sorted(candidates)}\n    phase_a = [name for name in sorted(candidates) if name.startswith(("tf_count_", "gb_count_"))]\n    phase_b = [name for name in sorted(candidates) if name not in phase_a]\n    return dict(protocol_version=3, search_stage="design", seeds=[42, 142, 242], candidates=candidates,\n                feature_subsets=subsets, candidate_tags=dict(candidate_tags), phase_a=phase_a, phase_b=phase_b,\n                count_anchor_rule="Within each family, highest mean VALIDATION AP across every prespecified seed; exact ties prefer fewer features, then lower mean complexity, then candidate name",\n                phase_b_rule="Resolve each family\'s ablation subset from its phase-A count winner; without_step0 stays all-eligible. Persist hashed resolution before any phase-B fit",\n                artifact_candidate_order=sorted(candidates),\n                prediction_counts={"train":16256, "validation":3481},\n                epochs=30, patience=5, cpu_threads=8, device="cpu",\n                reliance_diagnostic=dict(max_individual_features=30, top_train=10, top_internal=10,\n                    lowest_train=5, random_remaining=5, probe_seed=20261006, repeat_seeds=[271,811,1297],\n                    split="validation", population="latest snapshot per patient", strata="END_DT calendar month",\n                    correlation_threshold=.9, max_correlation_groups=10,\n                    groups=["selected TRAIN ranking terciles", "DX", "PX", "RX"], selection_changes=False),\n                input_hashes=input_hashes, implementation_hash=implementation_hash,\n                objective="Max mean per-seed frozen VALIDATION average precision, exact ties prefer simpler",\n                threshold_rule="F1 maximized on ensemble VALIDATION; top10% budget rule separately reported",\n                evaluation_estimand="Snapshot-level primary; latest-patient secondary; patient-cluster uncertainty",\n                final_predictor="Uniform average of all three prespecified seed predictions; never seed selection",\n                historical_test="Previously inspected: retrospective final evaluation only",\n                upstream_provenance="Unresolved; no leakage-free certification from these experiments",\n                budget="Same number of candidate fits and seeds per family; actual validation calls/seconds reported",\n                restrictions=["Proxy scores shortlist only", "No OOT inference without complete inputs and provenance",\n                              "Validation selection uncertainty is descriptive, not independent confirmation",\n                              "Dropping step0 is sensitivity, not a proven fix for all source leakage"])\n\n\ndef candidate_views(data, cfg, split):\n    rows = data["indices"][split]\n    # Slice rows first: callers never pass held-out rows during development.\n    x = np.asarray(data["X"][rows])[:, :, cfg["columns"]].copy()\n    if cfg.get("drop_step0", False):\n        x[:, 0, :] = 0\n    return x, data["y"][rows], data["metadata"].iloc[rows].PATIENT_ID.to_numpy()\n\n\ndef run_artifact_table(run_prefix, candidate_index, seed):\n    if (not re.fullmatch(r"[A-Z][A-Z0-9_]*", run_prefix)\n            or type(candidate_index) is not int or candidate_index < 0\n            or type(seed) is not int or seed < 0):\n        raise ValueError("Invalid run artifact identity")\n    return f"{run_prefix}_C{candidate_index:03d}_S{seed}"\n\n\ndef _payload_hash(payload):\n    return hashlib.sha256(payload).hexdigest()\n\n\ndef _validate_run(result, contract, expected_lengths=None, expected_config=None):\n    """Validate both count and encoded-49 run contracts without model unpickling."""\n    summary = result["summary"]\n    if summary.get("training_complete") is not True or summary.get("seed") != contract.get("seed"):\n        raise ValueError("Incomplete run or wrong seed")\n    if (summary.get("model_sha256") != _payload_hash(result["model_blob"])\n            or summary.get("config_sha256") != _payload_hash(json_bytes(summary.get("config")))\n            or summary.get("transform_sha256") != _payload_hash(json_bytes(summary.get("transform")))):\n        raise ValueError("Model/config/transform hash mismatch")\n    if "protocol_hash" in summary and summary["protocol_hash"] != contract.get("protocol_hash"):\n        raise ValueError("Run summary has another protocol identity")\n    if "input_hashes" in contract and summary.get("input_hashes") != contract["input_hashes"]:\n        raise ValueError("Run input identity mismatch")\n    if expected_config is not None:\n        config = summary["config"]\n        if any(key not in config or json_bytes(config[key]) != json_bytes(value)\n               for key, value in expected_config.items()):\n            raise ValueError("Saved configuration differs from the declared candidate")\n        if ("candidate_config_sha256" in contract and\n                contract["candidate_config_sha256"] != _payload_hash(json_bytes(expected_config))):\n            raise ValueError("Candidate configuration hash mismatch")\n    shape = summary.get("raw_input_shape")\n    if (not isinstance(shape, list) or len(shape) not in (1, 2)\n            or any(type(size) is not int or size <= 0 for size in shape)):\n        raise ValueError("Missing or invalid raw input shape")\n    if not isinstance(result["history"], list) or not result["history"]:\n        raise ValueError("A completed run requires its training history")\n    lengths = expected_lengths if expected_lengths is not None else contract.get("prediction_counts")\n    for split in ("train", "validation"):\n        values = np.asarray(result[split+"_scores"])\n        if (values.ndim != 1 or len(values) == 0 or not np.issubdtype(values.dtype, np.number)\n                or np.iscomplexobj(values) or not np.isfinite(values).all()\n                or ((values < 0) | (values > 1)).any()):\n            raise ValueError("Saved predictions must be nonempty aligned finite probability vectors")\n        if lengths is not None and (split not in lengths or len(values) != lengths[split]):\n            raise ValueError("Saved prediction length does not match the frozen split")\n        metric = summary.get(split+"_metrics", {}).get("average_precision")\n        if metric is None or not np.isfinite(metric) or not 0 <= metric <= 1:\n            raise ValueError("Missing or invalid completed-run average precision")\n\n\ndef pack_run(result, contract):\n    _validate_run(result, contract)\n    artifacts = {"summary.json": json_bytes(result["summary"]), "history.json": json_bytes(result["history"]),\n            "model.bin": result["model_blob"], "train.npy": array_bytes(result["train_scores"]),\n            "validation.npy": array_bytes(result["validation_scores"])}\n    envelope = dict(format_version=2, contract=contract,\n                    payload_sha256={name:_payload_hash(value) for name,value in artifacts.items()})\n    artifacts["contract.json"] = json_bytes(envelope)\n    return artifacts\n\n\nRUN_ARTIFACT_NAMES = {"summary.json", "history.json", "model.bin", "train.npy", "validation.npy", "contract.json"}\n\n\ndef unpack_run(artifacts, contract, expected_lengths=None, expected_config=None):\n    if set(artifacts) != RUN_ARTIFACT_NAMES:\n        raise ValueError("Incomplete run artifact bundle")\n    envelope = json.loads(artifacts["contract.json"])\n    if envelope.get("format_version") != 2 or envelope.get("contract") != contract:\n        raise ValueError("Run contract changed; never mix trials from different inputs/protocols")\n    expected_payloads = RUN_ARTIFACT_NAMES - {"contract.json"}\n    hashes = envelope.get("payload_sha256", {})\n    if set(hashes) != expected_payloads or any(hashes[name] != _payload_hash(artifacts[name]) for name in expected_payloads):\n        raise ValueError("Run artifact payload hash mismatch")\n    result = dict(summary=json.loads(artifacts["summary.json"]), history=json.loads(artifacts["history.json"]),\n                model_blob=artifacts["model.bin"], train_scores=np.load(io.BytesIO(artifacts["train.npy"]), allow_pickle=False),\n                validation_scores=np.load(io.BytesIO(artifacts["validation.npy"]), allow_pickle=False))\n    _validate_run(result, contract, expected_lengths, expected_config)\n    return result\n\n\ndef _candidate_names(protocol):\n    names = sorted(protocol["candidates"])\n    seeds = protocol["seeds"]\n    if not names or not seeds or len(set(seeds)) != len(seeds) or any(type(s) is not int or s < 0 for s in seeds):\n        raise ValueError("Declare nonempty candidates and unique valid seeds")\n    if protocol.get("artifact_candidate_order", names) != names:\n        raise ValueError("Artifact candidate order must be the deterministic sorted order")\n    return names\n\n\ndef _run_contract(protocol, name, seed):\n    design = protocol.get("design_protocol", protocol)\n    contract = dict(protocol_hash=stable_hash(design), candidate=name, seed=seed,\n                candidate_config_sha256=_payload_hash(json_bytes(protocol["candidates"][name])),\n                input_hashes=protocol["input_hashes"], implementation_hash=protocol["implementation_hash"],\n                prediction_counts=protocol["prediction_counts"])\n    if protocol.get("protocol_version") == 3:\n        if name in design["phase_a"]:\n            contract["phase"] = "A"\n        elif name in design["phase_b"] and protocol.get("search_stage") == "resolved":\n            contract.update(phase="B", resolution_sha256=protocol["resolution_sha256"])\n        else:\n            raise ValueError("Phase-B fit requires a persisted resolved anchor")\n    return contract\n\n\ndef _execute_candidates(data, protocol, run_prefix, requested_names):\n    """Internal phase executor; candidate table indices never depend on phase order."""\n    if protocol["input_hashes"] != data["hashes"]:\n        raise ValueError("Protocol input hash mismatch")\n    lengths = {split:len(data["indices"][split]) for split in ("train", "validation")}\n    if protocol.get("prediction_counts") != lengths:\n        raise ValueError("Protocol population lengths differ from frozen inputs")\n    all_names = _candidate_names(protocol)\n    if len(requested_names) != len(set(requested_names)) or not set(requested_names) <= set(all_names):\n        raise ValueError("Requested phase candidates differ from declared candidates")\n    results = {}\n    for name in requested_names:\n        number = all_names.index(name)\n        cfg = protocol["candidates"][name]\n        runs = []\n        for seed in protocol["seeds"]:\n            table = run_artifact_table(run_prefix, number, seed)\n            contract = _run_contract(protocol, name, seed)\n            if table_exists(table):\n                result = unpack_run(read_artifacts(table, RUN_ARTIFACT_NAMES), contract, lengths, cfg)\n            else:\n                xtr, ytr, groups = candidate_views(data, cfg, "train")\n                xv, yv, validation_groups = candidate_views(data, cfg, "validation")\n                result = train_candidate(xtr, ytr, groups, xv, yv, cfg, seed,\n                           epochs=protocol["epochs"], patience=protocol["patience"],\n                           threads=protocol["cpu_threads"], device=protocol["device"],\n                           groups_validation=validation_groups)\n                result["summary"]["protocol_hash"] = contract["protocol_hash"]\n                result["summary"]["input_hashes"] = data["hashes"]\n                _validate_run(result, contract, lengths, cfg)\n                save_artifacts(table, pack_run(result, contract))\n                del xtr, xv\n            runs.append(result)\n            print(name, "seed", seed, "VAL AP", round(result["summary"]["validation_metrics"]["average_precision"], 6),\n                  "seconds", round(result["summary"]["wall_seconds"], 1), flush=True)\n        results[name] = runs\n    return results\n\n\ndef execute_search(data, protocol, run_prefix):\n    """Legacy single-phase executor. Version 3 must use the registered two phases."""\n    if protocol.get("protocol_version") == 3:\n        raise ValueError("Use execute_two_phase_search for the registered count-anchor design")\n    return _execute_candidates(data, protocol, run_prefix, _candidate_names(protocol))\n\n\ndef _validate_design(design):\n    if design.get("protocol_version") != 3 or design.get("search_stage") != "design" or "design_protocol" in design:\n        raise ValueError("Expected immutable version-3 DESIGN protocol")\n    names = _candidate_names(design)\n    a, b = design.get("phase_a", []), design.get("phase_b", [])\n    if (not a or not b or a != sorted(set(a)) or b != sorted(set(b))\n            or set(a) & set(b) or set(a) | set(b) != set(names)):\n        raise ValueError("Every candidate must belong to exactly one declared phase")\n    subsets = design.get("feature_subsets", {})\n    if not subsets.get("all_eligible"):\n        raise ValueError("All-eligible feature control is required")\n    by_family = {}\n    for name in a:\n        cfg = design["candidates"][name]\n        family, subset = cfg.get("family"), cfg.get("subset")\n        if family not in {"transformer", "lightgbm"} or subset not in subsets or cfg.get("columns") != subsets[subset]:\n            raise ValueError("Phase A requires explicit TRAIN-derived count subsets")\n        by_family.setdefault(family, []).append(subset)\n    if set(by_family) != {"transformer", "lightgbm"} or any(sorted(v) != sorted(subsets) for v in by_family.values()):\n        raise ValueError("Both families require exactly every prespecified phase-A count arm")\n    for family in by_family:\n        settings = [{k:v for k,v in design["candidates"][name].items() if k not in {"columns", "subset"}}\n                    for name in a if design["candidates"][name]["family"] == family]\n        if any(settings[0] != other for other in settings[1:]):\n            raise ValueError("Phase-A count arms must share each family\'s base configuration")\n    b_counts = {family: 0 for family in by_family}\n    for name in b:\n        cfg = design["candidates"][name]\n        family = cfg.get("family")\n        if family not in b_counts:\n            raise ValueError("Unknown phase-B family")\n        b_counts[family] += 1\n        if name.endswith("without_step0"):\n            if cfg.get("subset") != "all_eligible" or cfg.get("columns") != subsets["all_eligible"]:\n                raise ValueError("Step-0 sensitivity must retain all eligible features")\n        elif cfg.get("subset") != "__PHASE_A_FAMILY_ANCHOR__" or cfg.get("columns") is not None:\n            raise ValueError("Phase-B ablations must be unresolved family-anchor templates")\n    if len(set(b_counts.values())) != 1:\n        raise ValueError("Both families require equal phase-B candidate and seed budgets")\n\n\ndef _verify_saved_design(design, run_prefix):\n    _validate_design(design)\n    if not table_exists(run_prefix + "_PROTOCOL"):\n        raise ValueError("Persist the immutable DESIGN protocol before phase-A fitting")\n    saved = read_artifacts(run_prefix + "_PROTOCOL", {"protocol.json", "runtime.json"})\n    if json.loads(saved["protocol.json"]) != design:\n        raise ValueError("Saved DESIGN differs; use a new run identifier")\n\n\ndef resolve_count_anchors(design, phase_a_results):\n    """Pure resolution: complete seed means, no TEST, no CI-based equivalence rule."""\n    _validate_design(design)\n    if set(phase_a_results) != set(design["phase_a"]):\n        raise ValueError("Cannot resolve count anchors from an incomplete phase A")\n    rows, evidence = [], []\n    for name in design["phase_a"]:\n        cfg = design["candidates"][name]\n        runs = phase_a_results[name]\n        if sorted(r["summary"]["seed"] for r in runs) != sorted(design["seeds"]):\n            raise ValueError("Every prespecified seed must complete exactly once before anchor resolution")\n        values, complexity = [], []\n        for result in sorted(runs, key=lambda r: r["summary"]["seed"]):\n            summary = result["summary"]\n            contract = _run_contract(design, name, summary["seed"])\n            _validate_run(result, contract, expected_config=cfg)\n            values.append(float(summary["validation_metrics"]["average_precision"]))\n            size = summary.get("parameter_count")\n            if size is None or not np.isfinite(size) or size < 0:\n                raise ValueError("Finite model complexity is required for declared exact-tie handling")\n            complexity.append(float(size))\n            evidence.append(dict(candidate=name, seed=summary["seed"], summary_sha256=_payload_hash(json_bytes(summary)),\n                                 model_sha256=summary["model_sha256"],\n                                 validation_predictions_sha256=_payload_hash(array_bytes(result["validation_scores"]))))\n        rows.append(dict(candidate=name, family=cfg["family"], subset=cfg["subset"],\n                         columns=cfg["columns"], n_features=len(cfg["columns"]),\n                         seed_validation_ap=values, mean_validation_ap=float(np.mean(values)),\n                         mean_complexity=float(np.mean(complexity))))\n    anchors = {}\n    for family in ("transformer", "lightgbm"):\n        eligible = [row for row in rows if row["family"] == family]\n        winner = sorted(eligible, key=lambda r: (-r["mean_validation_ap"], r["n_features"],\n                                               r["mean_complexity"], r["candidate"]))[0]\n        anchors[family] = copy.deepcopy(winner)\n    return dict(resolution_version=1, design_sha256=stable_hash(design),\n                rule=design["count_anchor_rule"], anchors=anchors, phase_a_summary=rows,\n                phase_a_evidence=evidence, selection_split="validation", test_used=False,\n                interpretation="Registered adaptive phase-B anchor; VALIDATION is development evidence, not independent confirmation")\n\n\ndef _resolved_protocol(design, resolution):\n    if resolution.get("design_sha256") != stable_hash(design):\n        raise ValueError("Anchor resolution belongs to another DESIGN")\n    resolved = copy.deepcopy(design)\n    for name in design["phase_b"]:\n        cfg = resolved["candidates"][name]\n        if name.endswith("without_step0"):\n            continue\n        anchor = resolution["anchors"][cfg["family"]]\n        cfg["subset"], cfg["columns"] = anchor["subset"], list(anchor["columns"])\n    resolved.update(search_stage="resolved", design_protocol=copy.deepcopy(design),\n                    design_sha256=stable_hash(design), resolution_sha256=stable_hash(resolution),\n                    count_anchor_resolution=copy.deepcopy(resolution))\n    return resolved\n\n\ndef _read_candidates(protocol, run_prefix, names):\n    results = {}\n    all_names = _candidate_names(protocol)\n    for name in names:\n        i = all_names.index(name)\n        results[name] = [unpack_run(read_artifacts(run_artifact_table(run_prefix, i, seed), RUN_ARTIFACT_NAMES),\n                        _run_contract(protocol, name, seed), expected_config=protocol["candidates"][name])\n                        for seed in protocol["seeds"]]\n    return results\n\n\ndef read_resolved_protocol(design, run_prefix):\n    """Recompute anchors from hashed phase-A artifacts, then verify saved resolution."""\n    _verify_saved_design(design, run_prefix)\n    phase_a_results = _read_candidates(design, run_prefix, design["phase_a"])\n    expected = resolve_count_anchors(design, phase_a_results)\n    stored = json.loads(read_artifacts(run_prefix + "_ANCHORS", {"resolution.json"})["resolution.json"])\n    if stored != expected:\n        raise ValueError("Saved anchor resolution differs from complete phase-A evidence")\n    return _resolved_protocol(design, stored)\n\n\ndef execute_two_phase_search(data, design, run_prefix):\n    """Run count arms, freeze anchors, then run ablations on each family\'s winner."""\n    _verify_saved_design(design, run_prefix)\n    phase_a_results = _execute_candidates(data, design, run_prefix, design["phase_a"])\n    resolution = resolve_count_anchors(design, phase_a_results)\n    table = run_prefix + "_ANCHORS"\n    if table_exists(table):\n        existing = json.loads(read_artifacts(table, {"resolution.json"})["resolution.json"])\n        if existing != resolution:\n            raise ValueError("Existing count anchors differ; never overwrite resolution or mix phase-B trials")\n    else:\n        # This immutable write must finish before any phase-B model sees data.\n        save_artifacts(table, {"resolution.json": json_bytes(resolution)})\n    resolved = read_resolved_protocol(design, run_prefix)\n    phase_b_results = _execute_candidates(data, resolved, run_prefix, design["phase_b"])\n    combined = {**phase_a_results, **phase_b_results}\n    return resolved, {name: combined[name] for name in _candidate_names(resolved)}\n\n\ndef read_search(protocol, run_prefix):\n    if protocol.get("protocol_version") == 3:\n        if protocol.get("search_stage") != "resolved" or "design_protocol" not in protocol:\n            raise ValueError("Load read_resolved_protocol(DESIGN, prefix) before reading the two-phase search")\n        recovered = read_resolved_protocol(protocol["design_protocol"], run_prefix)\n        if recovered != protocol:\n            raise ValueError("Resolved protocol differs from saved phase-A evidence")\n    return _read_candidates(protocol, run_prefix, _candidate_names(protocol))\n\n\ndef ensemble_scores(runs, split):\n    if not runs or split not in {"train", "validation"}:\n        raise ValueError("Development ensembles require complete TRAIN/VALIDATION runs")\n    seeds = [run["summary"]["seed"] for run in runs]\n    if len(set(seeds)) != len(seeds) or any(run["summary"].get("training_complete") is not True for run in runs):\n        raise ValueError("Ensemble contains duplicate seeds or incomplete runs")\n    arrays = [r[split+"_scores"] for r in runs]\n    if (len({a.shape for a in arrays}) != 1 or\n            any(a.ndim != 1 or not np.isfinite(a).all() or ((a < 0) | (a > 1)).any() for a in arrays)):\n        raise ValueError("Seed score alignment mismatch")\n    return np.mean(arrays, axis=0)\n', 'lineage': '"""Bounded TRAIN-only signals about saved count sequences, never lineage proof.\n\nExact overlapping-month equality can arise from correct windows, copied values,\nstable activity or no activity. It cannot establish historical orientation or an\nevent cutoff. E05B did not audit all count values; NaN is unknown, not zero. A\nmask from the separate 49-feature representation cannot certify this tensor.\n"""\nimport hashlib\nimport json\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score\n\n\ndef _require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef _inputs(X, metadata, feature_map, valid):\n    _require(isinstance(X, np.ndarray) and X.ndim == 3,\n             "X must be a raw-count ndarray/memmap in (snapshot, step, feature) order.")\n    _require(X.shape[1] >= 4 and X.shape[2] > 0, "Need at least four steps and one feature.")\n    _require(np.issubdtype(X.dtype, np.number) and not np.iscomplexobj(X),\n             "Raw counts must have a real numeric dtype.")\n    meta = pd.DataFrame(metadata)\n    _require(len(meta) == len(X) and {"SPLIT", "PATIENT_ID", "END_DT"} <= set(meta),\n             "Metadata needs aligned SPLIT, PATIENT_ID and END_DT columns.")\n    # Only the split selector is read across all rows. Non-TRAIN dates, IDs,\n    # labels, masks and tensor values are never inspected by these diagnostics.\n    rows = np.flatnonzero(meta.SPLIT.astype(str).str.lower().eq("train").to_numpy())\n    _require(len(rows) > 0, "No TRAIN rows available.")\n    train = meta.iloc[rows].copy().reset_index(drop=True)\n    _require(train.PATIENT_ID.map(lambda v: isinstance(v, str) and bool(v)).all(),\n             "TRAIN patient IDs must be nonempty strings.")\n    dates = pd.to_datetime(train.END_DT, errors="raise")\n    _require(not dates.isna().any() and dates.dt.tz is None,\n             "TRAIN END_DT must contain nonmissing timezone-naive dates.")\n    train["END_DT"] = dates\n    _require(not train[["PATIENT_ID", "END_DT"]].duplicated().any(),\n             "Duplicate TRAIN snapshot identity.")\n    train["_source_row"] = rows\n    train["_month"] = dates.dt.year * 12 + dates.dt.month\n    fmap = pd.DataFrame(feature_map).copy().reset_index(drop=True)\n    _require({"FEATURE_INDEX", "FEATURE_NAME", "FEATURE_COLUMN"} <= set(fmap),\n             "Feature map needs FEATURE_INDEX, FEATURE_NAME and FEATURE_COLUMN.")\n    _require(len(fmap) == X.shape[2] and\n             np.array_equal(fmap.FEATURE_INDEX.to_numpy(), np.arange(X.shape[2])),\n             "Feature map must already match exact zero-based tensor order.")\n    for col in ("FEATURE_NAME", "FEATURE_COLUMN"):\n        _require(fmap[col].map(lambda v: isinstance(v, str) and bool(v)).all()\n                 and fmap[col].is_unique, "Feature names and aliases must be unique strings.")\n    if valid is not None:\n        _require(isinstance(valid, np.ndarray) and valid.dtype == bool\n                 and valid.shape == X.shape[:2], "valid must be a boolean (N,T) array.")\n    return train, fmap\n\n\ndef _counts(block):\n    """Validate only an already permitted TRAIN slice; preserve missing values."""\n    _require(not np.isinf(block).any(), "Infinite value in inspected TRAIN counts.")\n    observed = block[np.isfinite(block)]\n    _require(not ((observed < 0) | (observed != np.floor(observed))).any(),\n             "Inspected raw TRAIN counts must be nonnegative integers or NaN.")\n\n\ndef _partial_month_risk(train):\n    dates = train.END_DT\n    remaining = dates.dt.days_in_month - dates.dt.day\n    return {\n        "status": "INCONCLUSIVE",\n        "train_snapshots": int(len(train)),\n        "anchors_before_calendar_month_end": int((remaining > 0).sum()),\n        "possible_post_anchor_days_if_whole_month_min": int(remaining.min()),\n        "possible_post_anchor_days_if_whole_month_max": int(remaining.max()),\n        "event_cutoff_verified": False,\n        "whole_month_aggregation_verified": False,\n        "reason": "Calendar dates describe exposure to risk only; saved counts do not establish event cutoffs.",\n        "without_step0_arm_advisable": True,\n        "arm_condition": "Predeclare without_step0 sensitivity while cutoff is unresolved; it cannot prove leakage removal.",\n    }\n\n\ndef _pairs(train, steps, limit):\n    candidates = []\n    for patient, group in train.groupby("PATIENT_ID", sort=True):\n        ordered = group.sort_values("END_DT", kind="stable")\n        records = list(ordered[["_source_row", "_month", "END_DT"]].itertuples(index=False, name=None))\n        for earlier, later in zip(records, records[1:]):\n            delta = int(later[1] - earlier[1])\n            if not 1 <= delta <= steps - 3:\n                continue  # At least one common interior month must remain.\n            identity = json.dumps([patient, earlier[2].isoformat(), later[2].isoformat()],\n                                  ensure_ascii=True, separators=(",", ":"))\n            digest = hashlib.sha256(identity.encode()).hexdigest()\n            candidates.append((digest, int(earlier[0]), int(later[0]), delta))\n    selected = sorted(candidates)[:limit]\n    # A combined fingerprint permits repeatability checks without exporting IDs.\n    fingerprint = hashlib.sha256("\\n".join(x[0] for x in selected).encode()).hexdigest()\n    return selected, len(candidates), fingerprint\n\n\ndef overlap_month_consistency(X, metadata, feature_map, valid=None, max_pairs=200,\n                              min_informative_pairs=3, min_varying_comparisons=20,\n                              agreement_threshold=0.99):\n    """Test both month-order hypotheses on <=200 deterministic adjacent TRAIN pairs.\n\n    The first and last positions are excluded in *both* snapshots. Consequently\n    no hypothesized current partial month or oldest window boundary participates.\n    Comparisons with NaN or a supplied invalid position are excluded explicitly.\n    Status describes only observed agreement on active, temporally varying count\n    cells. A high agreement status never certifies orientation or historical fit.\n    """\n    _require(type(max_pairs) is int and 1 <= max_pairs <= 200, "max_pairs must be in 1..200.")\n    _require(type(min_informative_pairs) is int and min_informative_pairs > 0,\n             "min_informative_pairs must be positive.")\n    _require(type(min_varying_comparisons) is int and min_varying_comparisons > 0,\n             "min_varying_comparisons must be positive.")\n    _require(0 < agreement_threshold <= 1, "agreement_threshold must be in (0,1].")\n    train, fmap = _inputs(X, metadata, feature_map, valid)\n    pairs, candidate_count, fingerprint = _pairs(train, X.shape[1], max_pairs)\n    results = {}\n    for orientation in ("newest_first", "oldest_first"):\n        counts = dict(finite_comparisons=0, equal_comparisons=0, active_comparisons=0,\n                      active_equal_comparisons=0, varying_active_comparisons=0,\n                      varying_active_equal_comparisons=0, missing_comparisons=0,\n                      masked_comparisons=0, pair_count=0, informative_pair_count=0)\n        for _, earlier, later, delta in pairs:\n            base = np.arange(1, X.shape[1] - 1 - delta)\n            old_steps, new_steps = ((base, base + delta) if orientation == "newest_first"\n                                    else (base + delta, base))\n            a, b = X[earlier, old_steps, :], X[later, new_steps, :]\n            _counts(a)\n            _counts(b)\n            step_ok = (np.ones(len(base), bool) if valid is None\n                       else valid[earlier, old_steps] & valid[later, new_steps])\n            finite = np.isfinite(a) & np.isfinite(b) & step_ok[:, None]\n            equal = (a == b) & finite\n            active = ((a > 0) | (b > 0)) & finite\n            # Stable nonzero columns cannot distinguish copied/static snapshots.\n            varying_feature = np.zeros(len(fmap), bool)\n            for j in np.flatnonzero(active.any(axis=0)):\n                values_a, values_b = a[finite[:, j], j], b[finite[:, j], j]\n                varying_feature[j] = ((len(values_a) > 1 and np.ptp(values_a) > 0)\n                                      or (len(values_b) > 1 and np.ptp(values_b) > 0))\n            varying = active & varying_feature[None, :]\n            counts["finite_comparisons"] += int(finite.sum())\n            counts["equal_comparisons"] += int(equal.sum())\n            counts["active_comparisons"] += int(active.sum())\n            counts["active_equal_comparisons"] += int((active & equal).sum())\n            counts["varying_active_comparisons"] += int(varying.sum())\n            counts["varying_active_equal_comparisons"] += int((varying & equal).sum())\n            counts["missing_comparisons"] += int(((~np.isfinite(a) | ~np.isfinite(b)) & step_ok[:, None]).sum())\n            counts["masked_comparisons"] += int((~step_ok).sum()) * len(fmap)\n            counts["pair_count"] += int(finite.any())\n            counts["informative_pair_count"] += int(varying.any())\n        for name, num, den in (("agreement_ratio", "equal_comparisons", "finite_comparisons"),\n                               ("active_agreement_ratio", "active_equal_comparisons", "active_comparisons"),\n                               ("varying_active_agreement_ratio", "varying_active_equal_comparisons", "varying_active_comparisons")):\n            counts[name] = counts[num] / counts[den] if counts[den] else None\n        informative = (counts["informative_pair_count"] >= min_informative_pairs\n                       and counts["varying_active_comparisons"] >= min_varying_comparisons)\n        counts["status"] = ("INCONCLUSIVE" if not informative else\n                            "CORROBORATED" if counts["varying_active_agreement_ratio"] >= agreement_threshold\n                            else "CONTRADICTED")\n        counts["interpretation"] = ("Insufficient active temporal variation; equality is uninformative."\n                                    if not informative else "Observed interior-month consistency only; not historical orientation or cutoff proof.")\n        results[orientation] = counts\n    return {\n        "scope": "TRAIN_ONLY_BOUNDED_INTERIOR_MONTH_CONSISTENCY",\n        "candidate_adjacent_pair_count": candidate_count,\n        "sampled_patient_pair_count": len(pairs),\n        "max_pairs": max_pairs,\n        "sample_fingerprint": fingerprint,\n        "sampling": "SHA256-ordered adjacent source TRAIN snapshots; independent of labels and counts",\n        "excluded_positions": [0, X.shape[1] - 1],\n        "thresholds": dict(min_informative_pairs=min_informative_pairs,\n                           min_varying_comparisons=min_varying_comparisons,\n                           agreement_threshold=agreement_threshold),\n        "mask_source": "all_positions_retained_no_coverage_claim" if valid is None else "caller_supplied_not_certified_here",\n        "hypotheses": results,\n        "orientation_status": "INCONCLUSIVE",\n        "orientation_verified": False,\n        "historical_provenance_verified": False,\n        "partial_month_risk": _partial_month_risk(train),\n        "limitation": "Agreement can reflect stable/copy/no-activity behavior. Even discriminating hypotheses cannot certify source execution or whole-month cutoffs.",\n    }\n\n\ndef step0_label_screen(X, metadata, feature_map, valid=None, feature_batch_size=128):\n    """Per-feature physical step-0/step-1 associations on paired observed TRAIN rows.\n\n    Returns a DataFrame sorted by descending AP difference, then positive-class\n    nonzero-rate difference. Both AP values use exactly the same complete cases\n    with equal total weight per represented patient. Missing counts are reported\n    and excluded, not assumed absent or imputed. No p-values or leakage verdicts\n    are produced; these supervised descriptive screens are not feature selection.\n    """\n    _require(type(feature_batch_size) is int and feature_batch_size > 0,\n             "feature_batch_size must be positive.")\n    train, fmap = _inputs(X, metadata, feature_map, valid)\n    _require("RESP" in train and train.RESP.isin([0, 1]).all(), "TRAIN RESP must be binary and nonmissing.")\n    rows, y = train._source_row.to_numpy(), train.RESP.to_numpy(dtype=np.int8)\n    ids = train.PATIENT_ID.to_numpy()\n    step_ok = (np.ones(len(rows), bool) if valid is None\n               else valid[rows, 0] & valid[rows, 1])\n    records = []\n    for start in range(0, len(fmap), feature_batch_size):\n        indices = np.arange(start, min(start + feature_batch_size, len(fmap)))\n        block = X[np.ix_(rows, np.array([0, 1]), indices)]\n        _counts(block)\n        for offset, j in enumerate(indices):\n            a, b = block[:, 0, offset], block[:, 1, offset]\n            observed = np.isfinite(a) & np.isfinite(b) & step_ok\n            yy, aa, bb, patients = y[observed], a[observed], b[observed], ids[observed]\n            positive, negative = yy == 1, yy == 0\n            weights = np.empty(0)\n            if len(patients):\n                _, inverse, multiplicity = np.unique(patients, return_inverse=True, return_counts=True)\n                weights = 1.0 / multiplicity[inverse]\n            estimable = bool(positive.any() and negative.any())\n            ap0 = float(average_precision_score(yy, aa, sample_weight=weights)) if estimable else None\n            ap1 = float(average_precision_score(yy, bb, sample_weight=weights)) if estimable else None\n            def rate(values, subset):\n                return float(np.average(values[subset] > 0, weights=weights[subset])) if subset.any() else None\n            pos0, pos1 = rate(aa, positive), rate(bb, positive)\n            row = fmap.iloc[j].to_dict()\n            row.update(paired_observed_snapshots=int(observed.sum()),\n                       paired_observed_patients=int(len(np.unique(patients))),\n                       missing_pair_snapshots=int((~(np.isfinite(a) & np.isfinite(b)) & step_ok).sum()),\n                       masked_pair_snapshots=int((~step_ok).sum()),\n                       positive_snapshots=int(positive.sum()), negative_snapshots=int(negative.sum()),\n                       step0_ap=ap0, step1_ap=ap1,\n                       ap_difference=None if not estimable else ap0 - ap1,\n                       step0_positive_nonzero_rate=pos0, step1_positive_nonzero_rate=pos1,\n                       positive_nonzero_rate_difference=None if pos0 is None else pos0 - pos1,\n                       step0_negative_nonzero_rate=rate(aa, negative),\n                       step1_negative_nonzero_rate=rate(bb, negative),\n                       status="CORROBORATED" if estimable and ap0 > ap1 and pos0 > pos1 else "INCONCLUSIVE",\n                       claim="descriptive_step0_enrichment_only_not_leakage_proof")\n            records.append(row)\n    screen = pd.DataFrame(records).sort_values(\n        ["ap_difference", "positive_nonzero_rate_difference", "FEATURE_INDEX"],\n        ascending=[False, False, True], na_position="last", kind="stable").reset_index(drop=True)\n    screen.insert(0, "association_rank", np.arange(1, len(screen) + 1))\n    screen.attrs.update(scope="TRAIN_ONLY", physical_steps=[0, 1],\n                        temporal_orientation="DECLARED_NEWEST_FIRST_NOT_VERIFIED",\n                        missingness="paired_complete_cases_no_imputation",\n                        ap_weighting="equal_total_weight_per_patient_among_feature_complete_cases",\n                        historical_provenance_verified=False,\n                        warning="Temporal association is neither a leakage test nor a feature-selection rule.")\n    return screen\n\n\ndef audit_count_lineage(X, metadata, feature_map, valid=None, max_pairs=200):\n    """Convenience API: JSON-compatible aggregate report plus full feature screen."""\n    overlap = overlap_month_consistency(X, metadata, feature_map, valid, max_pairs)\n    screen = step0_label_screen(X, metadata, feature_map, valid)\n    # pandas converts missing scalar estimates to JSON null, never invented zero.\n    overlap["step0_label_screen"] = {\n        "scope": "TRAIN_ONLY_DESCRIPTIVE_ASSOCIATION",\n        "feature_count": len(screen),\n        "top20": json.loads(screen.head(20).to_json(orient="records")),\n        "ap_weighting": screen.attrs["ap_weighting"],\n        "claim": "Suspicious associations can reflect real timing, missingness or confounding; never leakage proof.",\n    }\n    return overlap, screen\n', 'comparison': '"""Aggregate evaluation for locked finalists, never model selection on TEST.\n\nAverage precision (AP) and trapezoidal PR area are deliberately separate.\nBootstrap intervals condition on the fitted predictions and resample patients,\nkeeping all snapshots and both models paired. They do not include retraining\nvariation, resolve unknown provenance, or make a reused holdout prospective.\n"""\nimport hashlib\nimport json\nimport math\n\nimport numpy as np\nfrom sklearn.metrics import (average_precision_score, brier_score_loss, log_loss,\n                             precision_recall_curve, roc_auc_score)\n\n\ndef _inputs(y, scores):\n    y, scores = np.asarray(y), np.asarray(scores, dtype=float)\n    if y.ndim != 1 or scores.ndim != 1 or not len(y) or len(y) != len(scores):\n        raise ValueError("Labels and probability scores must be equal nonempty vectors")\n    if not np.isin(y, [0, 1]).all():\n        raise ValueError("Labels must be binary and nonmissing")\n    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():\n        raise ValueError("Scores must be finite probabilities in [0,1]; logits require an explicit conversion")\n    return y.astype(np.int8), scores\n\n\ndef _ratio(numerator, denominator):\n    return float(numerator / denominator) if denominator else None\n\n\ndef _json_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),\n                                     allow_nan=False).encode()).hexdigest()\n\n\ndef _tie_keys(n, tie_breaker):\n    if tie_breaker is None:\n        return np.arange(n), "frozen_input_row_order"\n    keys = np.asarray(tie_breaker).astype(str)\n    if keys.ndim != 1 or len(keys) != n or len(np.unique(keys)) != n:\n        raise ValueError("Tie keys must be unique, stable snapshot keys in prediction order")\n    return keys, "supplied_unique_snapshot_keys"\n\n\ndef topk_metrics(y, scores, fraction=0.1, tie_breaker=None):\n    """Exactly ceil(fraction*N) selected; stable ties are independent of labels."""\n    y, scores = _inputs(y, scores)\n    if not 0 < float(fraction) <= 1:\n        raise ValueError("Top fraction must lie in (0,1]")\n    keys, tie_rule = _tie_keys(len(y), tie_breaker)\n    k = max(1, int(math.ceil(float(fraction) * len(y))))\n    order = np.lexsort((keys, -scores))\n    selected = order[:k]\n    positives = int(y.sum())\n    captured = int(y[selected].sum())\n    boundary = float(scores[selected[-1]])\n    tied = scores == boundary\n    tied_selected = int(tied[selected].sum())\n    precision = captured / k\n    prevalence = positives / len(y)\n    return dict(fraction=float(fraction), selected=k, selected_positives=captured,\n                actual_population_fraction=k / len(y), precision=precision,\n                recall=_ratio(captured, positives), lift=_ratio(precision, prevalence),\n                cutoff_score=boundary, n_tied_at_cutoff=int(tied.sum()),\n                n_tied_selected=tied_selected, tie_crosses_boundary=bool(tied.sum() > tied_selected),\n                tie_rule=tie_rule, rule="ceil(fraction*N), descending probability, ascending stable key")\n\n\ndef binary_metrics(y, scores, threshold=None, top_fractions=(0.1,), calibration_bins=10,\n                   tie_breaker=None):\n    y, scores = _inputs(y, scores)\n    if isinstance(calibration_bins, bool) or not 1 <= int(calibration_bins) <= 100:\n        raise ValueError("Use 1 to 100 fixed-width calibration bins")\n    calibration_bins = int(calibration_bins)\n    positives, n = int(y.sum()), len(y)\n    both = 0 < positives < n\n    precision, recall, _ = precision_recall_curve(y, scores) if positives else (None, None, None)\n    integrate = getattr(np, "trapezoid", np.trapz if hasattr(np, "trapz") else None)\n    result = dict(snapshots=n, positives=positives, negatives=n - positives, prevalence=positives / n,\n                  average_precision=float(average_precision_score(y, scores)) if positives else None,\n                  pr_auc_trapezoidal=float(integrate(precision[::-1], recall[::-1])) if positives else None,\n                  roc_auc=float(roc_auc_score(y, scores)) if both else None,\n                  brier_score=float(brier_score_loss(y, scores)),\n                  log_loss=float(log_loss(y, scores, labels=[0, 1])),\n                  metric_definition="AP=sum(recall increment*precision); PR area uses trapezoids; these are not interchangeable",\n                  single_class=not both)\n    if threshold is not None:\n        if isinstance(threshold, dict):\n            if threshold.get("selection_split") != "validation" or threshold.get("objective") != "f1":\n                raise ValueError("Threshold policy must be locked on VALIDATION")\n            if threshold.get("policy_sha256") != _json_hash({k: v for k, v in threshold.items() if k != "policy_sha256"}):\n                raise ValueError("Validation threshold policy hash mismatch")\n            value = float(threshold["threshold"])\n        else:\n            value = float(threshold)\n        if not np.isfinite(value) or not 0 <= value <= 1:\n            raise ValueError("Probability threshold must lie in [0,1]")\n        predicted = scores >= value\n        tp, fp = int((predicted & (y == 1)).sum()), int((predicted & (y == 0)).sum())\n        fn, tn = positives - tp, n - positives - fp\n        result.update(threshold=value, tp=tp, fp=fp, fn=fn, tn=tn,\n                      precision=_ratio(tp, tp + fp), recall=_ratio(tp, tp + fn),\n                      specificity=_ratio(tn, tn + fp), f1=_ratio(2 * tp, 2 * tp + fp + fn),\n                      accuracy=(tp + tn) / n,\n                      confusion_matrix=[[tn, fp], [fn, tp]],\n                      threshold_scope="validation_locked" if isinstance(threshold, dict) else "caller_supplied")\n    bins = np.minimum((scores * calibration_bins).astype(int), calibration_bins - 1)\n    rows = []\n    for b in range(calibration_bins):\n        mask = bins == b\n        count = int(mask.sum())\n        rows.append(dict(bin=b, lower=b / calibration_bins, upper=(b + 1) / calibration_bins,\n                         count=count, mean_probability=float(scores[mask].mean()) if count else None,\n                         observed_rate=float(y[mask].mean()) if count else None))\n    result["calibration"] = rows\n    result["expected_calibration_error"] = float(sum(\n        row["count"] / n * abs(row["mean_probability"] - row["observed_rate"])\n        for row in rows if row["count"]))\n    result["calibration_note"] = "Fixed-width descriptive calibration; no calibration model fitted to evaluation labels"\n    result["topk"] = [topk_metrics(y, scores, f, tie_breaker) for f in top_fractions]\n    return result\n\n\ndef select_validation_threshold(y_validation, scores_validation, objective="f1", *, selection_split="validation"):\n    """Choose once on VALIDATION; highest threshold breaks an exact F1 tie."""\n    if selection_split != "validation" or objective != "f1":\n        raise ValueError("Only the prespecified VALIDATION F1 policy is supported")\n    y, scores = _inputs(y_validation, scores_validation)\n    if len(np.unique(y)) != 2:\n        raise ValueError("Threshold selection requires both classes in VALIDATION")\n    precision, recall, thresholds = precision_recall_curve(y, scores)\n    p, r = precision[:-1], recall[:-1]\n    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)\n    best = np.flatnonzero(f1 == f1.max())[-1]\n    record = dict(threshold=float(thresholds[best]), objective=objective, selection_split="validation",\n                  validation_f1=float(f1[best]), validation_snapshots=len(y),\n                  tie_rule="highest probability threshold among exactly equal maximum F1 values",\n                  decision_rule="score >= threshold")\n    record["policy_sha256"] = _json_hash(record)\n    return record\n\n\ndef lock_validation_policy(y_validation, scores_validation, top_fractions=(0.1,)):\n    """Top-K locks population fractions, not a score cutoff derived from TEST."""\n    fractions = list(map(float, top_fractions))\n    if any(not 0 < f <= 1 for f in fractions):\n        raise ValueError("Invalid operational population fractions")\n    return dict(f1=select_validation_threshold(y_validation, scores_validation),\n                top_fractions=fractions,\n                topk_rule="ceil(fraction*N); descending score; ascending prespecified unique snapshot key",\n                selection_split="validation")\n\n\ndef _patient_codes(patient_ids, n):\n    patients = np.asarray(patient_ids)\n    if patients.ndim != 1 or len(patients) != n:\n        raise ValueError("Patient groups must match predictions")\n    if any(v is None or str(v).strip().lower() in ("", "nan", "none", "nat") for v in patients):\n        raise ValueError("Patient IDs must be nonmissing")\n    _, codes = np.unique(patients.astype(str), return_inverse=True)\n    return codes\n\n\ndef paired_patient_bootstrap(y, scores_a, scores_b, patient_ids, n_bootstrap=1000,\n                             seed=20261006, confidence=0.95):\n    """Delta = model A minus B. Cluster multiplicities act as row weights."""\n    y, a = _inputs(y, scores_a)\n    _, b = _inputs(y, scores_b)\n    if len(np.unique(y)) != 2:\n        raise ValueError("Paired AP/AUC inference requires both classes")\n    if isinstance(n_bootstrap, bool) or int(n_bootstrap) != n_bootstrap or n_bootstrap < 20:\n        raise ValueError("At least 20 bootstrap draws required; use >=1000 for reports")\n    if not 0 < confidence < 1:\n        raise ValueError("Confidence must lie in (0,1)")\n    codes = _patient_codes(patient_ids, len(y))\n    groups = int(codes.max()) + 1\n    if groups < 2:\n        raise ValueError("At least two patients required")\n    rng = np.random.default_rng(seed)\n    draws, invalid = [], 0\n    for _ in range(int(n_bootstrap)):\n        multiplicity = np.bincount(rng.integers(0, groups, size=groups), minlength=groups)\n        weights = multiplicity[codes]\n        if weights[y == 1].sum() == 0 or weights[y == 0].sum() == 0:\n            invalid += 1\n            continue\n        ap = average_precision_score(y, a, sample_weight=weights) - average_precision_score(y, b, sample_weight=weights)\n        auc = roc_auc_score(y, a, sample_weight=weights) - roc_auc_score(y, b, sample_weight=weights)\n        draws.append((ap, auc))\n    result = dict(unit="patient", patients=groups, requested_draws=int(n_bootstrap), valid_draws=len(draws),\n                  single_class_draws=invalid, seed=int(seed), confidence=float(confidence),\n                  direction="model_a_minus_model_b", paired=True,\n                  uncertainty_scope="conditional on fitted predictions; excludes retraining and model-selection uncertainty")\n    enough = len(draws) >= max(20, int(math.ceil(0.8 * n_bootstrap)))\n    result["interval_reliable"] = enough\n    values = np.asarray(draws) if draws else np.empty((0, 2))\n    alpha = (1 - confidence) / 2\n    for index, (name, func) in enumerate((("average_precision", average_precision_score), ("roc_auc", roc_auc_score))):\n        result[name] = dict(delta=float(func(y, a) - func(y, b)),\n                            lower=float(np.quantile(values[:, index], alpha)) if enough else None,\n                            upper=float(np.quantile(values[:, index], 1 - alpha)) if enough else None)\n    return result\n\n\ndef latest_patient_indices(patient_ids, end_dates):\n    """One latest snapshot per patient; reject ambiguous duplicate latest keys."""\n    dates = np.asarray(end_dates, dtype="datetime64[D]")\n    codes = _patient_codes(patient_ids, len(dates))\n    if dates.ndim != 1 or np.isnat(dates).any():\n        raise ValueError("Latest-patient evaluation requires exact nonmissing dates")\n    result = []\n    for patient in range(int(codes.max()) + 1):\n        rows = np.flatnonzero(codes == patient)\n        latest = rows[dates[rows] == dates[rows].max()]\n        if len(latest) != 1:\n            raise ValueError("Duplicate latest patient-snapshot keys")\n        result.append(int(latest[0]))\n    return np.asarray(result, dtype=int)\n\n\ndef summarize_seed_runs(results):\n    """Accept training.py\'s candidate -> complete seed-run list; retain every seed."""\n    if not isinstance(results, dict) or not results:\n        raise ValueError("Expected candidate-to-seed-runs dictionary")\n    all_seeds, candidates = [], []\n    for candidate, runs in results.items():\n        seeds = []\n        for run in runs:\n            summary = run["summary"]\n            if not summary.get("training_complete"):\n                raise ValueError("Incomplete runs cannot disappear from the final comparison")\n            seed = int(summary["seed"])\n            if seed in seeds:\n                raise ValueError("Duplicate seed in candidate")\n            seeds.append(seed)\n            row = dict(candidate=candidate, seed=seed,\n                       family=summary["config"]["family"],\n                       parameter_count=summary.get("parameter_count"),\n                       complexity_unit=summary.get("complexity_unit"),\n                       best_epoch=summary.get("best_epoch"), wall_seconds=summary.get("wall_seconds"),\n                       validation_evaluations=summary.get("validation_evaluations"))\n            for split in ("train", "validation", "test"):\n                for metric, value in summary.get(split + "_metrics", {}).items():\n                    if isinstance(value, (int, float, np.number)) and not isinstance(value, (bool, np.bool_)):\n                        row[split + "_" + metric] = float(value)\n            train_ap, val_ap = row.get("train_average_precision"), row.get("validation_average_precision")\n            row["train_validation_gap_ap"] = train_ap - val_ap if train_ap is not None and val_ap is not None else None\n            test_ap = row.get("test_average_precision")\n            row["train_test_gap_ap"] = train_ap - test_ap if train_ap is not None and test_ap is not None else None\n            all_seeds.append(row)\n        if not seeds:\n            raise ValueError("Candidate has no completed seed repetitions")\n        candidate_rows = [r for r in all_seeds if r["candidate"] == candidate]\n        values = [r["validation_average_precision"] for r in candidate_rows]\n        candidates.append(dict(candidate=candidate, seeds=seeds, n_seeds=len(seeds),\n                               mean_validation_ap=float(np.mean(values)),\n                               sd_validation_ap=float(np.std(values, ddof=1)) if len(values) > 1 else None,\n                               min_validation_ap=float(np.min(values)), max_validation_ap=float(np.max(values)),\n                               seed_selection="all prespecified seeds, none selected by outcome"))\n    return dict(all_seeds=all_seeds, candidates=candidates,\n                note="Seed SD is training variability, not a confidence interval; bootstrap intervals are separate")\n\n\ndef comparison_conclusion(bootstrap, provenance):\n    """Wording is gated by comparison provenance, never VALIDATION significance."""\n    provenance = dict(provenance or {})\n    base = dict(status="UNRESOLVED", conclusion=None, restrictions=[])\n    if provenance.get("baseline_kind") == "original_v63" or not provenance.get("fit_membership_verified", False):\n        base.update(status="DESCRIPTIVE_ONLY", conclusion="The existing LightGBM scores have unresolved fitting provenance; they cannot establish comparative generalization or Transformer superiority.")\n        return base\n    required = ("same_cohort", "same_labels", "patient_disjoint_split", "selection_locked")\n    missing = [key for key in required if not provenance.get(key, False)]\n    if missing or provenance.get("test_used_for_selection", True):\n        base["restrictions"] = missing + (["test_used_for_selection_or_unknown"] if provenance.get("test_used_for_selection", True) else [])\n        base["conclusion"] = "A matched final comparison is not established; resolve the listed protocol conditions before making a superiority claim."\n        return base\n    if provenance.get("comparison_split") != "test":\n        base.update(status="DEVELOPMENT_ONLY", conclusion="Validation differences are development evidence after model selection; final-test superiority is not established.")\n        return base\n    for key in ("upstream_cutoff_verified", "holdout_never_previously_inspected"):\n        if not provenance.get(key, False):\n            base["restrictions"].append(key)\n    if not provenance.get("same_feature_information", False):\n        base["restrictions"].append("different_feature_information_architecture_effect_not_isolated")\n    ap = bootstrap["average_precision"]\n    if not bootstrap.get("interval_reliable") or ap["lower"] is None:\n        base.update(status="UNCERTAINTY_UNRESOLVED", conclusion="Too few valid patient-bootstrap draws support a reliable interval; report the point difference without a superiority conclusion.")\n        return base\n    prefix = "In this retrospective matched comparison" if base["restrictions"] else "On the locked patient-disjoint test cohort"\n    if ap["lower"] > 0:\n        verdict = "Transformer AP is higher than matched LightGBM AP"\n        status = "RETROSPECTIVE_AP_ADVANTAGE" if base["restrictions"] else "TEST_AP_ADVANTAGE"\n    elif ap["upper"] < 0:\n        verdict = "matched LightGBM AP is higher than Transformer AP"\n        status = "LIGHTGBM_AP_ADVANTAGE"\n    else:\n        verdict = "the AP interval includes no difference; superiority is not established and this is not proof of equivalence"\n        status = "NO_DEMONSTRATED_AP_ADVANTAGE"\n    base.update(status=status, conclusion=f"{prefix}, {verdict}. Delta AP (Transformer minus LightGBM) = {ap[\'delta\']:.6f}, patient-bootstrap interval [{ap[\'lower\']:.6f}, {ap[\'upper\']:.6f}]. This interval conditions on the fitted models and excludes training/search uncertainty.")\n    if base["restrictions"]:\n        base["conclusion"] += " Unresolved provenance or prior holdout inspection prevents a prospective generalization claim."\n    return base\n\n\ndef compare_predictions(y, transformer_scores, lightgbm_scores, patient_ids, *,\n                        threshold_policies=None, provenance=None, tie_breaker=None,\n                        top_fractions=(0.1,), n_bootstrap=1000, seed=20261006):\n    """Caller aligns both score vectors to identical frozen keys and persists lock first."""\n    policies = threshold_policies or {}\n    t = binary_metrics(y, transformer_scores, policies.get("transformer"), top_fractions=top_fractions,\n                       tie_breaker=tie_breaker)\n    l = binary_metrics(y, lightgbm_scores, policies.get("lightgbm"), top_fractions=top_fractions,\n                       tie_breaker=tie_breaker)\n    interval = paired_patient_bootstrap(y, transformer_scores, lightgbm_scores, patient_ids,\n                                        n_bootstrap=n_bootstrap, seed=seed)\n    return dict(transformer=t, lightgbm=l, paired_uncertainty=interval,\n                conclusion=comparison_conclusion(interval, provenance), provenance=dict(provenance or {}))\n', 'baseline49': '"""Secondary 49-feature LightGBM refit and descriptive original V63 scores.\n\nThese encoded snapshot predictors differ from the 1,028 raw count sequence.\nNeither reading current metadata nor refitting establishes upstream fitting or\nhistorical event-cutoff provenance. All patient-level arrays stay in runtime.\n"""\nimport hashlib\nimport json\nimport time\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score, log_loss, roc_auc_score\n\n\nDEFAULT_SOURCE_PREFIX = "DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_"\nDEFAULT_BASELINE_CONFIG = dict(family="lightgbm49", representation="encoded_snapshot",\n                              n_estimators=500, learning_rate=0.03, num_leaves=16,\n                              min_child_samples=200, scale_pos_weight=10.0,\n                              reg_lambda=0.0, early_stopping_rounds=30, n_jobs=4)\nPARAMETER_EVIDENCE = {\n    "min_child_samples": {"value": 200, "source_spelling": "min_data_in_leaf", "classification": "Recovered from existing artifact"},\n    "num_leaves": {"value": 16, "classification": "Recovered from existing artifact"},\n    "scale_pos_weight": {"value": 10.0, "classification": "Recovered from existing artifact"},\n    "source": "S09 original AutoML OCR: active visible settings at lines 101, 105, 107; lightgbm_v63_context.md",\n    "scope": "Visible requested settings, not proof of all effective historical fitted-model parameters",\n    "new_refit_policy": "learning_rate=0.03; maximum 500 rounds; validation-AP stopping patience 30; default L2=0; fixed config, not tuned",\n}\n\n\ndef _hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),\n                                     allow_nan=False).encode()).hexdigest()\n\n\ndef _frame(value, selected=None):\n    if isinstance(value, pd.DataFrame):\n        return value[selected].copy() if selected is not None else value.copy()\n    if selected is not None:\n        value = value.select(*selected)\n    if not hasattr(value, "toPandas"):\n        raise TypeError("Reader must return a Spark or pandas DataFrame")\n    return value.toPandas()\n\n\ndef _read_frozen_source(read_table, table, frozen, selected):\n    """Filter keys in Spark before collection, including the VAL-only SCORE path."""\n    value = read_table(table)\n    keys = frozen[["PATIENT_ID", "END_DT"]].copy()\n    if isinstance(value, pd.DataFrame):\n        value = _keys(value)\n        return value[[name.upper() for name in selected]].merge(keys, on=["PATIENT_ID", "END_DT"], how="inner", validate="one_to_one")\n    physical = {str(name).upper(): str(name) for name in value.columns}\n    if len(physical) != len(value.columns):\n        raise ValueError("Ambiguous physical column casing")\n    if any(name.upper() not in physical for name in selected):\n        raise ValueError("Required physical source column unavailable")\n    # Preserve actual identifier case and quote names before applying canonical aliases.\n    selected_frame = value.select(*[value["`" + physical[name.upper()].replace("`", "``") + "`"].alias(name.upper())\n                                    for name in selected])\n    if not hasattr(value, "sparkSession"):\n        raise TypeError("Spark reader must expose sparkSession to restrict frozen keys before collection")\n    keys["END_DT"] = keys.END_DT.dt.date\n    key_frame = value.sparkSession.createDataFrame(keys)\n    return selected_frame.join(key_frame, on=["PATIENT_ID", "END_DT"], how="inner").toPandas()\n\n\ndef _normalized_columns(frame):\n    names = [str(name).upper() for name in frame.columns]\n    if len(set(names)) != len(names):\n        raise ValueError("Case-insensitive duplicate source columns")\n    frame = frame.copy()\n    frame.columns = names\n    return frame\n\n\ndef _keys(frame, require_label=True):\n    frame = _normalized_columns(frame)\n    required = {"PATIENT_ID", "END_DT"} | ({"RESP"} if require_label else set())\n    if not required <= set(frame):\n        raise ValueError("Exact PATIENT_ID, END_DT and RESP contract is required")\n    if frame.PATIENT_ID.isna().any() or not frame.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x) and x == x.strip()).all():\n        raise ValueError("Patient IDs must be exact nonempty strings")\n    dates = pd.to_datetime(frame.END_DT, errors="raise")\n    if dates.isna().any() or not (dates == dates.dt.normalize()).all():\n        raise ValueError("Snapshot dates must be nonmissing midnight dates")\n    if dates.dt.tz is not None:\n        raise ValueError("Timezone-dependent snapshot dates are unsupported")\n    frame["END_DT"] = dates\n    if frame.duplicated(["PATIENT_ID", "END_DT"]).any():\n        raise ValueError("Duplicate patient-snapshot source keys")\n    if require_label and not frame.RESP.isin([0, 1]).all():\n        raise ValueError("Labels must be nonmissing binary RESP values")\n    return frame\n\n\ndef _feature_list(value):\n    if isinstance(value, str):\n        value = json.loads(value)\n    if isinstance(value, np.ndarray):\n        value = value.tolist()\n    if not isinstance(value, list) or not value or not all(isinstance(v, str) and v and v == v.strip() for v in value):\n        raise ValueError("MODEL_TYPE feature field must hold an ordered nonempty exact-name array")\n    if len({name.upper() for name in value}) != len(value):\n        raise ValueError("Duplicate configured feature names")\n    forbidden = {"PATIENT_ID", "END_DT", "START_DT", "RESP", "SPLIT", "RND", "SCORE", "DECILE", "CENTILE", "MILLILE"}\n    if forbidden & {v.upper() for v in value}:\n        raise ValueError("Identifier, label, split or score cannot be a baseline predictor")\n    return value\n\n\ndef _numeric_matrix(values):\n    values = np.asarray(values)\n    if values.ndim != 2 or not values.shape[0] or not values.shape[1]:\n        raise ValueError("Expected nonempty snapshot by feature matrix")\n    try:\n        values = values.astype(np.float32)\n    except (TypeError, ValueError) as error:\n        raise ValueError("Encoded feature values must be numeric") from error\n    if np.isinf(values).any():\n        raise ValueError("Infinite encoded values are not permitted")\n    return values\n\n\ndef _input_identity(metadata, values, features):\n    """Order-sensitive private-input digest; no individual records are returned."""\n    frame = metadata[["PATIENT_ID", "END_DT", "RESP"]].copy()\n    frame["END_DT"] = frame.END_DT.dt.strftime("%Y-%m-%d")\n    frame["RESP"] = frame.RESP.astype(int)\n    if "SPLIT" in metadata:\n        frame["SPLIT"] = metadata.SPLIT.astype(str).str.lower()\n    array = np.asarray(values, dtype="<f4").copy(order="C")\n    array[np.isnan(array)] = np.nan  # One canonical NaN representation.\n    digest = hashlib.sha256()\n    digest.update(json.dumps({"features": features, "shape": list(array.shape), "dtype": "float32",\n                              "metadata": frame.to_dict("records")}, sort_keys=True,\n                             separators=(",", ":"), allow_nan=False).encode())\n    digest.update(array.tobytes(order="C"))\n    return digest.hexdigest()\n\n\ndef _align(frozen, source, columns):\n    source = _keys(source)\n    source_columns = ["PATIENT_ID", "END_DT", "RESP"] + list(columns)\n    if not set(source_columns) <= set(source):\n        raise ValueError("Source lacks required predictors or scores")\n    left = frozen[["PATIENT_ID", "END_DT", "RESP"]].copy().reset_index(drop=True)\n    merged = left.merge(source[source_columns], how="left", on=["PATIENT_ID", "END_DT"],\n                        suffixes=("_FROZEN", "_SOURCE"), validate="one_to_one", indicator=True, sort=False)\n    if not merged["_merge"].eq("both").all():\n        raise ValueError("Source does not cover every exact frozen patient-snapshot key")\n    if not merged.RESP_FROZEN.eq(merged.RESP_SOURCE).all():\n        raise ValueError("Source labels disagree with frozen RESP")\n    return merged[list(columns)]\n\n\ndef load_encoded_baseline(read_table, frozen_metadata, source_prefix=DEFAULT_SOURCE_PREFIX, expected_features=49):\n    """Preserve MODEL_TYPE order; order-only disagreement with FINAL_MODEL is an audit."""\n    frozen = _keys(_frame(frozen_metadata))\n    config = _normalized_columns(_frame(read_table(source_prefix + "MODEL_TYPE")))\n    if len(config) != 1:\n        raise ValueError("Expected exactly one MODEL_TYPE row")\n    recognized = [name for name in ("FEATURES", "FEATURE_NAMES") if name in config]\n    if not recognized:\n        raise ValueError("No recognized ordered feature field in MODEL_TYPE")\n    # FEATURES is the recovered contract; FEATURE_NAMES is supported only if explicitly present.\n    features = _feature_list(config.iloc[0][recognized[0]])\n    if any(_feature_list(config.iloc[0][name]) != features for name in recognized[1:]):\n        raise ValueError("Conflicting configured feature arrays")\n    if len(features) != expected_features:\n        raise ValueError("Configured feature count differs; never fabricate or substitute 49 features")\n    final = _normalized_columns(_frame(read_table(source_prefix + "FINAL_MODEL")))\n    if not {"FEATURES", "SEQ"} <= set(final):\n        raise ValueError("FINAL_MODEL must expose FEATURES and SEQ for the recorded order audit")\n    if final.FEATURES.isna().any() or final.FEATURES.duplicated().any():\n        raise ValueError("FINAL_MODEL has missing or duplicate names")\n    rank = pd.to_numeric(final.SEQ, errors="coerce")\n    valid_ranks = bool(rank.notna().all() and np.isfinite(rank).all() and rank.eq(np.floor(rank)).all() and not rank.duplicated().any())\n    final_names = final.assign(_rank=rank).sort_values("_rank").FEATURES.tolist() if valid_ranks else None\n    configured_only = sorted(set(features) - set(final.FEATURES))\n    final_only = sorted(set(final.FEATURES) - set(features))\n    if configured_only or final_only:\n        raise ValueError("Configured and final feature sets differ; order-only discrepancy exception does not apply")\n    audit = dict(configured_feature_count=len(features), final_feature_count=len(final),\n                 configured_field=recognized[0], configured_order_sha256=_hash(features),\n                 final_seq_order_sha256=_hash(final_names) if final_names else None,\n                 final_seq_valid=valid_ranks, exact_order_agreement=features == final_names,\n                 rule="MODEL_TYPE array order retained for this refit; SEQ is an unresolved historical summary-order contract",\n                 historical_scoring_order_verified=False)\n    # Select only necessary columns before bringing Spark data into approved runtime memory.\n    source = _read_frozen_source(read_table, source_prefix + "MODEL_DATA", frozen,\n                                 ["PATIENT_ID", "END_DT", "RESP"] + features)\n    aligned = _align(frozen, source, [name.upper() for name in features])\n    values = _numeric_matrix(aligned.to_numpy())\n    return dict(metadata=frozen, X=values, features=features, feature_sha256=_hash(features),\n                input_sha256=_input_identity(frozen, values, features), order_audit=audit,\n                provenance=dict(source=source_prefix + "MODEL_DATA", baseline_kind="encoded49_refit",\n                                historical_fit_membership_verified=False, upstream_encoding_fit_membership="UNKNOWN",\n                                input_space="Existing encoded V63 snapshot values; fractions and AGE retained",\n                                preprocessing_refitted=False, architecture_comparison_to_1028="different feature information; contextual comparison"))\n\n\ndef load_original_v63_scores(read_table, frozen_metadata, split="validation", *, allow_test=False,\n                             selection_locked=False, source_prefix=DEFAULT_SOURCE_PREFIX):\n    """Read only requested split\'s aligned scores; never infer fitting exclusion."""\n    if split not in ("train", "validation", "test"):\n        raise ValueError("Unrecognized frozen split")\n    if split == "test" and not (allow_test and selection_locked):\n        raise ValueError("TEST original scores require explicit final comparison and a persisted selection lock")\n    frozen = _keys(_frame(frozen_metadata))\n    if "SPLIT" not in frozen:\n        raise ValueError("Frozen split labels required")\n    chosen = frozen.loc[frozen.SPLIT.str.lower().eq(split)].reset_index(drop=True)\n    if chosen.empty:\n        raise ValueError("Requested split has no snapshots")\n    source = _read_frozen_source(read_table, source_prefix + "UNIVERSE_W_FEATURES_SCORED", chosen,\n                                 ["PATIENT_ID", "END_DT", "RESP", "SCORE"])\n    aligned = _align(chosen, source, ["SCORE"])\n    scores = pd.to_numeric(aligned.SCORE, errors="raise").to_numpy(dtype=float)\n    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():\n        raise ValueError("Original SCORE requires confirmed finite probability semantics")\n    return dict(metadata=chosen, scores=scores, provenance=dict(baseline_kind="original_v63", split=split,\n                fit_membership_verified=False, historical_fitting_scope="UNKNOWN", same_keys_and_labels=True,\n                score_semantics="Candidate probability-like SCORE; semantics not established by range alone",\n                interpretation="Descriptive existing scores; may include fitted patients; no generalization/superiority claim"))\n\n\ndef _metrics(y, scores):\n    return dict(average_precision=float(average_precision_score(y, scores)),\n                roc_auc=float(roc_auc_score(y, scores)), log_loss=float(log_loss(y, scores, labels=[0, 1])))\n\n\ndef train_encoded_baseline(trainvalues, y, groups, valvalues, yval, seed, config=None):\n    """One fixed contextual baseline configuration across all prespecified seeds."""\n    import lightgbm as lgb\n    started = time.perf_counter()\n    cfg = dict(DEFAULT_BASELINE_CONFIG)\n    if config:\n        unknown = set(config) - set(cfg)\n        if unknown:\n            raise ValueError("Unexpected baseline configuration fields: " + repr(sorted(unknown)))\n        cfg.update(config)\n    if cfg["family"] != "lightgbm49" or cfg["representation"] != "encoded_snapshot":\n        raise ValueError("49-feature context baseline has a fixed family and representation")\n    train, validation = _numeric_matrix(trainvalues), _numeric_matrix(valvalues)\n    y, yval = np.asarray(y), np.asarray(yval)\n    if train.shape[1] != validation.shape[1] or len(y) != len(train) or len(yval) != len(validation):\n        raise ValueError("Train and validation dimensions differ")\n    if not np.isin(y, [0, 1]).all() or not np.isin(yval, [0, 1]).all() or len(np.unique(y)) != 2 or len(np.unique(yval)) != 2:\n        raise ValueError("Both fitting and validation require valid binary labels and both classes")\n    if len(groups) != len(y) or any(v is None or not str(v).strip() for v in groups):\n        raise ValueError("Valid TRAIN patient groups required; split disjointness is checked by data contract")\n    fit_cfg = {key: value for key, value in cfg.items() if key not in ("family", "representation", "early_stopping_rounds")}\n    model = lgb.LGBMClassifier(**fit_cfg, objective="binary", metric="None", random_state=int(seed),\n                               deterministic=True, force_col_wise=True, verbosity=-1)\n    history = {}\n    def ap_metric(target, score):\n        return "average_precision", float(average_precision_score(target, score)), True\n    model.fit(train, y, eval_set=[(validation, yval)], eval_metric=ap_metric,\n              callbacks=[lgb.early_stopping(int(cfg["early_stopping_rounds"]), first_metric_only=True, verbose=False),\n                         lgb.record_evaluation(history), lgb.log_evaluation(0)])\n    best = int(model.best_iteration_ or cfg["n_estimators"])\n    blob = model.booster_.model_to_string(num_iteration=best).encode()\n    train_scores = model.booster_.predict(train, num_iteration=best)\n    val_scores = model.booster_.predict(validation, num_iteration=best)\n    trajectory = history.get("valid_0", {}).get("average_precision", [])\n    rows = [dict(epoch=i + 1, validation_average_precision=float(value)) for i, value in enumerate(trajectory)]\n    summary = dict(config=cfg, seed=int(seed), training_complete=True, best_epoch=best,\n                   epochs_observed=len(rows), validation_evaluations=len(rows), train_metrics=_metrics(y, train_scores),\n                   validation_metrics=_metrics(yval, val_scores), generalization_gap_ap=float(average_precision_score(y, train_scores) - average_precision_score(yval, val_scores)),\n                   parameter_count=int(sum(tree["num_leaves"] for tree in model.booster_.dump_model()["tree_info"])),\n                   complexity_unit="tree_leaves", wall_seconds=time.perf_counter() - started,\n                   artifact_format="lightgbm_text", model_sha256=hashlib.sha256(blob).hexdigest(),\n                   transform=dict(kind="identity_encoded_numeric", input_dim=train.shape[1], imputation="native LightGBM missing-value handling"),\n                   importance=model.booster_.feature_importance(importance_type="gain").tolist(),\n                   train_missing_values=int(np.isnan(train).sum()), test_inference_performed=False,\n                   historical_fit_membership_verified=False, upstream_encoding_fit_membership="UNKNOWN",\n                   parameter_evidence=PARAMETER_EVIDENCE,\n                   configured_parameter_overrides={k: v for k, v in cfg.items() if v != DEFAULT_BASELINE_CONFIG[k]},\n                   selection_scope="Fixed 49-feature contextual refit; no configuration search; validation-AP early stopping",\n                   source_of_defaults="Visible historical leaf/weight settings plus explicit new refit learning/iteration policy; not recovered full historical parameters")\n    summary["config_sha256"] = _hash(summary["config"])\n    summary["transform_sha256"] = _hash(summary["transform"])\n    summary["raw_input_shape"] = list(train.shape[1:])\n    return dict(summary=summary, history=rows, model_blob=blob, train_scores=train_scores, validation_scores=val_scores)\n\n\ndef predict_encoded(result, values):\n    import lightgbm as lgb\n    values = _numeric_matrix(values)\n    summary, blob = result["summary"], result["model_blob"]\n    if hashlib.sha256(blob).hexdigest() != summary["model_sha256"]:\n        raise ValueError("Baseline artifact checksum mismatch")\n    if _hash(summary["config"]) != summary["config_sha256"] or _hash(summary["transform"]) != summary["transform_sha256"]:\n        raise ValueError("Baseline configuration/transform checksum mismatch")\n    if values.shape[1] != summary["transform"]["input_dim"]:\n        raise ValueError("Encoded baseline feature count differs")\n    model = lgb.Booster(model_str=blob.decode())\n    return model.predict(values)\n', 'reliance': '"""Post-lock validation diagnostics: association rank is not learned reliance.\n\nThese routines do not fit, tune, select a new model, or inspect TEST. Internal\nparameter/gain diagnostics remain separate from measured permutation effects.\nPermutation results are conditional marginal diagnostics, not causal effects.\n"""\nimport hashlib\nimport json\nimport re\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score\n\n\ndef _require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef _hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),\n                                     allow_nan=False).encode()).hexdigest()\n\n\ndef _map(feature_map):\n    frame = pd.DataFrame(feature_map).copy()\n    _require({"FEATURE_INDEX", "FEATURE_NAME"} <= set(frame), "Feature map requires index and exact name.")\n    _require(np.issubdtype(frame.FEATURE_INDEX.dtype, np.integer), "Feature map indices must be integers.")\n    _require(np.array_equal(frame.FEATURE_INDEX.to_numpy(), np.arange(len(frame))), "Feature map is not in original tensor order.")\n    _require(frame.FEATURE_NAME.map(lambda v: isinstance(v, str) and bool(v)).all() and frame.FEATURE_NAME.is_unique,\n             "Feature names must be nonempty and unique.")\n    return frame[["FEATURE_INDEX", "FEATURE_NAME"]].reset_index(drop=True)\n\n\ndef _agreement(frame, k):\n    paired = frame.loc[frame.TRAIN_RANK.notna() & frame.INTERNAL_RANK.notna()]\n    if len(paired) >= 2 and paired.TRAIN_RANK.nunique() > 1 and paired.INTERNAL_RANK.nunique() > 1:\n        rho = float(paired.TRAIN_RANK.corr(paired.INTERNAL_RANK, method="spearman"))\n    else:\n        rho = None\n    actual_k = min(k, len(paired))\n    train_top = set(paired.sort_values(["TRAIN_RANK", "FEATURE_INDEX"]).head(actual_k).FEATURE_INDEX)\n    model_top = set(paired.sort_values(["INTERNAL_RANK", "FEATURE_INDEX"]).head(actual_k).FEATURE_INDEX)\n    overlap = len(train_top & model_top)\n    return dict(paired_selected_features=len(paired), spearman_rank_agreement=rho,\n                top_k_within_selected=actual_k, top_k_overlap=overlap,\n                top_k_overlap_fraction=overlap / actual_k if actual_k else None,\n                interpretation="Descriptive agreement only; disagreement is an investigation flag, never a model failure criterion.")\n\n\ndef join_rank_and_internal_diagnostics(feature_map, train_ranking, runs, expected_columns,\n                                       expected_seeds, top_k=10):\n    """Return full-universe and every-seed tables keyed by original feature ID.\n\n    Internal magnitude is projection-column norm for a Transformer or summed\n    temporal-block gain for count LightGBM. Neither is held-out decision influence.\n    Config equality prevents combining different architectures or feature axes.\n    """\n    fm = _map(feature_map)\n    columns = np.asarray(expected_columns)\n    _require(columns.ndim == 1 and np.issubdtype(columns.dtype, np.integer) and len(columns) > 0,\n             "Expected columns must be nonempty original integer indices.")\n    _require(len(np.unique(columns)) == len(columns) and np.all((columns >= 0) & (columns < len(fm))),\n             "Expected columns are duplicate or outside the map.")\n    _require(np.array_equal(columns, np.sort(columns)), "Model columns must preserve original FEATURE_MAP order.")\n    seeds = list(expected_seeds)\n    _require(seeds and len(set(seeds)) == len(seeds) and all(type(s) is int for s in seeds), "Declare every unique prespecified seed.")\n    _require(type(top_k) is int and top_k >= 1, "top_k must be a positive integer.")\n    ranking = pd.DataFrame(train_ranking).copy()\n    _require({"FEATURE_INDEX", "FEATURE_NAME", "RANK"} <= set(ranking), "TRAIN ranking lacks its index/name/rank identity.")\n    _require(np.issubdtype(ranking.FEATURE_INDEX.dtype, np.integer), "TRAIN feature indices must be integers.")\n    _require(ranking.FEATURE_INDEX.is_unique and ranking.FEATURE_NAME.is_unique, "Duplicate TRAIN ranking keys.")\n    _require(ranking.FEATURE_INDEX.isin(fm.FEATURE_INDEX).all(), "TRAIN ranking contains an unknown feature index.")\n    reference = fm.set_index("FEATURE_INDEX").FEATURE_NAME\n    _require(ranking.FEATURE_NAME.eq(ranking.FEATURE_INDEX.map(reference)).all(), "TRAIN feature name/index mismatch.")\n    ranks = pd.to_numeric(ranking.RANK, errors="coerce")\n    _require(np.isfinite(ranks).all() and (ranks > 0).all(), "TRAIN ranks must be finite and positive.")\n    ranking["RANK"] = ranks\n    _require(set(columns) <= set(ranking.FEATURE_INDEX), "Every selected feature must have an explicit TRAIN ranking.")\n    rename = {key: "TRAIN_" + key for key in ranking.columns if key not in {"FEATURE_INDEX", "FEATURE_NAME"}}\n    ranked = ranking.drop(columns="FEATURE_NAME").rename(columns=rename)\n    full = fm.merge(ranked, on="FEATURE_INDEX", how="left", validate="one_to_one")\n    position = {int(original): i for i, original in enumerate(columns)}\n    full["SELECTED_IN_MODEL"] = full.FEATURE_INDEX.isin(columns)\n    full["MODEL_POSITION"] = full.FEATURE_INDEX.map(position).astype("Int64")\n    full["TRAIN_RANK_AVAILABLE"] = full.TRAIN_RANK.notna()\n    full["PERMUTATION_STATUS"] = np.where(full.SELECTED_IN_MODEL, "NOT_TESTED", "NOT_IN_MODEL")\n    _require(len(runs) == len(seeds), "Missing seed runs; no favorable-seed subset is allowed.")\n    observed_seeds = [run["summary"]["seed"] for run in runs]\n    _require(sorted(observed_seeds) == sorted(seeds), "Missing or duplicate seed runs.")\n    canonical_config = runs[0]["summary"]["config"]\n    family = canonical_config.get("family")\n    _require(family in {"transformer", "lightgbm"}, "This diagnostic uses the ordered count-model contract.")\n    rows, agreements, hashes = [], [], []\n    for run in sorted(runs, key=lambda run: run["summary"]["seed"]):\n        summary = run["summary"]\n        config = summary["config"]\n        _require(summary.get("training_complete") is True, "Incomplete model run.")\n        _require(config == canonical_config, "Seed configurations differ; cannot align their internal diagnostics.")\n        _require(config.get("columns") == columns.tolist(), "Saved model columns do not match the locked input order.")\n        _require(summary.get("config_sha256") == _hash(config), "Saved model configuration hash mismatch.")\n        model_hash = summary.get("model_sha256")\n        _require(isinstance(model_hash, str) and re.fullmatch(r"[0-9a-f]{64}", model_hash) is not None,\n                 "Saved model identity hash is missing or invalid.")\n        if "model_blob" in run:\n            _require(hashlib.sha256(run["model_blob"]).hexdigest() == model_hash, "Saved model bytes/hash mismatch.")\n        values = np.asarray(summary.get("aggregated_original_importance"), dtype=float)\n        _require(values.shape == (len(columns),) and np.isfinite(values).all() and (values >= 0).all(),\n                 "Internal diagnostic width/values do not match original selected features.")\n        representation = config.get("representation")\n        expected_blocks = 1 if family == "transformer" else {"flattened": 12, "windows": 4, "annual": 1}.get(representation)\n        _require(expected_blocks is not None and summary.get("importance_blocks_per_original_feature") == expected_blocks,\n                 "Internal diagnostic temporal-block aggregation does not match representation.")\n        raw = np.asarray(summary.get("importance"), dtype=float)\n        _require(raw.shape == (expected_blocks * len(columns),) and np.isfinite(raw).all() and (raw >= 0).all(),\n                 "Original internal diagnostic array has wrong width/values.")\n        _require(np.allclose(raw.reshape(expected_blocks, len(columns)).sum(axis=0), values, rtol=1e-10, atol=1e-12),\n                 "Aggregated importance is not aligned to the original feature axis.")\n        part = full.loc[full.SELECTED_IN_MODEL].sort_values("MODEL_POSITION").copy()\n        part["SEED"] = int(summary["seed"])\n        part["MODEL_FAMILY"] = family\n        part["INTERNAL_MAGNITUDE"] = values\n        part["INTERNAL_RANK"] = pd.Series(values).rank(method="average", ascending=False).to_numpy()\n        part["INTERNAL_SHARE"] = values / values.sum() if values.sum() else np.zeros_like(values)\n        part["INTERNAL_DIAGNOSTIC"] = "projection_column_norm" if family == "transformer" else "temporal_block_gain_sum"\n        part["NOT_HELDOUT_RELIANCE"] = True\n        # A TRAIN-only transformed standard deviation adjusts the raw projection\n        # norm for input scale. This remains a parameter diagnostic, not reliance.\n        train_sd = summary.get("transform", {}).get("training_transformed_sd") if family == "transformer" else None\n        part["SCALE_AWARE_INTERNAL_AVAILABLE"] = train_sd is not None\n        part["PROJECTION_NORM_X_TRAIN_SD"] = np.nan\n        part["SCALE_AWARE_INTERNAL_RANK"] = np.nan\n        if train_sd is not None:\n            sd = np.asarray(train_sd, dtype=float)\n            _require(sd.shape == (len(columns),) and np.isfinite(sd).all() and (sd >= 0).all(),\n                     "TRAIN transformed SD width/values do not match selected model features.")\n            scale_aware = values * sd\n            _require(np.isfinite(scale_aware).all(), "Scale-aware internal magnitude overflowed.")\n            part["PROJECTION_NORM_X_TRAIN_SD"] = scale_aware\n            part["SCALE_AWARE_INTERNAL_RANK"] = pd.Series(scale_aware).rank(method="average", ascending=False).to_numpy()\n        rows.append(part)\n        agreements.append(dict(seed=int(summary["seed"]), model_family=family, **_agreement(part, top_k)))\n        hashes.append(dict(seed=int(summary["seed"]), model_sha256=summary.get("model_sha256"), config_sha256=summary["config_sha256"]))\n    seed_table = pd.concat(rows, ignore_index=True)\n    aggregate = seed_table.groupby("FEATURE_INDEX", as_index=False).agg(\n        MEAN_INTERNAL_MAGNITUDE=("INTERNAL_MAGNITUDE", "mean"), SD_INTERNAL_MAGNITUDE=("INTERNAL_MAGNITUDE", "std"),\n        MEAN_INTERNAL_RANK=("INTERNAL_RANK", "mean"), SD_INTERNAL_RANK=("INTERNAL_RANK", "std"),\n        MEAN_INTERNAL_SHARE=("INTERNAL_SHARE", "mean"), SEEDS_REPORTED=("SEED", "size"),\n        MEAN_PROJECTION_NORM_X_TRAIN_SD=("PROJECTION_NORM_X_TRAIN_SD", "mean"),\n        SD_PROJECTION_NORM_X_TRAIN_SD=("PROJECTION_NORM_X_TRAIN_SD", "std"),\n        MEAN_SCALE_AWARE_INTERNAL_RANK=("SCALE_AWARE_INTERNAL_RANK", "mean"),\n        SD_SCALE_AWARE_INTERNAL_RANK=("SCALE_AWARE_INTERNAL_RANK", "std"),\n        SCALE_AWARE_SEEDS_REPORTED=("PROJECTION_NORM_X_TRAIN_SD", "count"))\n    full = full.merge(aggregate, on="FEATURE_INDEX", how="left", validate="one_to_one")\n    aggregate_agreement = _agreement(full.rename(columns={"MEAN_INTERNAL_RANK": "INTERNAL_RANK"}), top_k)\n    summary = dict(model_family=family, original_feature_count=len(fm), selected_feature_count=len(columns),\n                   selected_columns=columns.tolist(), selected_names=reference.loc[columns].tolist(),\n                   seeds=seeds, model_identities=hashes, aggregate_agreement=aggregate_agreement,\n                   permutation_measured=False, evidence="Internal diagnostics joined to TRAIN-only screening ranks; no production result implied.",\n                   scale_aware_available_seeds=sorted(seed_table.loc[seed_table.SCALE_AWARE_INTERNAL_AVAILABLE, "SEED"].unique().tolist()),\n                   scale_aware_definition="Projection column norm times per-feature TRAIN transformed-input SD; unavailable when absent. Internal, noncausal, not measured reliance.",\n                   warning="Association can be weak while interactions are useful. Norms/gain can be large without useful held-out influence. Never force matching ranks.")\n    return dict(full_table=full, seed_table=seed_table, agreement_table=pd.DataFrame(agreements), summary=summary)\n\n\ndef prespecify_probes(joined, *, max_features=30, random_seed=20261006,\n                     top_train=10, top_internal=10, lowest_train=5, random_remaining=5,\n                     correlation_pairs=None, correlation_threshold=.9, max_correlation_groups=10):\n    """Fix a bounded probe union before permutation results are inspected.\n\n    Top internal probes use mean within-seed rank, so a seed\'s gain scale cannot\n    dominate. Source-family groups recognize DX_/DX__, etc., preserving names.\n    No random feature is selected using VALIDATION labels or permutation results.\n    """\n    _require(all(type(n) is int and n >= 0 for n in (max_features, top_train, top_internal, lowest_train, random_remaining))\n             and max_features > 0, "Probe limits must be nonnegative integers with a positive cap.")\n    _require(top_train + top_internal + lowest_train + random_remaining <= max_features,\n             "Probe quota sum exceeds the declared feature cap; do not silently truncate a quota.")\n    _require(type(random_seed) is int and random_seed >= 0, "Invalid label-independent probe seed.")\n    _require(type(max_correlation_groups) is int and 0 <= max_correlation_groups <= 10 and\n             np.isfinite(correlation_threshold) and .9 <= correlation_threshold <= 1.,\n             "Declare a correlation threshold >=.9 and at most ten correlation groups.")\n    full = joined["full_table"]\n    selected = full.loc[full.SELECTED_IN_MODEL].sort_values("MODEL_POSITION")\n    columns = selected.FEATURE_INDEX.astype(int).tolist()\n    names = selected.FEATURE_NAME.tolist()\n    _require(columns == joined["summary"]["selected_columns"] and names == joined["summary"]["selected_names"],\n             "Joined table was reordered or relabeled after the mapping audit.")\n    choices = {}\n    def add(indices, reason):\n        for index in indices:\n            choices.setdefault(int(index), []).append(reason)\n    add(selected.sort_values(["TRAIN_RANK", "FEATURE_INDEX"]).head(top_train).FEATURE_INDEX, "top_train_rank")\n    add(selected.sort_values(["MEAN_INTERNAL_RANK", "FEATURE_INDEX"]).head(top_internal).FEATURE_INDEX, "top_internal_rank")\n    add(selected.sort_values(["TRAIN_RANK", "FEATURE_INDEX"], ascending=[False, True]).head(lowest_train).FEATURE_INDEX, "lowest_train_rank")\n    remaining = sorted(set(columns) - set(choices))\n    rng = np.random.default_rng(random_seed)\n    random = rng.choice(remaining, size=min(random_remaining, len(remaining)), replace=False) if remaining else []\n    add(sorted(random), "label_independent_random_remaining")\n    _require(len(choices) <= max_features, "Probe cap exceeded.")\n    index_to_position = {index: i for i, index in enumerate(columns)}\n    probes = []\n    for index in sorted(choices):\n        pos = index_to_position[index]\n        probes.append(dict(probe_id=f"feature:{index}", kind="feature", feature_indices=[index],\n                           model_positions=[pos], feature_names=[names[pos]], reasons=choices[index]))\n    family = {}\n    for pos, name in enumerate(names):\n        match = re.match(r"^(DX|PX|RX)_+", name, flags=re.IGNORECASE)\n        if match:\n            family.setdefault(match.group(1).upper(), []).append(pos)\n    for group, positions in sorted(family.items()):\n        probes.append(dict(probe_id="family:" + group, kind="family", model_positions=positions,\n                           feature_indices=[columns[p] for p in positions], feature_names=[names[p] for p in positions],\n                           reasons=["joint_source_family_probe; not a validated clinical interaction group"]))\n    train_order = selected.sort_values(["TRAIN_RANK", "FEATURE_INDEX"]).FEATURE_INDEX.astype(int).tolist()\n    for label, indices in zip(("top", "middle", "bottom"), np.array_split(train_order, 3)):\n        indices = sorted(int(i) for i in indices)\n        if indices:\n            positions = [index_to_position[i] for i in indices]\n            probes.append(dict(probe_id="train_tercile:" + label, kind="train_tercile", model_positions=positions,\n                               feature_indices=indices, feature_names=[names[p] for p in positions],\n                               reasons=["Selected-feature TRAIN composite-rank tercile; fixed-model joint reliance, not added predictive information."]))\n    # Only TRAIN-derived pairs may be passed. Restrict the graph to selected nodes;\n    # an unselected feature must not bridge two otherwise disconnected groups.\n    components = []\n    if correlation_pairs is not None:\n        pairs = pd.DataFrame(correlation_pairs).copy()\n        required = {"FEATURE_INDEX_A", "FEATURE_INDEX_B", "FEATURE_NAME_A", "FEATURE_NAME_B", "SPEARMAN"}\n        _require(required <= set(pairs), "TRAIN correlation pairs lack exact feature identities or Spearman values.")\n        if "SPLIT" in pairs:\n            _require(pairs.SPLIT.eq("train").all(), "Correlation groups must be defined on TRAIN only.")\n        reference = full.set_index("FEATURE_INDEX").FEATURE_NAME\n        for side in ("A", "B"):\n            index, name = "FEATURE_INDEX_" + side, "FEATURE_NAME_" + side\n            _require(pairs[index].isin(reference.index).all() and pairs[name].eq(pairs[index].map(reference)).all(),\n                     "TRAIN correlation name/index mismatch.")\n        rho = pd.to_numeric(pairs.SPEARMAN, errors="coerce")\n        _require(np.isfinite(rho).all() and rho.abs().le(1. + 1e-12).all(), "Invalid TRAIN Spearman values.")\n        parents = {i:i for i in columns}\n        def find(index):\n            while parents[index] != index:\n                parents[index] = parents[parents[index]]\n                index = parents[index]\n            return index\n        edges = pairs.loc[rho.abs().ge(correlation_threshold) & pairs.FEATURE_INDEX_A.isin(columns) & pairs.FEATURE_INDEX_B.isin(columns)]\n        for a, b in sorted(zip(edges.FEATURE_INDEX_A.astype(int), edges.FEATURE_INDEX_B.astype(int))):\n            aa, bb = find(a), find(b)\n            parents[max(aa, bb)] = min(aa, bb)\n        connected = {}\n        for index in columns:\n            connected.setdefault(find(index), []).append(index)\n        components = sorted((v for v in connected.values() if len(v) >= 2), key=lambda v:(-len(v), tuple(v)))\n    component_coverage = []\n    for number, indices in enumerate(components):\n        positions = [index_to_position[i] for i in indices]\n        included = number < max_correlation_groups\n        probe_id = "train_correlation:" + str(number)\n        component_coverage.append(dict(component_id=probe_id, feature_indices=indices,\n                                       status="INCLUDED" if included else "OMITTED_BY_PRESPECIFIED_GROUP_CAP"))\n        if included:\n            probes.append(dict(probe_id=probe_id, kind="train_correlation", model_positions=positions,\n                               feature_indices=indices, feature_names=[names[p] for p in positions],\n                               reasons=["TRAIN-only absolute-Spearman connected component; largest groups first, original-index tie break."]))\n    _require(bool(probes), "Declare at least one feature or source-family probe.")\n    plan = dict(schema_version=2, selected_columns=columns, selected_names=names,\n                model_family=joined["summary"]["model_family"], model_identities=joined["summary"]["model_identities"],\n                random_seed=random_seed, max_individual_features=max_features,\n                quotas=dict(top_train=top_train, top_internal=top_internal, lowest_train=lowest_train, random_remaining=random_remaining),\n                probes=probes, individual_probe_count=len(choices), family_probe_count=len(family),\n                tercile_probe_count=sum(p["kind"] == "train_tercile" for p in probes),\n                correlation_probe_count=sum(p["kind"] == "train_correlation" for p in probes),\n                correlation_threshold=float(correlation_threshold), max_correlation_groups=max_correlation_groups,\n                correlation_pairs_supplied=correlation_pairs is not None, correlation_component_coverage=component_coverage,\n                omitted_correlation_components=max(0, len(components)-max_correlation_groups),\n                grouping_scope="TRAIN rankings/pairs only; pair provenance is a caller contract, not inferred from values.",\n                selection_rule="Union of fixed TRAIN/internal extremes and label-independent random remainder; no permutation-result selection.",\n                intended_use="Post-lock VALIDATION diagnosis only; no automatic feature or configuration changes.")\n    plan["plan_sha256"] = _hash(plan)\n    return plan\n\n\ndef _validate_plan(plan, feature_names):\n    _require(plan.get("schema_version") == 2 and plan.get("plan_sha256") == _hash({k:v for k,v in plan.items() if k != "plan_sha256"}),\n             "Permutation probe plan hash mismatch.")\n    _require(list(feature_names) == plan["selected_names"], "Raw tensor feature names/order differ from the locked probe plan.")\n    _require(len(set(feature_names)) == len(feature_names) and len(feature_names) == len(plan["selected_columns"]),\n             "Invalid selected feature identity.")\n    probe_ids = set()\n    _require(bool(plan["probes"]), "The permutation probe plan is empty.")\n    for probe in plan["probes"]:\n        positions = probe["model_positions"]\n        _require(probe["probe_id"] not in probe_ids and positions and len(set(positions)) == len(positions), "Duplicate/empty probe.")\n        _require(all(type(p) is int and 0 <= p < len(feature_names) for p in positions), "Probe position outside model input.")\n        _require(probe["feature_names"] == [feature_names[p] for p in positions] and\n                 probe["feature_indices"] == [plan["selected_columns"][p] for p in positions], "Probe feature name/index/position mismatch.")\n        _require(probe["kind"] in {"feature", "family", "train_tercile", "train_correlation"}, "Unrecognized prespecified probe kind.")\n        probe_ids.add(probe["probe_id"])\n\n\ndef _probabilities(predictor, values):\n    output = np.asarray(predictor(values), dtype=float)\n    _require(output.shape == (len(values),) and np.isfinite(output).all() and ((output >= 0) & (output <= 1)).all(),\n             "Frozen predictor must return one finite probability per input patient.")\n    return output\n\n\ndef _lift_at_ten_percent(labels, probabilities):\n    # Same ceil(top-fraction*N) rule and frozen-input-order score-tie policy as\n    # comparison.py. Diagnostic unit here is latest validation patient only.\n    count = max(1, int(np.ceil(.1 * len(labels))))\n    top = np.argsort(-probabilities, kind="stable")[:count]\n    return float(np.mean(labels[top]) / np.mean(labels))\n\n\ndef validation_permutation_diagnostic(X, metadata, feature_names, predictor, plan, lock_hash, *,\n                                     split="validation", repeat_seeds=(271, 811, 1297),\n                                     all_validation_metadata=None):\n    """Whole 12-step trajectories within END_DT month; groups share permutations.\n\n    X and metadata must contain one latest validation snapshot per patient. If\n    full validation metadata is supplied, latest dates and labels are verified.\n    Otherwise latest-row choice remains a documented caller precondition. The\n    predictor must be the frozen saved ensemble, with no refitting/calibration.\n    """\n    _require(split == "validation", "TEST and TRAIN permutation diagnostics are not permitted by this API.")\n    _require(isinstance(lock_hash, str) and bool(lock_hash.strip()), "A persisted nonempty decision-lock hash is required.")\n    _validate_plan(plan, list(feature_names))\n    seeds = tuple(repeat_seeds)\n    _require(len(seeds) >= 2 and len(set(seeds)) == len(seeds) and all(type(s) is int and s >= 0 for s in seeds),\n             "Declare at least two unique label-independent repeat seeds.")\n    values = np.asarray(X)\n    _require(values.ndim == 3 and values.shape[1:] == (12, len(feature_names)) and len(values) >= 2,\n             "Permutation input must be raw (patients,12,selected_features).")\n    _require(np.issubdtype(values.dtype, np.number) and not np.iscomplexobj(values), "Raw counts must be real numeric values.")\n    for start in range(0, len(values), 128):\n        block = values[start:start+128]\n        observed = block[np.isfinite(block)]\n        _require(not np.isinf(block).any() and np.all(observed >= 0) and np.all(observed == np.floor(observed)),\n                 "Permutation input must preserve nonnegative integer raw counts or NaN.")\n    meta = pd.DataFrame(metadata).reset_index(drop=True)\n    _require(len(meta) == len(values) and {"PATIENT_ID", "END_DT", "RESP", "SPLIT"} <= set(meta), "Missing/alignment-error validation metadata.")\n    _require(meta.SPLIT.eq("validation").all(), "Only frozen VALIDATION rows may be diagnosed.")\n    _require(meta.PATIENT_ID.map(lambda v: isinstance(v, str) and bool(v)).all() and meta.PATIENT_ID.is_unique,\n             "Supply exactly one latest validation snapshot per distinct patient.")\n    dates = pd.to_datetime(meta.END_DT, errors="raise")\n    _require(dates.notna().all() and dates.eq(dates.dt.normalize()).all() and dates.dt.tz is None, "Invalid snapshot dates.")\n    y = meta.RESP.to_numpy()\n    _require(set(np.unique(y)) == {0, 1}, "Diagnostic labels must contain both binary RESP classes.")\n    latest_verified = False\n    if all_validation_metadata is not None:\n        all_meta = pd.DataFrame(all_validation_metadata).copy()\n        _require({"PATIENT_ID", "END_DT", "RESP", "SPLIT"} <= set(all_meta) and all_meta.SPLIT.eq("validation").all(),\n                 "Latest-row reference must contain only validation metadata.")\n        all_meta["END_DT"] = pd.to_datetime(all_meta.END_DT, errors="raise")\n        _require(all_meta.PATIENT_ID.map(lambda v: isinstance(v, str) and bool(v)).all() and\n                 all_meta.END_DT.notna().all() and all_meta.END_DT.eq(all_meta.END_DT.dt.normalize()).all() and\n                 all_meta.END_DT.dt.tz is None and all_meta.RESP.isin([0, 1]).all(), "Invalid latest-row reference metadata.")\n        _require(not all_meta.duplicated(["PATIENT_ID", "END_DT"]).any(), "Duplicate validation reference keys.")\n        latest = all_meta.sort_values(["PATIENT_ID", "END_DT"]).groupby("PATIENT_ID", sort=False).tail(1).set_index("PATIENT_ID")\n        _require(set(meta.PATIENT_ID) == set(latest.index), "Latest-patient diagnostic omits validation patients.")\n        reference = latest.loc[meta.PATIENT_ID]\n        _require(np.array_equal(reference.END_DT.to_numpy(), dates.to_numpy()) and np.array_equal(reference.RESP.to_numpy(), y),\n                 "Selected rows are not the latest frozen validation dates/labels.")\n        latest_verified = True\n    base = np.array(values, copy=True)\n    base.flags.writeable = False\n    baseline = _probabilities(predictor, base)\n    baseline_ap = float(average_precision_score(y, baseline))\n    baseline_lift = _lift_at_ten_percent(y, baseline)\n    month = dates.dt.to_period("M").astype(str).to_numpy()\n    strata = [np.flatnonzero(month == value) for value in sorted(set(month))]\n    permutations = {}\n    for seed in seeds:\n        rng = np.random.default_rng(seed)\n        order = np.arange(len(base))\n        for indices in strata:\n            order[indices] = rng.permutation(indices)\n        permutations[seed] = order\n    repeats = []\n    for probe in plan["probes"]:\n        positions = probe["model_positions"]\n        for seed in seeds:\n            order = permutations[seed]\n            changed = base.copy()\n            changed[:, :, positions] = base[np.ix_(order, np.arange(12), positions)]\n            before_values, after_values = base[:, :, positions], changed[:, :, positions]\n            equal = (before_values == after_values) | (np.isnan(before_values) & np.isnan(after_values))\n            changed_rows = int((~equal.all(axis=(1, 2))).sum())\n            changed.flags.writeable = False\n            predicted = _probabilities(predictor, changed)\n            shuffled_ap = float(average_precision_score(y, predicted))\n            repeats.append(dict(PROBE_ID=probe["probe_id"], KIND=probe["kind"], REPEAT_SEED=seed,\n                                FEATURES_IN_PROBE=len(positions), BASELINE_AP=baseline_ap, PERMUTED_AP=shuffled_ap,\n                                AP_DROP=baseline_ap-shuffled_ap, MEAN_ABS_PREDICTION_CHANGE=float(np.mean(np.abs(predicted-baseline))),\n                                MEAN_SIGNED_PREDICTION_CHANGE=float(np.mean(predicted-baseline)),\n                                BASELINE_LIFT_AT10=baseline_lift, PERMUTED_LIFT_AT10=_lift_at_ten_percent(y, predicted),\n                                LIFT_AT10_DROP=baseline_lift-_lift_at_ten_percent(y, predicted),\n                                PATIENTS_ACTUALLY_SHUFFLED=int((order != np.arange(len(base))).sum()),\n                                MOVED_PATIENT_FRACTION=float(np.mean(order != np.arange(len(base)))),\n                                RAW_VALUE_CHANGED_PATIENTS=changed_rows, RAW_VALUE_CHANGED_FRACTION=changed_rows/len(base)))\n    repeat_table = pd.DataFrame(repeats)\n    summary = repeat_table.groupby(["PROBE_ID", "KIND"], sort=False, as_index=False).agg(\n        FEATURES_IN_PROBE=("FEATURES_IN_PROBE", "first"), BASELINE_AP=("BASELINE_AP", "first"),\n        MEAN_AP_DROP=("AP_DROP", "mean"), REPEAT_SD_AP_DROP=("AP_DROP", "std"),\n        BASELINE_LIFT_AT10=("BASELINE_LIFT_AT10", "first"), MEAN_LIFT_AT10_DROP=("LIFT_AT10_DROP", "mean"),\n        REPEAT_SD_LIFT_AT10_DROP=("LIFT_AT10_DROP", "std"),\n        MEAN_MOVED_PATIENT_FRACTION=("MOVED_PATIENT_FRACTION", "mean"),\n        MEAN_RAW_VALUE_CHANGED_FRACTION=("RAW_VALUE_CHANGED_FRACTION", "mean"),\n        MEAN_ABS_PREDICTION_CHANGE=("MEAN_ABS_PREDICTION_CHANGE", "mean"),\n        REPEAT_SD_PREDICTION_CHANGE=("MEAN_ABS_PREDICTION_CHANGE", "std"), REPEATS=("REPEAT_SEED", "size"))\n    individual = {p["feature_indices"][0]:p["probe_id"] for p in plan["probes"] if p["kind"] == "feature"}\n    family_members = {index:[] for index in plan["selected_columns"]}\n    for probe in plan["probes"]:\n        if probe["kind"] != "feature":\n            for index in probe["feature_indices"]:\n                family_members[index].append(probe["probe_id"])\n    coverage = pd.DataFrame(dict(FEATURE_INDEX=plan["selected_columns"], FEATURE_NAME=feature_names,\n                                 MODEL_POSITION=np.arange(len(feature_names))))\n    coverage["INDIVIDUALLY_TESTED"] = coverage.FEATURE_INDEX.isin(individual)\n    coverage["PERMUTATION_STATUS"] = np.where(coverage.INDIVIDUALLY_TESTED, "MEASURED_VALIDATION_MARGINAL", "NOT_TESTED_UNKNOWN")\n    coverage["PROBE_ID"] = coverage.FEATURE_INDEX.map(individual)\n    coverage["JOINT_GROUP_PROBES"] = coverage.FEATURE_INDEX.map(lambda index:";".join(family_members[index]))\n    coverage = coverage.merge(summary.loc[summary.KIND.eq("feature"), ["PROBE_ID", "MEAN_AP_DROP", "REPEAT_SD_AP_DROP", "MEAN_ABS_PREDICTION_CHANGE",\n                              "MEAN_LIFT_AT10_DROP", "REPEAT_SD_LIFT_AT10_DROP", "MEAN_RAW_VALUE_CHANGED_FRACTION"]],\n                              on="PROBE_ID", how="left", validate="many_to_one")\n    protocol = dict(lock_hash=lock_hash, plan_sha256=plan["plan_sha256"], split="validation",\n                    evaluation_unit="one_latest_snapshot_per_patient", latest_selection_verified=latest_verified,\n                    comparison_scope="Secondary latest-patient diagnostic; not the primary repeated-snapshot metric population.",\n                    patients=len(base), positive_patients=int(y.sum()), baseline_ap=baseline_ap, baseline_lift_at10=baseline_lift,\n                    permutation_strata="END_DT_calendar_month", calendar_strata=len(strata),\n                    singleton_strata=sum(len(indices) == 1 for indices in strata),\n                    patients_in_singleton_strata=sum(len(indices) for indices in strata if len(indices) == 1),\n                    repeat_seeds=list(seeds), predictor_calls=1+len(plan["probes"])*len(seeds),\n                    full_population_reliance_measured=False, all_features_individually_tested=bool(coverage.INDIVIDUALLY_TESTED.all()),\n                    fitting_performed=False, configuration_changes_allowed=False, test_rows_used=False,\n                    uncertainty="Repeat SD measures permutation randomness, not a confidence interval or patient sampling uncertainty.",\n                    limitation="Marginal shuffling can break correlations and create unrealistic combinations; joint groups preserve only within-group trajectories/dependence. AP drop is conditional predictive reliance, not causation or optimal feature selection.",\n                    baseline_reuse="One frozen ensemble prediction reused across probes; identical label-independent patient permutations for each repeat seed.")\n    return dict(probe_summary=summary, repeats=repeat_table, coverage=coverage, protocol=protocol)\n', 'warehouse': '"""Embedded by the delivery notebooks; no repository dependency at runtime."""\nimport base64\nimport hashlib\nimport io\nimport json\nimport math\nimport re\nimport tempfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\ndef canonical_json(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)\n\n\ndef digest_json(value):\n    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()\n\n\ndef read_sf(suffix):\n    return (spark.read.format("snowflake").options(**sf_options_dl_poc)\n            .option("dbtable", f"{PREFIX}_{suffix}").load())\n\n\ndef table_exists(table):\n    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):\n        raise ValueError("Use uppercase letters, numbers and underscores in table names.")\n    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "\n             f"WHERE TABLE_SCHEMA = \'DS_ML\' AND TABLE_NAME = \'{table}\'")\n    return (spark.read.format("snowflake").options(**sf_options_dl_poc)\n            .option("query", query).load().limit(1).count() > 0)\n\n\ndef checked_metadata(frame, include_split=False):\n    columns = ["PATIENT_ID", "END_DT", "RESP"]\n    if include_split:\n        columns += ["SPLIT", "SPLIT_CONFIG"]\n    out = frame.loc[:, columns].copy()\n    if out.empty or out.isna().any().any():\n        raise ValueError("Snapshot metadata must be nonempty and contain no nulls.")\n    if not out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x)).all():\n        raise ValueError("PATIENT_ID must retain its original nonempty string value.")\n    dates = pd.to_datetime(out.END_DT, errors="raise")\n    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():\n        raise ValueError("END_DT must be a date without an intraday time/timezone.")\n    out["END_DT"] = dates.dt.strftime("%Y-%m-%d")\n    if not out.RESP.isin([0, 1]).all():\n        raise ValueError("RESP must be exactly 0 or 1 before conversion.")\n    out["RESP"] = out.RESP.astype("int64")\n    if out.duplicated(["PATIENT_ID", "END_DT"]).any():\n        raise ValueError("Duplicate patient/date keys in snapshot metadata.")\n    if include_split:\n        if set(out.SPLIT) != {"train", "validation", "test"}:\n            raise ValueError("Expected the saved train, validation and test assignments.")\n        if out.groupby("PATIENT_ID", observed=True).SPLIT.nunique().gt(1).any():\n            raise ValueError("Patient overlap between splits.")\n        if len(out.SPLIT_CONFIG.unique()) != 1:\n            raise ValueError("The split contains inconsistent creation settings.")\n        for _, part in out.groupby("SPLIT", observed=True):\n            if set(part.RESP) != {0, 1}:\n                raise ValueError("Each split must contain both response classes.")\n    return out.sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)\n\n\ndef validate_metadata_pair(snapshot_frame, manifest_frame, features):\n    source = checked_metadata(snapshot_frame)\n    manifest = checked_metadata(manifest_frame, include_split=True)\n    if not source.equals(manifest[["PATIENT_ID", "END_DT", "RESP"]]):\n        raise ValueError("Frozen split and source snapshots differ in keys or labels.")\n    creation = json.loads(manifest.SPLIT_CONFIG.iloc[0])\n    # Match the feature-order hash created in the completed split notebook.\n    feature_order_hash = hashlib.sha256(\n        json.dumps(features, ensure_ascii=False).encode("utf-8")).hexdigest()\n    if creation.get("feature_order_sha256") != feature_order_hash or creation.get("n_timesteps") != 12:\n        raise ValueError("Feature order or timesteps differ from the frozen split.")\n    return manifest, creation\n\n\ndef new_tensor_buffer(metadata, feature_count, seq_len=12):\n    temporary = tempfile.TemporaryDirectory(prefix="dl_poc_")\n    path = Path(temporary.name) / "counts.float32"\n    X = np.memmap(path, dtype="<f4", mode="w+", shape=(len(metadata), seq_len, feature_count))\n    return temporary, X\n\n\ndef fill_tensor(X, metadata, sequence_rows):\n    """Place rows by canonical keys, independent of Spark partition order."""\n    positions = {(r.PATIENT_ID, r.END_DT): i for i, r in enumerate(metadata.itertuples())}\n    seen = np.zeros(len(metadata), dtype=bool)\n    seq_len, feature_count = X.shape[1:]\n    for row in sequence_rows:\n        raw_date = row["END_DT"]\n        if raw_date is None or row["PATIENT_ID"] is None:\n            raise ValueError("Null monthly snapshot key.")\n        date = pd.Timestamp(raw_date)\n        if date.tzinfo is not None or date != date.normalize():\n            raise ValueError("Monthly END_DT is not an exact date.")\n        key = (row["PATIENT_ID"], date.strftime("%Y-%m-%d"))\n        if key not in positions:\n            raise ValueError("Unexpected monthly snapshot key.")\n        i = positions[key]\n        if seen[i] or row["RESP"] != int(metadata.RESP.iloc[i]):\n            raise ValueError("Duplicate monthly snapshot or changed label.")\n        sequence = row["SEQUENCE"]\n        steps = [month["T"] for month in sequence]\n        if len(sequence) != seq_len or any(t is None for t in steps):\n            raise ValueError("Expected exactly 12 complete timesteps per snapshot.")\n        # Check original values before any integer conversion.\n        if sorted(steps) != list(range(seq_len)):\n            raise ValueError("Timesteps must be unique integers 0 through 11.")\n        sequence = sorted(sequence, key=lambda month: month["T"])\n        values = np.asarray([month["V"] for month in sequence], dtype=np.float32)\n        if values.shape != (seq_len, feature_count):\n            raise ValueError("Monthly feature shape does not match the vocabulary.")\n        if not np.isfinite(values).all() or (values < 0).any():\n            raise ValueError("Counts must be finite, nonnegative float32 values with no nulls.")\n        X[i] = values\n        seen[i] = True\n    if not seen.all():\n        raise ValueError("Monthly data is missing original snapshots, including zero-activity sequences.")\n    X.flush()\n\n\ndef input_fingerprints(X, metadata, features):\n    tensor_hash = hashlib.sha256()\n    tensor_hash.update(canonical_json(list(X.shape)).encode("utf-8"))\n    for start in range(0, len(X), 128):\n        tensor_hash.update(np.asarray(X[start:start + 128], dtype="<f4").tobytes(order="C"))\n    records = [[str(r.PATIENT_ID), str(r.END_DT), int(r.RESP), str(r.SPLIT)]\n               for r in metadata.itertuples()]\n    return {"model_input_sha256": tensor_hash.hexdigest(),\n            "snapshot_manifest_sha256": digest_json(records),\n            "feature_names_sha256": digest_json(features),\n            "split_config_sha256": digest_json(json.loads(metadata.SPLIT_CONFIG.iloc[0]))}\n\n\ndef load_inputs():\n    from pyspark.sql import functions as F\n    mapping = read_sf("FEATURE_MAP").orderBy("FEATURE_INDEX").collect()\n    if not mapping or [r["FEATURE_INDEX"] for r in mapping] != list(range(len(mapping))):\n        raise ValueError("Invalid feature indices.")\n    features = [r["FEATURE_NAME"] for r in mapping]\n    aliases = [r["FEATURE_COLUMN"] for r in mapping]\n    if (aliases != [f"F{i:04d}" for i in range(len(features))]\n            or not all(isinstance(f, str) and f for f in features)\n            or len(set(features)) != len(features)):\n        raise ValueError("Invalid feature names, aliases or order.")\n    source = read_sf("SNAPSHOTS").select("PATIENT_ID", "END_DT", "RESP").toPandas()\n    frozen = read_sf("PATIENT_SPLIT").select(\n        "PATIENT_ID", "END_DT", "RESP", "SPLIT", "SPLIT_CONFIG").toPandas()\n    metadata, creation = validate_metadata_pair(source, frozen, features)\n    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()), len(features))\n    if observed != (23151, 12447, 1345, 1028):\n        raise ValueError(f"V63 population changed: snapshots/patients/positives/features = {observed}.")\n    monthly = read_sf("TENSOR_MONTHLY")\n    if set(monthly.columns) != set(["PATIENT_ID", "END_DT", "RESP", "TIME_STEP"] + aliases):\n        raise ValueError("Monthly table columns differ from the frozen feature map.")\n    print("Building the model input in a temporary driver file (about 1.06 GiB).", flush=True)\n    grouped = (monthly.groupBy("PATIENT_ID", "END_DT", "RESP")\n        .agg(F.collect_list(F.struct(\n            F.col("TIME_STEP").alias("T"),\n            F.array(*[F.when(F.col(name) >= 0, F.col(name).cast("float"))\n                      .otherwise(F.lit(None).cast("float")) for name in aliases]).alias("V")\n        )).alias("SEQUENCE"))\n        .repartition(128))\n    temporary, X = new_tensor_buffer(metadata, len(features))\n    try:\n        fill_tensor(X, metadata, grouped.toLocalIterator(prefetchPartitions=False))\n        hashes = input_fingerprints(X, metadata, features)\n        X.flags.writeable = False\n    except BaseException:\n        X._mmap.close()\n        temporary.cleanup()\n        raise\n    y = metadata.RESP.to_numpy(dtype=np.float32)\n    indices = {name: np.flatnonzero(metadata.SPLIT.to_numpy() == name)\n               for name in ("train", "validation", "test")}\n    print(f"Validated tensor shape {X.shape}; patient overlap = 0.", flush=True)\n    return {"X": X, "y": y, "metadata": metadata, "features": features,\n            "indices": indices, "hashes": hashes, "split_creation": creation,\n            "temporary_directory": temporary}\n\n\ndef pack_artifacts(artifacts, chunk_size=50000):\n    rows = []\n    for name, blob in artifacts.items():\n        encoded = base64.b64encode(blob).decode("ascii")\n        pieces = [encoded[i:i + chunk_size] for i in range(0, len(encoded), chunk_size)] or [""]\n        digest = hashlib.sha256(blob).hexdigest()\n        rows.extend((name, i, len(pieces), len(blob), digest, piece)\n                    for i, piece in enumerate(pieces))\n    return rows\n\n\ndef unpack_artifacts(rows, expected_names):\n    groups = {}\n    for row in rows:\n        name, i, count, size, digest, payload = tuple(row)\n        if any(value != int(value) for value in (i, count, size)):\n            raise ValueError("Nonintegral artifact chunk metadata.")\n        groups.setdefault(name, []).append((int(i), int(count), int(size), digest, payload))\n    if set(groups) != set(expected_names):\n        raise ValueError("Missing or unexpected saved artifacts.")\n    result = {}\n    for name, pieces in groups.items():\n        pieces.sort(key=lambda p: p[0])\n        count, size, digest = pieces[0][1:4]\n        if (count < 1 or size < 0 or len(pieces) != count\n                or [p[0] for p in pieces] != list(range(count))\n                or any(p[1:4] != (count, size, digest) for p in pieces)):\n            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")\n        blob = base64.b64decode("".join(p[4] for p in pieces), validate=True)\n        if len(blob) != size or hashlib.sha256(blob).hexdigest() != digest:\n            raise ValueError("Artifact length/hash mismatch.")\n        result[name] = blob\n    return result\n\n\nARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]\n\n\ndef read_artifacts(table, expected_names):\n    rows = (spark.read.format("snowflake").options(**sf_options_dl_poc)\n            .option("dbtable", table).load().select(*ARTIFACT_COLUMNS).collect())\n    return unpack_artifacts(rows, expected_names)\n\n\ndef save_artifacts(table, artifacts):\n    # A matching existing result can be verified after an interrupted read-back.\n    if table_exists(table):\n        if read_artifacts(table, artifacts) != artifacts:\n            raise FileExistsError("Destination contains different artifacts; choose a new RUN_ID.")\n        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")\n        return\n    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "\n              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")\n    frame = spark.createDataFrame(pack_artifacts(artifacts), schema=schema)\n    (frame.write.format("snowflake").options(**sf_options_dl_poc)\n     .option("dbtable", table).option("truncate_columns", "off")\n     .mode("errorifexists").save())\n    if read_artifacts(table, artifacts) != artifacts:\n        raise ValueError("Saved artifact read-back differs from the completed run.")\n    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")\n'}
modules = {}
for name, source in EMBEDDED_SOURCES.items():
    module_name = "_tak861_final_" + name
    module = types.ModuleType(module_name)
    sys.modules[module_name] = module
    exec(compile(source, module_name, "exec"), module.__dict__)
    modules[name] = module
fs, tr, wf, lin, cmp, b49, rel, wh = [modules[n] for n in
    ("features", "training", "workflow", "lineage", "comparison", "baseline49", "reliance", "warehouse")]
wh.spark, wh.sf_options_dl_poc, wh.PREFIX = spark, sf_options_dl_poc, SOURCE_PREFIX
for name in ("read_sf", "checked_metadata", "input_fingerprints", "table_exists", "read_artifacts", "save_artifacts"):
    setattr(wf, name, getattr(wh, name))
wf.stable_hash, wf.train_candidate = tr.stable_hash, tr.train_candidate
IMPLEMENTATION_SHA256 = '48f7a1e0b10113320b4be9cd3a52039aec5888ba52c2c6d54a479373b56641a3'
def read_table(table):
    return spark.read.format("snowflake").options(**sf_options_dl_poc).option("dbtable", table).load()
def safe_table(frame, limit=30):
    # Feature/aggregate tables only; never pass patient-level metadata here.
    display(frame.head(limit))
def frozen_predictor_identities(runs):
    # A predictor includes preprocessing and configuration, not just model bytes.
    return [{key:run["summary"][key] for key in
             ("seed","model_sha256","config_sha256","transform_sha256")} for run in runs]
def verify_frozen_predictors(runs, expected, label):
    if frozen_predictor_identities(runs) != expected:
        raise ValueError("Locked predictor model/config/preprocessing identity changed: " + label)
def count_input_contract(config):
    return {"columns":config["columns"],"representation":config["representation"],
            "step0_excluded":bool(config.get("drop_step0",False) or config["representation"]=="without_step0")}
def same_count_feature_information(transformer_config, lightgbm_config):
    tf,gb=count_input_contract(transformer_config),count_input_contract(lightgbm_config)
    return (tf["columns"]==gb["columns"] and tf["representation"] in {"monthly","without_step0"}
            and gb["representation"]=="flattened" and tf["step0_excluded"]==gb["step0_excluded"])
RELIANCE_NAMES = {"reliance.json", "transformer_matched_count_lgb_internal.csv"} | {
    family+"_"+kind+".csv" for family in ("transformer", "lightgbm")
    for kind in ("feature_audit", "internal_seeds", "rank_agreement", "probes", "repeats", "coverage")}
print("Implementation:", IMPLEMENTATION_SHA256)


### Load all complete trials

In [ ]:
if "data" in globals():
    wf.release_count_inputs(data)
data = wf.load_count_inputs()
prep = json.loads(wh.read_artifacts(RUN_PREFIX+"_PREPARATION", {"preparation.json", "lineage.json", "step0_screen.csv"})["preparation.json"])
wf.verify_stage(data, prep, IMPLEMENTATION_SHA256)
print("Verified all stage input hashes; no TEST predictions computed.")

design_protocol = json.loads(wh.read_artifacts(RUN_PREFIX+"_PROTOCOL", {"protocol.json","runtime.json"})["protocol.json"])
if design_protocol["input_hashes"] != data["hashes"] or design_protocol["implementation_hash"] != IMPLEMENTATION_SHA256:
    raise ValueError("Protocol and input/code identity mismatch")
protocol = wf.read_resolved_protocol(design_protocol, RUN_PREFIX)
results = wf.read_search(protocol,RUN_PREFIX)
winners,search_rows = tr.choose_finalists(results,list(protocol["candidates"]),protocol["seeds"])

selection_names = {"selection.json","quality.csv","ranking.csv","cv_scores.csv","cv_summary.csv",
                   "fold_rankings.csv","fold_audit.csv","correlations.csv","validation_shift.csv"}
selection_artifacts = wh.read_artifacts(RUN_PREFIX+"_SELECTION", selection_names)
selection_report = json.loads(selection_artifacts["selection.json"])
wf.verify_stage(data, selection_report, IMPLEMENTATION_SHA256)


### All-seed results and controlled experiments
These validation comparisons are exploratory after tuning; do not label their intervals independent confirmation.

In [ ]:
seed_report = cmp.summarize_seed_runs(results)
safe_table(pd.DataFrame(seed_report["all_seeds"]),200)
safe_table(pd.DataFrame(search_rows).sort_values(["family","mean_validation_ap"],ascending=[True,False]),100)
fig,ax = plt.subplots(figsize=(12,5))
rows = pd.DataFrame(search_rows)
ax.errorbar(np.arange(len(rows)),rows.mean_validation_ap,yerr=rows.sd_validation_ap,fmt="o")
ax.set_xticks(np.arange(len(rows)),rows.candidate,rotation=90)
ax.set_ylabel("Mean validation AP ± seed SD"); ax.set_title("All declared candidates; no seed picking")
plt.tight_layout(); plt.show()


### Learning curves and overfitting evidence
Checkpoint epochs are chosen by validation AP. Penalized training loss is not directly comparable with unpenalized validation log loss.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
for family,name in winners.items():
    for run in results[name]:
        history=pd.DataFrame(run["history"])
        axes[0].plot(history.epoch,history.validation_ap,label=family+" seed "+str(run["summary"]["seed"]))
        if "train_ap" in history:
            axes[1].plot(history.epoch,history.train_ap-history.validation_ap,label="seed "+str(run["summary"]["seed"]))
axes[0].set_title("Finalist validation learning curves"); axes[0].legend(fontsize=8)
axes[1].set_title("Transformer TRAIN minus VALIDATION AP"); axes[1].legend()
for ax in axes: ax.set_xlabel("Epoch / boosting iteration")
plt.tight_layout(); plt.show()


### Freeze models, thresholds, feature ordering and the comparison contract
        Threshold F1 is optimized on ensemble VALIDATION, not TEST. Top-10% is a capacity rule with deterministic frozen-key tie ordering,
        not a test-optimized probability threshold. Feature projection norms are scale-confounded internal parameter magnitude,
        not predictive reliance or causal importance. The separate post-lock audit below measures perturbation effects.
        The selected ordered list, full universe, removal ledger and rank list all remain in the saved artifacts.

In [ ]:
vi=data["indices"]["validation"]
policies={family:cmp.select_validation_threshold(data["y"][vi],wf.ensemble_scores(results[name],"validation"))
          for family,name in winners.items()}
selected_orders={family:[data["features"][i] for i in protocol["candidates"][name]["columns"]] for family,name in winners.items()}
feature_counts={family:len(v) for family,v in selected_orders.items()}
display({"finalists":winners,"feature_counts":feature_counts,"thresholds":policies})
for family,name in winners.items():
    safe_table(pd.DataFrame({"model_position":range(len(selected_orders[family])),"feature":selected_orders[family]}),40)
tfname=winners["transformer"]
projection=np.mean([r["summary"]["importance"] for r in results[tfname]],axis=0)
if len(projection)!=len(selected_orders["transformer"]): raise ValueError("Projection columns do not match selected feature order")
reliance=pd.DataFrame({"feature":selected_orders["transformer"],"mean_projection_norm":projection})
safe_table(reliance.sort_values("mean_projection_norm",ascending=False),25)
base_manifest=json.loads(wh.read_artifacts(RUN_PREFIX+"_BASE49",{"baseline.json"})["baseline.json"])
baseline_runs=[]
for seed in protocol["seeds"]:
    contract={"protocol_hash":tr.stable_hash(protocol),"candidate":"lightgbm49_refit","seed":seed,
              "feature_sha256":base_manifest["feature_sha256"],"input_sha256":base_manifest["input_sha256"]}
    baseline_runs.append(wf.unpack_run(wh.read_artifacts(RUN_PREFIX+"_BASE49_S"+str(seed),wf.RUN_ARTIFACT_NAMES),contract,
          expected_lengths={s:len(data["indices"][s]) for s in ("train","validation")},expected_config=b49.DEFAULT_BASELINE_CONFIG))
baseline_lock={"input_sha256":base_manifest["input_sha256"],"feature_sha256":base_manifest["feature_sha256"],
     "threshold_policy":cmp.select_validation_threshold(data["y"][vi],wf.ensemble_scores(baseline_runs,"validation")),
     "model_hashes":[r["summary"]["model_sha256"] for r in baseline_runs],
     "predictor_identities":frozen_predictor_identities(baseline_runs)}
lock={"input_hashes":data["hashes"],"implementation_hash":IMPLEMENTATION_SHA256,
      "protocol_hash":tr.stable_hash(protocol),"winners":winners,"threshold_policies":policies,
      "seeds":protocol["seeds"],"ordered_features":selected_orders,"feature_counts":feature_counts,
      "model_hashes":{f:[r["summary"]["model_sha256"] for r in results[n]] for f,n in winners.items()},"baseline49":baseline_lock,
      "predictor_identities":{f:frozen_predictor_identities(results[n]) for f,n in winners.items()},
      "test_used_for_new_selection":False,"historical_test_previously_inspected":True,
      "final_predictor":"uniform_three_seed_ensemble","primary_metric":"average_precision",
      "test_label":"retrospective frozen TEST; not an untouched prospective/OOT claim"}
wh.save_artifacts(RUN_PREFIX+"_LOCK", {"lock.json":wf.json_bytes(lock),"projection_parameter_magnitude.csv":wf.frame_bytes(reliance)})
print("Immutable final decision saved. Complete the post-lock validation reliance audit before Notebook 5.")


### Does measured model reliance agree with the TRAIN ranking?
        A screening rank is not a requirement for the fitted model's importance order. Join every original feature by index/name to
        its selected input position, TRAIN ranking/stability and each seed's internal magnitude. Projection norms are scale-confounded;
        norm × TRAIN transformed-input SD is also an internal diagnostic. Gain from the reference count LightGBM arm with the same feature IDs
        provides another view. Its temporal representation or step-0 policy may differ; both are recorded. This is not an architecture-controlled
        reliance comparison, even when the effective input information matches.

        The registered bounded probe union uses top 10 TRAIN ranks, top 10 internal ranks, bottom 5 TRAIN ranks and 5 seeded remaining
        features (at most 30); untested individual reliance stays UNKNOWN. Joint groups cover source families, selected-feature TRAIN-rank
        terciles and up to 10 largest TRAIN |Spearman|≥0.9 connected components. Omitted groups are reported.
        Only one **latest VALIDATION snapshot per patient** is used, with whole 12-position trajectories permuted within END_DT month.
        This is a secondary diagnostic population. It avoids treating repeated patient snapshots as independent permutation units.

        The locked uniform ensemble and its preprocessing never refit. Three fixed permutations report AP drop, lift@10% drop,
        prediction change and the fraction whose actual raw values changed. Repeat SD is not a confidence interval. Sparse constant values
        or singleton month strata may yield no perturbation. Correlated inputs can hide individual reliance or create unrealistic combinations;
        joint groups preserve within-group relationships only. VALIDATION was already used for selection, so these are exploratory diagnostics,
        not independent confirmation. They cannot change the locked features, models, thresholds or TEST policy.

In [ ]:
import io
train_ranking=pd.read_csv(io.BytesIO(selection_artifacts["ranking.csv"]),keep_default_na=False)
train_pairs=pd.read_csv(io.BytesIO(selection_artifacts["correlations.csv"]),keep_default_na=False)
audits={};plans={};settings=protocol["reliance_diagnostic"]
for family,name in winners.items():
    audits[family]=rel.join_rank_and_internal_diagnostics(data["feature_map"],train_ranking,results[name],
                        protocol["candidates"][name]["columns"],protocol["seeds"])
    plans[family]=rel.prespecify_probes(audits[family],correlation_pairs=train_pairs,
        max_features=settings["max_individual_features"],random_seed=settings["probe_seed"],
        top_train=settings["top_train"],top_internal=settings["top_internal"],
        lowest_train=settings["lowest_train"],random_remaining=settings["random_remaining"],
        correlation_threshold=settings["correlation_threshold"],max_correlation_groups=settings["max_correlation_groups"])
plan_record={"lock_hash":tr.stable_hash(lock),"settings":settings,"plans":plans}
wh.save_artifacts(RUN_PREFIX+"_RELIANCE_PLAN",{"plan.json":wf.json_bytes(plan_record)})
if wh.table_exists(RUN_PREFIX+"_RELIANCE"):
    audit_artifacts=wh.read_artifacts(RUN_PREFIX+"_RELIANCE",RELIANCE_NAMES)
    audit_report=json.loads(audit_artifacts["reliance.json"])
    if audit_report["lock_hash"]!=tr.stable_hash(lock) or audit_report["plan_record_sha256"]!=tr.stable_hash(plan_record):
        raise ValueError("Saved reliance audit changed its lock or declared plan")
else:
    audit_artifacts={};family_reports={}
    tf_columns=protocol["candidates"][winners["transformer"]]["columns"]
    twin_names=[name for name in protocol["phase_a"] if protocol["candidates"][name]["family"]=="lightgbm"
                and protocol["candidates"][name]["columns"]==tf_columns]
    if len(twin_names)!=1: raise ValueError("No unique count LightGBM arm matches Transformer feature identities")
    twin=rel.join_rank_and_internal_diagnostics(data["feature_map"],train_ranking,results[twin_names[0]],tf_columns,protocol["seeds"])
    tf_config=protocol["candidates"][winners["transformer"]];twin_config=protocol["candidates"][twin_names[0]]
    twin_contract={"label":"Same feature IDs; reference count-arm gain, not an architecture-controlled reliance comparison",
        "transformer_input":count_input_contract(tf_config),"reference_count_lgb_input":count_input_contract(twin_config),
        "same_effective_input_information":same_count_feature_information(tf_config,twin_config),
        "architecture_controlled_reliance_comparison":False}
    audit_artifacts["transformer_matched_count_lgb_internal.csv"]=wf.frame_bytes(twin["full_table"])
    for family,name in winners.items():
        x_val,y_val,groups_val=wf.candidate_views(data,protocol["candidates"][name],"validation")
        val_meta=data["metadata"].iloc[data["indices"]["validation"]].reset_index(drop=True)
        latest=cmp.latest_patient_indices(val_meta.PATIENT_ID,val_meta.END_DT)
        x_latest=x_val[latest];latest_meta=val_meta.iloc[latest].reset_index(drop=True)
        predictors=[tr.make_saved_predictor(run) for run in results[name]]
        def frozen_ensemble(values): return np.mean([predict(values) for predict in predictors],axis=0)
        np.testing.assert_allclose(frozen_ensemble(x_latest),wf.ensemble_scores(results[name],"validation")[latest],rtol=1e-5,atol=1e-6)
        diagnostic=rel.validation_permutation_diagnostic(x_latest,latest_meta,selected_orders[family],frozen_ensemble,
            plans[family],tr.stable_hash(lock),repeat_seeds=tuple(settings["repeat_seeds"]),all_validation_metadata=val_meta)
        joined=audits[family];full=joined["full_table"].copy();covered=diagnostic["coverage"].set_index("FEATURE_INDEX")
        for column in covered.columns:
            if column not in {"FEATURE_NAME","MODEL_POSITION","PERMUTATION_STATUS"}:
                full[column]=full.FEATURE_INDEX.map(covered[column])
        full["PERMUTATION_STATUS"]=full.FEATURE_INDEX.map(covered.PERMUTATION_STATUS).fillna("NOT_IN_MODEL")
        measured=full.loc[full.PERMUTATION_STATUS.eq("MEASURED_VALIDATION_MARGINAL")].copy()
        measured["AP_DROP_RANK_WITHIN_PROBES"]=measured.MEAN_AP_DROP.rank(ascending=False,method="average")
        measured["TRAIN_RANK_WITHIN_PROBES"]=measured.TRAIN_RANK.rank(method="average")
        measured["RANK_DIFFERENCE_WITHIN_PROBES"]=measured.TRAIN_RANK_WITHIN_PROBES-measured.AP_DROP_RANK_WITHIN_PROBES
        for column in ("AP_DROP_RANK_WITHIN_PROBES","TRAIN_RANK_WITHIN_PROBES","RANK_DIFFERENCE_WITHIN_PROBES"):
            full[column]=full.FEATURE_INDEX.map(measured.set_index("FEATURE_INDEX")[column])
        rho=None
        if len(measured)>1 and measured.MEAN_AP_DROP.nunique()>1 and measured.TRAIN_RANK.nunique()>1:
            rho=float(measured.TRAIN_RANK.corr(measured.AP_DROP_RANK_WITHIN_PROBES,method="spearman"))
        if family=="transformer":
            twin_table=twin["full_table"].set_index("FEATURE_INDEX")
            for column in ("MEAN_INTERNAL_MAGNITUDE","MEAN_INTERNAL_RANK"):
                full["MATCHED_COUNT_LGB_"+column]=full.FEATURE_INDEX.map(twin_table[column])
        family_reports[family]={"mapping":joined["summary"],"permutation":diagnostic["protocol"],
            "screen_vs_permutation_spearman_among_probed":rho,"individually_measured_features":len(measured),
            "agreement_rule":"No required rank agreement; inspect signed rank differences and groups; do not retune the locked model"}
        for kind,frame in {"feature_audit":full,"internal_seeds":joined["seed_table"],"rank_agreement":joined["agreement_table"],
                           "probes":diagnostic["probe_summary"],"repeats":diagnostic["repeats"],"coverage":diagnostic["coverage"]}.items():
            audit_artifacts[family+"_"+kind+".csv"]=wf.frame_bytes(frame)
        del predictors,x_val,x_latest
    audit_report={"lock_hash":tr.stable_hash(lock),"plan_record_sha256":tr.stable_hash(plan_record),"families":family_reports,
        "evidence_class":"Reproduced","matched_count_lgb_candidate":twin_names[0],
        "matched_count_lgb_reference_contract":twin_contract,
        "interpretation":"Secondary exploratory VALIDATION marginal reliance; no causal claim, no TEST use, no forced rank agreement"}
    audit_artifacts["reliance.json"]=wf.json_bytes(audit_report)
    wh.save_artifacts(RUN_PREFIX+"_RELIANCE",audit_artifacts)
display(audit_report)
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,family in zip(axes,("transformer","lightgbm")):
    frame=pd.read_csv(io.BytesIO(audit_artifacts[family+"_feature_audit.csv"]))
    safe_table(frame.sort_values("MEAN_AP_DROP",ascending=False),30)
    measured=frame[frame.PERMUTATION_STATUS.eq("MEASURED_VALIDATION_MARGINAL")]
    ax.errorbar(measured.TRAIN_RANK,measured.MEAN_AP_DROP,yerr=measured.REPEAT_SD_AP_DROP,fmt="o")
    ax.set(title=family+": secondary VALIDATION reliance",xlabel="TRAIN screening rank (1=highest)",ylabel="Permutation AP drop ± repeat SD")
    safe_table(pd.read_csv(io.BytesIO(audit_artifacts[family+"_probes.csv"])),50)
plt.tight_layout();plt.show()
print("Reliance audit saved; immutable decisions unchanged. Notebook 5 may now evaluate retrospective TEST.")
